<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.en/cap04/cap04_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 4 Mathematical Morphology and Image Segmentation

This chapter presents two fundamental topics in Digital Image Processing (DIP): **mathematical morphology** and **image segmentation**. Mathematical morphology provides a theoretical framework based on set theory for analyzing, refining, and quantifying the shape of objects in binary and grayscale images, through fundamental operators such as **erosion** and **dilation**. Segmentation, in turn, aims to partition the image into regions of interest, separating objects from the background and producing suitable representations for analysis and interpretation.

The chapter begins with **thresholding**, one of the most important segmentation techniques, introducing the automatic **Otsu** method and revisiting histogram analysis through interclass variance, as presented in Chapter 1. Next, the main mathematical morphology operators are studied, including erosion, dilation, opening, closing, and morphological reconstruction, which allow refining binary masks and preserving relevant object structures. Finally, region-based segmentation techniques are presented, such as **connected component labeling**, the **distance transform**, and the marker-controlled ***watershed*** algorithm, culminating in the extraction of geometric descriptors and the generation of *bounding boxes* compatible with modern object detection systems.

## 4.1 Objectives

By the end of this chapter, you will be able to:

* **Apply thresholding:** Understand Otsu's automatic criterion by maximizing inter-class variance ($\sigma_B^2$) and select appropriate preprocessing strategies to facilitate segmentation;
* **Master binary morphology:** Understand and apply erosion ($A\ominus B$) and dilation ($A\oplus B$) as fundamental operators, deriving opening ($A\circ B$), closing ($A\bullet B$), and operations based on morphological reconstruction, such as `mm::clohole` and `mm::edgeoff`;
* **Apply grayscale morphology:** Use morphological gradient and *top-hat* filters for enhancement and analysis of local structures;
* **Label connected components:** Identify and separate connected regions in binary images through labeling algorithms;
* **Apply distance transform:** Interpret and compute distances to the background using morphological approaches and geometric metrics;
* **Segment by regions:** Build marker-based segmentation *pipelines* using Distance Transform and the ***watershed*** algorithm;
* **Extract geometric descriptors:** Calculate properties such as area, perimeter, centroid, circularity, and *bounding boxes* through `mm::label0` and contour extraction;
* **Relate DIP and computer vision:** Understand how descriptors extracted by segmentation can be converted to formats used by modern detectors, such as YOLO.

In [1]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # build artifacts for the C++ track (.cpp, binary, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# The kernel is Python even in the C++ track: `mm` (morph.py) is used by the
# simulators, by the display of the figures that the C++ binary generates, and by the
# mm::Image state between cells. cpp=True also downloads the compiled track
# (morph.hpp + stb_image*.h), used in the #include of the %%writefile *.cpp cells.
# The C++ cells in this chapter compile WITH OpenCV (-DMM_USE_OPENCV +
# pkg-config opencv4): mm::dil/ero → cv::dilate/erode, so open/close/asf/
# gradm/tophat/blackhat run full-res and match the py track bit by bit.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

✅ Environment ready. Morph: 1.1.9 | OpenCV: 5.0.0


## 4.2 Thresholding

**Thresholding** is one of the simplest and most efficient forms of image segmentation. Its goal is to classify each pixel into two intensity classes, typically associated with *object* and *background*:

<a id="eq-04-limiar"></a>
$$
g(x,y) =
\begin{cases}
255, & \text{if } f(x,y) > T \\
0,   & \text{otherwise}
\end{cases} \tag{4.1}
$$


where $f(x,y)$ represents the pixel intensity in the original image and $g(x,y)$ is the resulting binary image.

The choice of the threshold $T$ is important for the quality of the segmentation. The **Otsu** method automatically determines the optimal threshold by maximizing the **between-class variance** $\sigma_B^2$, defined by:

<a id="eq-04-otsu"></a>
$$
\sigma_B^2(T) =
w_0(T)\,w_1(T)\,
\bigl[\mu_0(T)-\mu_1(T)\bigr]^2 \tag{4.2}
$$


where:

- $w_0(T)$ and $w_1(T)$ are the cumulative probabilities of the background and object classes;
- $\mu_0(T)$ and $\mu_1(T)$ are the mean intensities of these classes;
- $\sigma_B^2(T)$ represents the between-class variance for a given threshold $T$.

The method works best when the histogram presents two relatively separated groups of intensities. To that end, the algorithm evaluates all possible thresholds of the image — typically in the interval $[0,255]$ for 8-bit images — and selects the value that maximizes the between-class variance, denoted by $\sigma_B^2$:

$$
T^* =
\arg\max_{T \in [0,255]}
\sigma_B^2(T)
$$

> ### 📝 Otsu assumes bimodal histograms
>
> The Otsu method yields better results when the histogram presents two well-defined peaks (*bimodality*), corresponding to the background and the object. The greater the separation between these peaks and the more pronounced the maximum of $\sigma_B^2$, the more reliable the obtained threshold tends to be.
>
> In images with non-uniform illumination or multiple intensity regions, **adaptive thresholding** techniques — in which the threshold is computed locally — often produce more robust segmentations.
>
> The subscript $B$ in $\sigma_B^2$ stands for ***between classes***. Thus, $\sigma_B^2$ represents the **between-class variance**.

### 4.2.1 Image of Coins

The image used to practice segmentation is a photograph of a collection of coins from different countries and eras ([Figure 4.1](#fig-04-coins)). Credit: GAZI.MD.AHAD (CC BY-SA 4.0). It features circular objects with well-defined edges, making it ideal for demonstrating thresholding, morphological operators, distance transform, *watershed*, and shape descriptors.

In [2]:
%%writefile tmp/fig_04_coins.cpp
#define MM_OUT "tmp/fig_04_coins.png"
//| label: fig-04-coins
//| fig-cap: "Image with coins of various types. Credit: GAZI.MD.AHAD (CC BY-SA 4.0)."
//| echo: true

// The image is already in the chapter directory (imagens/coins.png); the Python
// track handles download/cache when the notebook runs standalone.
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    mm::Image img_coins_color = mm::read("imagens/coins.png");
    mm::Image img_coins_gray  = mm::gray(img_coins_color);

    std::cout << "Dimensions [y,x,c]: [" << img_coins_color.h << ", "
              << img_coins_color.w << ", " << img_coins_color.channels << "]\n";
    mm::show(img_coins_color, MM_OUT);
    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_coins_gray, "tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_04_coins.cpp


In [3]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_coins.cpp -o tmp/fig_04_coins -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_coins \
  && test -f "tmp/fig_04_coins.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_coins.png"

Dimensions [y,x,c]: [2560, 1920, 3]


In [4]:
mm.show(mm.read("tmp/fig_04_coins.png"))

<Figure size 960x1280 with 1 Axes>

**Figure 4.1:** Imagem com moedas de vários tipos. Crédito: GAZI.MD.AHAD (CC BY-SA 4.0).


### 4.2.2 Preprocessing for Otsu

Otsu's method relies on a well **bimodal** histogram. The coin image has non-uniform illumination and dark coins close to the background, which hinders this. In the C++ track, **global histogram equalization** (`mm::equalize`) is applied before thresholding — the CLAHE × Gaussian comparison and the $\sigma_B^2(T)$ curves are left for the Python track ([Figure 4.2](#fig-04-otsu-comparacao-histogramas)).

In [5]:
%%writefile tmp/fig_04_otsu_comparacao_histogramas.cpp
#define MM_OUT "tmp/fig_04_otsu_comparacao_histogramas.png"
//| label: fig-04-otsu-comparacao-histogramas
//| fig-cap: "CLAHE (realce adaptativo com limite de contraste) antes da limiarização de Otsu: original, realçado, histograma e binarização. (A trilha Python também traça as curvas de $\\sigma^2_B(T)$.)"
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_clahe = mm::clahe(img_coins_gray, 2.0, 8);   // adaptive contrast-limited enhancement
    mm::Image img_gauss = mm::gaussian(img_clahe, 5, 0);

    mm::show(
        std::vector<mm::Image>{img_coins_gray, img_clahe, mm::histImg(img_clahe), mm::threshold(img_clahe)},
        MM_OUT,
        std::vector<std::string>{"Original", "CLAHE", "Histogram (CLAHE)", "Otsu"},
        4
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_clahe, "tmp/state/img_clahe_12.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_04_otsu_comparacao_histogramas.cpp


In [6]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_otsu_comparacao_histogramas.cpp -o tmp/fig_04_otsu_comparacao_histogramas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_otsu_comparacao_histogramas \
  && test -f "tmp/fig_04_otsu_comparacao_histogramas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_otsu_comparacao_histogramas.png"

[1] Original
[2] CLAHE
[3] Histogram (CLAHE)
[4] Otsu


In [7]:
mm.show(mm.read("tmp/fig_04_otsu_comparacao_histogramas.png"))

<Figure size 1044x450 with 1 Axes>

**Figure 4.2:** CLAHE (realce adaptativo com limite de contraste) antes da limiarização de Otsu: original, realçado, histograma e binarização. (A trilha Python também traça as curvas de $\\sigma^2_B(T)$.)


### 4.2.3 Result: CLAHE as the Best Pre-processing

The analysis of [Figure 4.2](#fig-04-otsu-comparacao-histogramas) indicates that **CLAHE** achieved the highest inter-class variance ($\sigma_B^2 \approx 2{,}47 \times 10^3$), with an optimal threshold $T^* = 122$. Although the **CLAHE+Gaussian** combination produced a very similar result ($\sigma_B^2 \approx 2{,}43 \times 10^3$, $T^* = 123$), the quantitative criterion of Otsu's method slightly favors the use of CLAHE alone.

In visual terms, the binarized images obtained with CLAHE and CLAHE+Gaussian are practically equivalent. The difference between the two approaches becomes more evident in the analysis of the histograms and the values of $\sigma_B^2(T)$ than in the direct inspection of the resulting segmentations. Thus, the choice of CLAHE is mainly based on maximizing the statistical separation between the background and object classes.


> ### 💡 Interpretation of the results
>
> Note that the CLAHE and CLAHE+Gaussian pre-processings produce very similar histograms and optimal thresholds ($T^*=122$ and $T^*=123$). Consequently, the resulting binarized images are also quite similar. In this case, the decision is not based on striking visual differences but on the objective criterion of Otsu's method: the highest value of $\sigma_B^2$ indicates the best separation between the classes.

## 4.3 Mathematical Morphology

**Mathematical morphology** is a set-based theory used to analyze the shape and structure of objects in images. Unlike the linear filters presented in Chapter 3, morphological operators are **nonlinear**, as they are based on minimum, maximum, and spatial inclusion operations, rather than linear combinations of intensities. These operators act on the neighborhood of each pixel through a **structuring element** $\mathbb{B}$, which defines the shape and size of the analyzed region.

In binary and grayscale images with flat elements, the structuring element translated to position $x$ is defined spatially as:

$$
\mathbb{B}_x = \{ x + b \mid b \in \mathbb{B} \}
$$

At image border regions, part of the set $\mathbb{B}_x$ may extend beyond the physical domain of the scene ($\mathbb{E}$). To ensure the mathematical consistency of the primitive operators at these boundaries, it is theoretically assumed that the space exterior to the image domain is filled with the **neutral element** of the corresponding operation (positive infinity for erosion and negative infinity for dilation), preventing the external environment from corrupting the internal structures of the object.

When the structuring element associates weights to its elements — that is, $b: \mathbb{B} \to \mathbb{Z}$ — it is termed a **structuring function** or **non-flat structuring element**.

Developed by Matheron and Serra in the 1960s for binary images and subsequently extended to grayscale, mathematical morphology underpins operators such as morphological gradient, *top-hat*, *watershed*, and distance transform, all derived from two primitives: **erosion** and **dilation** [@matheron1975random; Serra (1982)].

### 4.3.1 Erosion and Dilation

The two primitive operators are defined in a unified manner for grayscale images ($f: \mathbb{E} \to \mathbb{Z}$) and, by restriction to the domain $\{0,1\}$, also for binary images.

#### 4.3.1.1 Erosion

The **erosion** of an image $f$ by a structuring function $b: \mathbb{B} \to \mathbb{Z}$ is formally defined by:

<a id="eq-04-erosao"></a>
$$
\varepsilon_b(f)(x) = (f \ominus b)(x) = \min_{z \in \mathbb{B}}\{\, f(x + z) - b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.3}
$$


In practice, erosion replaces the intensity of pixel $x$ with the minimum value resulting from the difference between the image and the structuring element in the neighborhood defined by the domain $\mathbb{B}$. Positive values in the weights of $b(z)$ force the local result downward, "digging" the image relief more deeply and intensifying the erosion.

In the **flat** case (where the weights are null within the domain, i.e., $b \equiv 0$), the expression simplifies to the pure local minimum:

$$
\varepsilon_B(f)(x) = \min\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

In binary images, this operation is equivalent to requiring that the set $\mathbb{B}$, translated to coordinate $x$, be **completely contained** in the object $A$:

$$
A \ominus \mathbb{B} = \{\, z \in \mathbb{E} \mid \mathbb{B}_z \subseteq A \,\}
$$

**Visual Effect:** *Shrinks* bright objects and structures, eliminating protrusions, bright peaks, or noise that are geometrically smaller than the domain $\mathbb{B}$.

#### 4.3.1.2 Implementation of erosion

The didactic version `mm::ero0` implements the particular case of erosion with a **flat structuring element**. For each pixel $(y,x)$, the function traverses the spatial neighbors allowed by $B$ and stores the smallest value found in the input image $f$, directly reproducing the local minimum operation described in [Equation 4.3](#eq-04-erosao) for $b \equiv 0$.

Note that the neighbor values are always read statically from the original image $f$; the output matrix $g$ is used exclusively to record the accumulated minimum of the current neighborhood. Thus, the final result is invariant with respect to the pixel scanning order (whether by rows or columns).

The auxiliary function `_viz` computes the coordinates of valid neighbors within the physical boundaries of the image. At the borders, the initialization of the accumulator to `255` exactly emulates the padding with the neutral element required by the theory. The interface function `mm::ero`, in turn, resorts to the native and optimized OpenCV implementation (`mm::ero`) when the structuring element is flat, switching to the general routine `mm::ero1` if the element has topographic weights.

The following computational example illustrates the application of a cross-shaped structuring element (`mm::secross()`) highlighted in [Figure 4.3](#fig-04-elemento-cruz), comparing the execution of the didactic loop-based variant (`mm::ero0`) with the computational engine of OpenCV (`mm::ero`).

In [8]:
%%writefile tmp/fig_04_elemento_cruz.cpp
#define MM_OUT "tmp/fig_04_elemento_cruz.png"
// Compile with: g++ -std=c++17 -O2 -o program program.cpp -I. -I/usr/include/opencv4 `pkg-config --cflags --libs opencv4` -DMM_USE_OPENCV -lcurl
#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
    //| label: fig-04-elemento-cruz
    //| fig-cap: "Elemento estruturante em formato de cruz ($B_{\\text{cruz}}$) utilizado para conectividade-4."
    //| echo: true
    //| output: true

    mm::Image B_cruz = mm::secross();
    mm::drawImgPlt(B_cruz, MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_04_elemento_cruz.cpp


In [9]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_elemento_cruz.cpp -o tmp/fig_04_elemento_cruz -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_elemento_cruz \
  && test -f "tmp/fig_04_elemento_cruz.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_elemento_cruz.png"

0 1 0 
1 1 1 
0 1 0 


In [10]:
mm.show(mm.read("tmp/fig_04_elemento_cruz.png"))

<Figure size 450x450 with 1 Axes>

**Figure 4.3:** Elemento estruturante em formato de cruz ($B_{\\text{cruz}}$) utilizado para conectividade-4.


In [11]:
# The morph.hpp is header-only; below, the _viz helper and the body of mm::ero0().
import re, pathlib
hpp = pathlib.Path("morph.hpp").read_text()
for nome, pat in [("_viz", r"template <class F>\ninline void _viz\(.*?\n\}"),
                  ("ero0", r"inline Image ero0\(.*?\n\}")]:
    m = re.search(pat, hpp, re.S)
    print(f"// ---- mm::{nome} ----")
    print(m.group(0) if m else f"({nome} não encontrada)")
    print()

// ---- mm::_viz ----
template <class F>
inline void _viz(const Image& f, const SE& B, int y, int x, F&& cb) {
    double oh = -B.h / 2.0 + 0.5;
    double ow = -B.w / 2.0 + 0.5;
    for (int by = 0; by < B.h; ++by)
        for (int bx = 0; bx < B.w; ++bx) {
            int vy = (int)(y + by + oh);
            int vx = (int)(x + bx + ow);
            if (vy >= 0 && vy < f.h && vx >= 0 && vx < f.w)
                cb(vy, vx, B.at(by, bx));
        }
}

// ---- mm::ero0 ----
inline Image ero0(const Image& f, SE Bc = SE::box(3)) {
    _require_gray(f, "ero0");
    Image g(f.h, f.w, 1);
    for (int y = 0; y < f.h; ++y)
        for (int x = 0; x < f.w; ++x) {
            int mn = 255;
            _viz(f, Bc, y, x, [&](int vy, int vx, int bv) {
                if (bv != 0 && (int)f.at(vy, vx) < mn) mn = f.at(vy, vx);
            });
            g.at(y, x) = (unsigned char)mn;
        }
    return g;
}



#### 4.3.1.3 Dilation

**Dilation** of an image $f$ by a structuring function $b: \mathbb{B} \to \mathbb{Z}$ is formally defined by:

<a id="eq-04-dilatacao"></a>
$$
\delta_b(f)(x) = (f \oplus b)(x) = \max_{z \in \mathbb{B}}\{\, f(x - z) + b(z) \,\}, \quad \forall\, x \in \mathbb{E} \tag{4.4}
$$


In practice, dilation replaces the intensity of pixel $x$ with the largest value resulting from the sum between the image and the structuring element within the defined neighborhood. The spatial inversion argument ($x - z$) indicates that dilation implicitly evaluates the transposed (reflected) element $\hat{b}$, a fundamental property for ensuring mathematical duality with respect to erosion.

In the **flat** case (where the weights are null within the domain, i.e., $b \equiv 0$), the expression reduces to pure local maximum:

$$
\delta_B(f)(x) = \max\{\, f(y) : y \in \mathbb{B}_x \,\}
$$

For binary images, this operation is equivalent to requiring that the reflected set $\hat{\mathbb{B}}$, translated to coordinate $x$, has a **non-empty intersection** with object $A$:

$$
A \oplus \mathbb{B} = \{\, z \in \mathbb{E} \mid \hat{\mathbb{B}}_z \cap A \neq \varnothing \,\}
$$

**Visual Effect:** *Expands* the bright structures of the image, increasing object filling, connecting nearby components, and eliminating channels, dark pits, or valleys that are geometrically smaller than the domain $\mathbb{B}$.

#### 4.3.1.4 Implementation of dilation

The didactic version `mm::dil0` implements the particular case of dilation with a **flat structuring element**. For each pixel $(y,x)$, the function traverses the spatial neighbors allowed by $B$ and stores the largest value found in the input image $f$, directly reproducing the local maximum operation for $b \equiv 0$.

Before starting the spatial scan, the structuring element undergoes a geometric reflection through the `np.flip(Bc)` instruction to explicitly construct the transposed matrix $\hat{B}$ required by the theory. In perfectly symmetric masks (such as crosses, squares, and disks centered at the origin), this reflection does not alter the pixel arrangement; however, for asymmetric elements, this step is strictly necessary to ensure equivalence with the formal definitions and to safeguard the duality laws.

As verified in the erosion operator, neighbor values are always read statically from the original matrix $f$, while the output matrix $g$ acts purely as the register of the accumulated neighborhood maximum. At the image boundaries, initializing the accumulator to `0` exactly emulates the external padding with the neutral element ($-\infty$, or zero in 8-bit representations), ensuring that the physical edges of the scene are dilated in perfect conformity with the standard adopted by OpenCV.

The computational example below illustrates the practical application of a cross-shaped element (`mm::secross()`), validating the consistency between the loop-based logic (`mm::dil0`) and the native industrial method (`mm::dil`).

In [12]:
# Body of mm::dil0() in morph.hpp (reflects the SE before scanning, like np.flip).
import re, pathlib
hpp = pathlib.Path("morph.hpp").read_text()
m = re.search(r"inline Image dil0\(.*?\n\}", hpp, re.S)
print(m.group(0) if m else "(dil0 não encontrada)")

inline Image dil0(const Image& f, SE Bc = SE::box(3)) {
    _require_gray(f, "dil0");
    SE B = Bc.reflected();
    Image g(f.h, f.w, 1);
    for (int y = 0; y < f.h; ++y)
        for (int x = 0; x < f.w; ++x) {
            int mx = 0;
            _viz(f, B, y, x, [&](int vy, int vx, int bv) {
                if (bv != 0 && (int)f.at(vy, vx) > mx) mx = f.at(vy, vx);
            });
            g.at(y, x) = (unsigned char)mx;
        }
    return g;
}


> ### 📝 Note: The Sign Confrontation ($f(x+z)$ vs $f(x-z)$)
>
> Compare the formal definitions of erosion ([Equation 4.3](#eq-04-erosao)) and dilation ([Equation 4.4](#eq-04-dilatacao)). Consider an asymmetric structuring element to the right $\mathbb{B}=\{0,1\}$ (origin and one pixel to the right) applied at position $x=10$.
>
> 1. **In Erosion** ([Equation 4.3](#eq-04-erosao)):
>
>    $$
>    \min\{f(x+z)-b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10+0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10+1)=\mathbf{f(11)}$
>
>    The operator queries the current pixel ($10$) and the pixel to the right ($11$), preserving the original orientation of $\mathbb{B}$.
>
> 2. **In Dilation** ([Equation 4.4](#eq-04-dilatacao)):
>
>    $$
>    \max\{f(x-z)+b(z)\}
>    $$
>
>    * $z=0 \Rightarrow f(10-0)=\mathbf{f(10)}$
>    * $z=1 \Rightarrow f(10-1)=\mathbf{f(9)}$
>
>    Due to the negative sign ($-z$), advancing in the structuring element corresponds to moving backward in the image, causing dilation to query the pixel to the left ($9$).
>
> The `_viz` function, used in `morph.py`, generates neighbors by additive shifts of the form $x+z$. For this reason, the implementation of `mm::dil0` previously reflects the structuring element using `np.flip(B)`. After reflection, the scan based on $x+z$ accesses exactly the same points defined by the theoretical expression $f(x-z)$ of dilation in [Equation 4.4](#eq-04-dilatacao).
>
> For symmetric structuring elements (such as discs, squares, and centered crosses), reflection does not alter the mask. For asymmetric elements, however, this step is essential for the implementation to correctly reproduce the mathematical definition of dilation and preserve the erosion–dilation duality.

> ### 📝 Erosion–dilation duality
>
> Erosion and dilation are **dual by complement**. This means that one operator can be fully obtained from the other, provided that one operates on the complement of the image using the reflected structuring element $\hat{B}$:
>
> $$
> (A \ominus B)^c = A^c \oplus \hat{B} \quad \Longleftrightarrow \quad A \ominus B = (A^c \oplus \hat{B})^c
> $$
>
> Similarly, **dilation can also be obtained from erosion**:
>
> $$
> (A \oplus B)^c = A^c \ominus \hat{B} \quad \Longleftrightarrow \quad A \oplus B = (A^c \ominus \hat{B})^c
> $$
>
> In practical terms, the erosion of an object can be obtained by dilating its complement, followed by complementing the result (and vice versa). In the implementation of the `morph.py` package, the didactic versions `mm::ero0` and `mm::dil0` make this structure explicit through loops, while `mm::ero` and `mm::dil` delegate the operations to OpenCV for greater computational efficiency.

> ### 📝 Boundary conditions and finite images
>
> In classical mathematical morphology, defined over an infinite domain (typically $\mathbb{Z}^2$), this duality is exact. In digital images, however, one works with finite matrices, and the result depends on how pixels located outside the image are treated.
>
> For the duality identities to remain valid, the complement must be defined relative to the same universe, and the boundary conditions adopted for erosion and dilation must be complementary to each other. For example, if erosion assumes that external pixels belong to the object ($255$), then dilation applied to the complement must assume that these same external pixels belong to the background ($0$).
>
> When different filling strategies are used (replication, reflection, constant value, etc.), the theoretical duality may no longer be satisfied exactly in regions near the image borders.

To numerically illustrate the morphological operators and the erosion–dilation duality, [Figure 4.4](#fig-04-ero-dil-didatico) presents a 10×10 binary image processed with an "L"-shaped structuring element. In the implementation of `morph.py`, the origin of $B$ is set at the geometric center of the mask — position $(1,1)$ for a 3×3 *kernel* — and must correspond to an active element for the erosion to behave correctly (as discussed earlier). The structuring element $B_L$ defined below satisfies this condition. [Figure 4.5](#fig-04-sim-04-erosao) complements the analysis with an interactive erosion simulator, allowing visualization of the displacement of the structuring element over the image and identification of the positions where it remains completely contained within the object.

In [13]:
%%writefile tmp/fig_04_ero_dil_didatico.cpp
#define MM_OUT "tmp/fig_04_ero_dil_didatico.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
    //| label: fig-04-ero-dil-didatico
    //| fig-cap: "Erosão e dilatação em imagem binária 10×10 com elemento estruturante 'L' assimétrico 3×3. Validação da dualidade erosão–dilatação."
    //| echo: true
    //| output: true

    mm::Image A(10, 10);
    // Fill the 10x10 image with the pattern (values are 0 or 255)
    unsigned char pattern[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,0,255,255,255,0,0,0,0},
        {0,0,255,255,255,255,255,0,0,0},
        {0,255,255,255,255,255,255,255,0,0},
        {0,255,255,255,255,255,255,255,0,0},
        {0,255,255,255,255,255,255,0,0,0},
        {0,0,255,255,255,255,255,255,0,0},
        {0,0,0,255,255,255,255,0,0,0},
        {0,0,0,0,255,0,0,0,0,0},
        {0,0,0,0,0,0,0,0,0,0}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            A.at(y, x) = pattern[y][x];

    mm::SE B_L{{1,0,0},{1,1,0},{1,1,0}};   // 'L', origem no centro
    mm::SE B_hat{{0,1,1},{0,1,1},{0,0,1}};   // B_L girado 180° (B̂)

    std::cout << "Elemento estruturante B_L:" << std::endl; 
    mm::Image B_L_img = mm::sebox(0);
    // Create the B_L binary image for display
    B_L_img = mm::Image(3, 3);
    B_L_img.at(0,0) = 0; B_L_img.at(0,1) = 255; B_L_img.at(0,2) = 255;
    B_L_img.at(1,0) = 0; B_L_img.at(1,1) = 255; B_L_img.at(1,2) = 255;
    B_L_img.at(2,0) = 0; B_L_img.at(2,1) = 0; B_L_img.at(2,2) = 255;
    std::cout << mm::drawImg(B_L_img) << std::endl;

    std::cout << "Elemento estruturante refletido B̂:" << std::endl;
    mm::Image B_hat_img(3, 3);
    B_hat_img.at(0,0) = 255; B_hat_img.at(0,1) = 0; B_hat_img.at(0,2) = 0;
    B_hat_img.at(1,0) = 255; B_hat_img.at(1,1) = 255; B_hat_img.at(1,2) = 0;
    B_hat_img.at(2,0) = 255; B_hat_img.at(2,1) = 255; B_hat_img.at(2,2) = 0;
    std::cout << mm::drawImg(B_hat_img) << std::endl;

    mm::Image img_ero0 = mm::ero0(A, B_L);

    // Dualidade: (A ⊖ B)ᶜ == Aᶜ ⊕ B̂
    mm::Image A_c    = mm::neg(A);
    mm::Image ero_c  = mm::neg(img_ero0);
    mm::Image dil_Ac = mm::dil0(A_c, B_hat);

    bool equal = true;
    for (int i = 0; i < ero_c.h * ero_c.w; i++) {
        if (ero_c.data[i] != dil_Ac.data[i]) {
            equal = false;
            break;
        }
    }
    std::cout << "Dualidade (A ⊖ B)ᶜ == Aᶜ ⊕ B̂ : " << (equal ? "True" : "False") << std::endl;

    mm::show(
        std::vector<mm::Image>{A, A_c, img_ero0, ero_c, dil_Ac},
        MM_OUT,
        std::vector<std::string>{"A", "Aᶜ", "A ⊖ B", "(A ⊖ B)ᶜ", "Aᶜ ⊕ B̂"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(A, "tmp/fig_04_ero_dil_didatico_0.png");
mm::write(A_c, "tmp/fig_04_ero_dil_didatico_1.png");
mm::write(img_ero0, "tmp/fig_04_ero_dil_didatico_2.png");
mm::write(ero_c, "tmp/fig_04_ero_dil_didatico_3.png");
mm::write(dil_Ac, "tmp/fig_04_ero_dil_didatico_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_ero_dil_didatico.cpp


In [14]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_ero_dil_didatico.cpp -o tmp/fig_04_ero_dil_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_ero_dil_didatico \
  && test -f "tmp/fig_04_ero_dil_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_ero_dil_didatico.png"

Elemento estruturante B_L:
  0 255 255 
  0 255 255 
  0   0 255 

Elemento estruturante refletido B̂:
255   0   0 
255 255   0 
255 255   0 

Dualidade (A ⊖ B)ᶜ == Aᶜ ⊕ B̂ : True
[1] A
[2] Aᶜ
[3] A ⊖ B
[4] (A ⊖ B)ᶜ
[5] Aᶜ ⊕ B̂


In [15]:
mm.show(
    [
        mm.read("tmp/fig_04_ero_dil_didatico_0.png"),
        mm.read("tmp/fig_04_ero_dil_didatico_1.png"),
        mm.read("tmp/fig_04_ero_dil_didatico_2.png"),
        mm.read("tmp/fig_04_ero_dil_didatico_3.png"),
        mm.read("tmp/fig_04_ero_dil_didatico_4.png"),
    ],
    titles=[
        'A',
        'Aᶜ',
        'A ⊖ B',
        '(A ⊖ B)ᶜ',
        'Aᶜ ⊕ B̂',
    ],
    cols=5,
    figsize=(15, 3),
)

<Figure size 2250x450 with 5 Axes>

**Figure 4.4:** Erosão e dilatação em imagem binária 10×10 com elemento estruturante 


In [16]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-erosao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🪨 Simulator: Morphological Erosion</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">A ⊖ B_L · offsets via _viz</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Click on a canvas cell to move the kernel B_L or use the sliders to test containment of contents.</p>

    <!-- Estatísticas Principais -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:16px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">X Position (col)</div>
        <div id="sim-04-erosao_mX" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Y Position (row)</div>
        <div id="sim-04-erosao_mY" style="font-size:16px;font-weight:700;font-family:monospace;color:#2980b9;">4</div>
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:9.5px;color:#8a8371;text-transform:uppercase;margin-bottom:2px;">Pixel Erosion</div>
        <div id="sim-04-erosao_mEro" style="font-size:16px;font-weight:700;font-family:monospace;color:#27ae60;">255</div>
      </div>
    </div>

    <!-- Área Gráfica e Controles Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(260px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Canvas Interativo -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <canvas id="sim-04-erosao_Canvas" style="display:block;max-width:100%;height:auto;border-radius:8px;border:1px solid #e4dcc8;cursor:crosshair;background:#ffffff;margin:0 auto;"></canvas>
        <div style="font-size:10px;color:#8a8371;margin-top:8px;">🖱️ Click a cell to move the kernel B_L</div>
      </div>

      <!-- Painel de Controles e Status -->
      <div style="display:flex;flex-direction:column;gap:12px;">
        
        <!-- Status de Inclusão -->
        <div id="sim-04-erosao_stBox" style="border-radius:10px;border:1.5px solid #27ae60;padding:10px 12px;background:#eafaf1;transition:all 0.15s ease;">
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:4px;">
            <span style="font-size:16px;">🟢</span>
            <div id="sim-04-erosao_stTitle" style="font-size:11px;font-weight:700;color:#27ae60;">Success: contained!</div>
          </div>
          <div id="sim-04-erosao_stDesc" style="font-size:10.5px;color:#27ae60;line-height:1.4;">Pixel receives 1 (255) in the eroded image.</div>
        </div>

        <!-- Sliders de Posição -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Coordinate Controls</span>
          <div style="display:flex;align-items:center;gap:8px;margin-bottom:6px;">
            <label for="sim-04-erosao_slX" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">X (col)</label>
            <input type="range" id="sim-04-erosao_slX" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vX" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
          <div style="display:flex;align-items:center;gap:8px;">
            <label for="sim-04-erosao_slY" style="font-size:10.5px;color:#8a8371;width:40px;flex-shrink:0;">Y (row)</label>
            <input type="range" id="sim-04-erosao_slY" min="0" max="9" step="1" value="4" style="flex:1;cursor:pointer;">
            <span id="sim-04-erosao_vY" style="font-size:11px;font-family:monospace;font-weight:700;min-width:14px;text-align:right;">4</span>
          </div>
        </div>

        <button id="sim-04-erosao_rstBtn" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;width:100%;transition:all 0.15s ease;">↩ Reset Position (4, 4)</button>

        <!-- Kernel B_L Informativo -->
        <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 12px;text-align:center;">
          <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;">Kernel B_L (3×3)</span>
          <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ebf4fd;border:1px solid #2980b9;border-radius:4px;color:#2980b9;">★</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">1</div>
            <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fafaf7;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          </div>
          <div id="sim-04-erosao_offsetInfo" style="font-size:10px;font-family:monospace;color:#5e5a4a;line-height:1.5;background:#ffffff;border:1px solid #e4dcc8;border-radius:6px;padding:6px 8px;margin-top:6px;text-align:left;"></div>
        </div>

      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSim04Ero(root){
    if (!root || root.dataset.sim04EroInit) return;
    root.dataset.sim04EroInit = "1";

    const ero_COLS = 10, ero_ROWS = 10, ero_CELL = 32, ero_PAD = 14;
    const ero_W = ero_COLS * ero_CELL + ero_PAD * 2, ero_H = ero_ROWS * ero_CELL + ero_PAD * 2;
    const ero_cv = root.querySelector('#sim-04-erosao_Canvas');
    ero_cv.width = ero_W; 
    ero_cv.height = ero_H;
    const ero_ctx = ero_cv.getContext('2d');

    const ero_A = [
      [0,0,0,0,0,0,0,0,0,0],
      [0,0,0,1,1,1,0,0,0,0],
      [0,0,1,1,1,1,1,0,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,1,0,0],
      [0,1,1,1,1,1,1,0,0,0],
      [0,0,1,1,1,1,1,1,0,0],
      [0,0,0,1,1,1,1,0,0,0],
      [0,0,0,0,1,0,0,0,0,0],
      [0,0,0,0,0,0,0,0,0,0]
    ];

    const ero_B = [
      [1,0,0],
      [1,1,0],
      [1,1,0]
    ];
    const ero_Bh = 3, ero_Bw = 3;

    function ero_vizOffsets(){
      const offs = [];
      for(let by=0; by<ero_Bh; by++) {
        for(let bx=0; bx<ero_Bw; bx++){
          const dr = by - Math.floor(ero_Bh/2);
          const dc = bx - Math.floor(ero_Bw/2);
          offs.push({dr, dc, bv: ero_B[by][bx], by, bx});
        }
      }
      return offs;
    }
    
    const ero_OFFSETS = ero_vizOffsets();
    const ero_ACTIVE  = ero_OFFSETS.filter(o => o.bv === 1);
    const ero_INACTIVE= ero_OFFSETS.filter(o => o.bv === 0);

    function ero_computeEro(){
      const e = Array.from({length:ero_ROWS}, () => new Array(ero_COLS).fill(0));
      for(let y=0; y<ero_ROWS; y++) {
        for(let x=0; x<ero_COLS; x++) {
          let ok = true;
          for(const {dr,dc,bv} of ero_OFFSETS){
            if(!bv) continue;
            const vy = y + dr, vx = x + dc;
            if(vy < 0 || vy >= ero_ROWS || vx < 0 || vx >= ero_COLS || !ero_A[vy][vx]) { 
              ok = false; 
              break; 
            }
          }
          e[y][x] = ok ? 1 : 0;
        }
      }
      return e;
    }
    
    const ero_ERO = ero_computeEro();
    let ero_cx = 4, ero_cy = 4;

    function ero_updateOffsetInfo(){
      const lines = ero_ACTIVE.map(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        const inBounds = vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS;
        const inA = inBounds && ero_A[vy][vx];
        const mark = inA ? '✓' : '✗';
        return 'B[' + o.by + '][' + o.bx + '] → (' + vy + ',' + vx + ') ' + mark;
      });
      
      root.querySelector('#sim-04-erosao_offsetInfo').innerHTML =
        '<span style="color:#8a8371; font-size:9.5px">offsets ativos @ (y=' + ero_cy + ', x=' + ero_cx + '):</span><br>' +
        lines.map(l => '<span style="color:' + (l.endsWith('✓') ? '#27ae60' : '#c0392b') + '">' + l + '</span>').join('<br>');
    }

    function ero_draw(){
      ero_ctx.clearRect(0, 0, ero_W, ero_H);

      const activeMap = new Map();
      for(const o of ero_ACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) activeMap.set(vy + ',' + vx, ero_A[vy][vx]);
      }
      const inactiveSet = new Set();
      for(const o of ero_INACTIVE){
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        if(vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS) inactiveSet.add(vy + ',' + vx);
      }

      const contained = ero_ACTIVE.every(o => {
        const vy = ero_cy + o.dr, vx = ero_cx + o.dc;
        return vy >= 0 && vy < ero_ROWS && vx >= 0 && vx < ero_COLS && ero_A[vy][vx];
      });

      for(let r=0; r<ero_ROWS; r++) {
        for(let c=0; c<ero_COLS; c++) {
          const x = ero_PAD + c * ero_CELL, y = ero_PAD + r * ero_CELL;
          const key = r + ',' + c;
          const isActive = activeMap.has(key);
          const isInactive = !isActive && inactiveSet.has(key);
          const isCenter = r === ero_cy && c === ero_cx;
          const inA = ero_A[r][c];
          const inE = ero_ERO[r][c];
          
          let fill, stroke, lw=0.5, dash=[];

          if(isActive){
            const inside = activeMap.get(key);
            fill = inside ? 'rgba(243, 156, 18, 0.35)' : 'rgba(192, 57, 43, 0.35)';
            stroke = inside ? '#b9770e' : '#c0392b';
            lw = 2;
          } else if(isCenter){
            fill = 'rgba(41, 128, 185, 0.25)';
            stroke = '#2980b9'; lw = 2; dash = [4,3];
          } else if(isInactive){
            fill = 'rgba(235, 244, 253, 0.4)';
            stroke = '#b9770e'; lw = 1; dash = [3,3];
          } else if(inE){
            fill = 'rgba(39, 174, 96, 0.15)';
            stroke = '#27ae60';
          } else if(inA){
            fill = 'rgba(142, 68, 173, 0.15)';
            stroke = '#8e44ad';
          } else {
            fill = '#fafaf7';
            stroke = '#e4dcc8';
          }

          ero_ctx.setLineDash(dash);
          ero_ctx.fillStyle = fill; 
          ero_ctx.fillRect(x+1, y+1, ero_CELL-2, ero_CELL-2);
          ero_ctx.strokeStyle = stroke; 
          ero_ctx.lineWidth = lw; 
          ero_ctx.strokeRect(x+0.5, y+0.5, ero_CELL-1, ero_CELL-1);
          ero_ctx.setLineDash([]);

          ero_ctx.textAlign = 'center'; 
          ero_ctx.textBaseline = 'middle';
          
          if(isCenter && !isActive){
            ero_ctx.fillStyle = '#2980b9';
            ero_ctx.font = '13px monospace';
            ero_ctx.fillText('★', x+ero_CELL/2, y+ero_CELL/2);
          } else if(isActive){
            const inside = activeMap.get(key);
            ero_ctx.fillStyle = inside ? '#7d5a00' : '#78281f';
            ero_ctx.font = 'bold 11px monospace';
            ero_ctx.fillText(inA ? '1' : '0', x+ero_CELL/2, y+ero_CELL/2);
          } else if(inA){
            ero_ctx.fillStyle = '#4a235a';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('1', x+ero_CELL/2, y+ero_CELL/2);
          } else {
            ero_ctx.fillStyle = '#8a8371';
            ero_ctx.font = '11px monospace';
            ero_ctx.fillText('0', x+ero_CELL/2, y+ero_CELL/2);
          }

          if(r===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(c, x+ero_CELL/2, ero_PAD/2);}
          if(c===0){ero_ctx.fillStyle='#8a8371'; ero_ctx.font='9px monospace'; ero_ctx.textAlign='center'; ero_ctx.fillText(r, ero_PAD/2, y+ero_CELL/2);}
        }
      }

      root.querySelector('#sim-04-erosao_mX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_mY').textContent = ero_cy;
      const ev = ero_ERO[ero_cy][ero_cx];
      const mEro = root.querySelector('#sim-04-erosao_mEro');
      mEro.textContent = ev ? '255' : '0';
      mEro.style.color = ev ? '#27ae60' : '#c0392b';
      root.querySelector('#sim-04-erosao_vX').textContent = ero_cx;
      root.querySelector('#sim-04-erosao_vY').textContent = ero_cy;
      root.querySelector('#sim-04-erosao_slX').value = ero_cx;
      root.querySelector('#sim-04-erosao_slY').value = ero_cy;

      const box = root.querySelector('#sim-04-erosao_stBox');
      const title = root.querySelector('#sim-04-erosao_stTitle');
      const desc = root.querySelector('#sim-04-erosao_stDesc');
      
      if(contained){
        box.style.background = '#eafaf1'; box.style.borderColor = '#27ae60';
        title.textContent = 'Sucesso: contido!'; title.style.color = '#27ae60';
        desc.textContent = 'Pixel recebe 1 (255) na imagem erodida.'; desc.style.color = '#27ae60';
      } else {
        box.style.background = '#fdecea'; box.style.borderColor = '#c0392b';
        title.textContent = 'Aviso: B_L sai do objeto!'; title.style.color = '#c0392b';
        desc.textContent = 'Pixel recebe 0 na imagem erodida.'; desc.style.color = '#c0392b';
      }
      
      ero_updateOffsetInfo();
    }

    ero_cv.addEventListener('click', function(e){
      const rect = ero_cv.getBoundingClientRect();
      const sx = ero_W / rect.width, sy = ero_H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - ero_PAD) / ero_CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - ero_PAD) / ero_CELL);
      if(c >= 0 && c < ero_COLS && r >= 0 && r < ero_ROWS){
        ero_cx = c; ero_cy = r; 
        ero_draw();
      }
    });
    
    root.querySelector('#sim-04-erosao_slX').addEventListener('input', function(){ ero_cx = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_slY').addEventListener('input', function(){ ero_cy = +this.value; ero_draw(); });
    root.querySelector('#sim-04-erosao_rstBtn').addEventListener('click', function(){ ero_cx = 4; ero_cy = 4; ero_draw(); });
    
    ero_draw();
  }

  function tryInitSim04Ero(){
    var root = document.getElementById('sim-04-erosao');
    if (root) initSim04Ero(root); else setTimeout(tryInitSim04Ero, 200);
  }
  tryInitSim04Ero();
})();
</script>
</div>
""")

**Figure 4.5:** Simulator: Morphological Erosion (A ⊖ B_L)


<figure id="fig-04-sim-04-erosao">
  <img src="imagens/fig-04-sim-04-erosao.png" alt=" Simulator: Morphological Erosion (A ⊖ B_L) " style="max-width:80%" />
  <figcaption><strong>Figure 4.5:</strong>  Simulator: Morphological Erosion (A ⊖ B_L) </figcaption>
</figure>

### 4.3.2 Opening and Closing

Combining erosion and dilation yields two operators of great practical utility: **opening** and **closing**, defined by Equations [Equation 4.5](#eq-04-abertura) and [Equation 4.6](#eq-04-fechamento). Their main effects are summarized in [Table 4.1](#tbl-04-open-close).

**Opening** (*opening*) — erosion followed by dilation using the same $B$:

<a id="eq-04-abertura"></a>
$$
A \circ B = (A \ominus B) \oplus B \tag{4.5}
$$


**Closing** (*closing*) — dilation followed by erosion using the same $B$:

<a id="eq-04-fechamento"></a>
$$
A \bullet B = (A \oplus B) \ominus B \tag{4.6}
$$


In practice, `mm::open` and `mm::close` apply the same structuring element in both stages. For symmetric structuring elements (the most common ones), this implementation coincides with the mathematical definition presented above.

<a id="tbl-04-open-close"></a>

**Tabela 4.1:** Properties of opening and closing.

| Operator | Sequence | Main effect |
|:--------:|:----------|:-----------------|
| Opening $A \circ B$ | erosion → dilation | Removes structures unable to contain the structuring element; smooths external contours |
| Closing $A \bullet B$ | dilation → erosion | Fills holes smaller than $B$; smooths internal contours |


**Important property:** both are **idempotent**. For example,

$$
(A \circ B) \circ B = A \circ B,
$$

that is, after the first application, further applications of the same operator no longer alter the result.

#### 4.3.2.1 Implementation of opening and closing

Unlike erosion and dilation, opening and closing do not introduce
new computational mechanisms. Both are obtained by the sequential composition
of the primitive operators already presented:

```python
def open0(f, B):
    return mm.dil0(mm.ero0(f, B), B)

def close0(f, B):
    return mm.ero0(mm.dil0(f, B), B)
```

The `mm::open` function delegates the operation to `mm::open(f, B)`,
while `mm::close` uses `mm::close(f, B)`, producing
the same result more efficiently.

Opening inherits from erosion the ability to remove structures smaller than the
structuring element and from dilation the partial restoration of preserved
regions. Closing performs the reverse process: it first expands the
objects and then restores their original dimensions, filling gaps and
holes smaller than the structuring element.

#### 4.3.2.2 Alternating Sequential Filter

In practice, opening and closing are often applied in sequence to simultaneously remove external noise and fill internal holes. The `mm::asf` (*Alternating Sequential Filter*) function generalizes this strategy by applying openings and closings alternately with progressively larger structuring elements. The available sequences are presented in [Table 4.2](#tbl-04-asf).

<a id="tbl-04-asf"></a>

**Tabela 4.2:** Sequences of the alternating sequential filter `mm::asf`.

| Sequence | Order                             | Typical use                                                |
| :------: | :-------------------------------- | :--------------------------------------------------------- |
|   `'OC'` | opening → closing                 | removes external noise before filling small holes          |
|   `'CO'` | closing → opening                 | fills small holes before removing external noise           |
|  `'OCO'` | opening → closing → opening       | emphasizes the removal of external noise                   |
|  `'COC'` | closing → opening → closing       | emphasizes the filling of holes and gaps                   |


The parameter `n` controls the number of scales used by the filter. In each iteration $i$, the structuring element is enlarged by Minkowski addition (`mm::sesum(b, i)`), producing a sequence of increasingly comprehensive morphological filters. Unlike a single opening or closing with a large structuring element, the ASF performs progressive smoothing across multiple scales, better preserving the geometry of relevant objects while eliminating smaller structures. [Figure 4.6](#fig-04-open-close) presents an example of applying opening, closing, and ASF to the coins image.

In [17]:
%%writefile tmp/fig_04_open_close.cpp
#define MM_OUT "tmp/fig_04_open_close.png"
// Compile: g++ -std=c++17 -I. -o program program.cpp -lcurl -lpng -lz

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_clahe = mm::_read_state("tmp/state/img_clahe_12.png");
// [pdi:state-io:end]

    //| label: fig-04-open-close
    //| fig-cap: "Opening, closing, composition and alternating sequential filter applied to Otsu binarization of coins (pre-processed by contrast enhancement). Structuring element: 13×13 disk."
    //| echo: true
    //| output: true

    mm::Image img_bin = mm::threshold(img_clahe);
    mm::Image B_disk = mm::sedisk(19);

    mm::Image img_open = mm::open(img_bin, B_disk);             // erosion → dilation
    mm::Image img_close = mm::close(img_bin, B_disk);            // dilation → erosion
    mm::Image img_oc = mm::close(img_open, B_disk);           // opening followed by closing
    mm::Image img_asf = mm::asf(img_bin, "OC", mm::sedisk(3), 7);
    // ASF: 3×3 base disk, grows each iteration

    mm::show(
        std::vector<mm::Image>{img_bin, img_open, img_close, img_oc, img_asf},
        MM_OUT,
        std::vector<std::string>{"Otsu Binarization", "Opening (A∘B)", "Closing (A∙B)",
                "Opening→Closing", "ASF-OC (n=7)"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_bin, "tmp/fig_04_open_close_0.png");
mm::write(img_open, "tmp/fig_04_open_close_1.png");
mm::write(img_close, "tmp/fig_04_open_close_2.png");
mm::write(img_oc, "tmp/fig_04_open_close_3.png");
mm::write(img_asf, "tmp/fig_04_open_close_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_open_close.cpp


In [18]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_open_close.cpp -o tmp/fig_04_open_close -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_open_close \
  && test -f "tmp/fig_04_open_close.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_open_close.png"

[1] Otsu Binarization
[2] Opening (A∘B)
[3] Closing (A∙B)
[4] Opening→Closing
[5] ASF-OC (n=7)


In [19]:
mm.show(
    [
        mm.read("tmp/fig_04_open_close_0.png"),
        mm.read("tmp/fig_04_open_close_1.png"),
        mm.read("tmp/fig_04_open_close_2.png"),
        mm.read("tmp/fig_04_open_close_3.png"),
        mm.read("tmp/fig_04_open_close_4.png"),
    ],
    titles=[
        'Binarização Otsu',
        'Abertura (A∘B)',
        'Fechamento (A∙B)',
        'Abertura→Fechamento',
        'ASF-OC (n=7)',
    ],
    cols=5,
    figsize=(15, 12),
)

<Figure size 2250x1800 with 5 Axes>

**Figure 4.6:** Abertura, fechamento, composição e filtro sequencial alternado aplicados à binarização Otsu das moedas (pré-processadas por realce de contraste). Elemento estruturante: disco 13×13.


### 4.3.3 Geodesic Operators

Geodesic operators introduce an additional constraint to classical morphological operators through a control image called the **mask** $g$. Instead of allowing erosion or dilation to propagate freely across the image, the result of each iteration is limited pointwise by the mask values, restricting the operation's evolution to permitted regions.

#### 4.3.3.1 Geodesic Dilation

The **geodesic dilation** of a marker image $f$ under a mask image $g$, using a flat structuring element $b$, is defined by:

<a id="eq-04-cdil"></a>
$$
f \oplus_g b = (f \oplus b) \wedge g, \tag{4.7}
$$


where $\wedge$ represents the pointwise minimum.

In other words, a conventional dilation is initially performed on the marker, and then the result is constrained by the mask $g$. In this way, the propagation can never exceed the regions permitted by the mask.

The classical formulation of geodesic dilation assumes that the marker is contained within the mask, that is, $f \le g$, ensuring that the evolution of the operation always remains bounded by the mask.

#### 4.3.3.2 Implementation of geodesic dilation

The `mm::cdil` function directly implements this operator and allows executing multiple consecutive iterations:

```python
def cdil(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Dilatação geodésica do marcador f sob a máscara g."""
    y = f.copy()
    for _ in range(n):
        y = np.minimum(mm.dil(y, b), g)
    return y
```

The `np.minimum(mm::dil(y, b), g)` instruction exactly implements the mathematical definition of geodesic dilation, that is, $(y \oplus b)\wedge g$.

When $n=1$, the function performs a single geodesic dilation. For $n>1$, the result of each step becomes the marker for the next step, producing a progressive propagation controlled by the mask.

The interactive simulator [Figure 4.7](#fig-04-sim-04-cdil) allows you to follow, step by step, the propagation of the marker $f$ along the corridors of the maze. At each iteration of `mm::cdil`, the dilation front advances to the neighboring free cells — those where $g = 1$ — while the walls ($g = 0$) remain impassable. The number of steps required for the marker to reach the exit corresponds exactly to the geodesic length of the shortest path within the mask, highlighting the direct connection between iterated geodesic dilation and the notion of distance in graphs.

In [20]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-cdil" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🗺️ Simulator: Geodesic Dilation in the Maze</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">δ_g^(n)(f)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    
    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:14px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:14px;">
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#2c2c2a;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Wall</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#eafaf1;border:1px solid #a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Free path (g)</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#27ae60;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Marker f</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#a3e4d7;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Propagation</span></div>
      <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;background:#fef9e7;border:1px solid #f8c471;border-radius:3px;"></div><span style="font-size:11px;color:#8a8371;">Exit</span></div>
    </div>

    <!-- Canvas Centralizado -->
    <div style="display:flex;justify-content:center;margin-bottom:14px;">
      <canvas id="sim-04-cdil_Canvas" width="435" height="435" style="width:100%;max-width:435px;display:block;border-radius:12px;border:1px solid #e4dcc8;background:#ffffff;"></canvas>
    </div>

    <!-- Controles e Informações do Passo -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;gap:6px;flex-wrap:wrap;">
        <button id="sim-04-cdil_btnReset" style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">↺ Restart</button>
        <button id="sim-04-cdil_btnPrev" disabled style="padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">‹ Previous</button>
        <button id="sim-04-cdil_btnNext" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">Next ›</button>
        <button id="sim-04-cdil_btnPlay" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">▶ Animate</button>
      </div>
      <div id="sim-04-cdil_info" style="font-size:11px;font-family:monospace;color:#26241d;flex:1;min-width:180px;text-align:right;">Step 0 — initial marker f (entry)</div>
    </div>

  </div>
</div>

<script>
(function() {
  function initSim04Cdil(root){
    if (!root || root.dataset.sim04CdilInit) return;
    root.dataset.sim04CdilInit = "1";

    const G = [
      [0,0,0,0,0,0,0,0,0,0,0,0],
      [0,1,1,1,0,1,1,1,1,1,1,0],
      [0,0,0,1,0,1,0,0,0,0,1,0],
      [0,1,0,1,0,1,0,1,1,0,1,0],
      [0,1,0,1,1,1,0,1,0,0,1,0],
      [0,1,0,0,0,0,0,1,0,1,1,0],
      [0,1,1,1,1,1,1,1,0,1,0,0],
      [0,0,0,0,0,0,1,0,0,1,0,0],
      [0,1,1,1,1,0,1,1,1,1,0,0],
      [0,1,0,0,1,0,0,0,0,1,0,0],
      [0,1,1,1,1,1,1,1,1,1,0,0],
      [0,0,0,0,0,0,0,0,0,0,0,0]
    ];
    const ROWS = G.length, COLS = G[0].length;
    const START = [1,1];
    const EXIT  = [10,9];

    function dilate(prev, mask) {
      const next = prev.map(r => [...r]);
      const dirs = [[-1,0],[1,0],[0,-1],[0,1]];
      for (let r=0; r<ROWS; r++)
        for (let c=0; c<COLS; c++)
          if (prev[r][c]) {
            for (const [dr,dc] of dirs) {
              const nr=r+dr, nc=c+dc;
              if (nr>=0&&nr<ROWS&&nc>=0&&nc<COLS&&mask[nr][nc])
                next[nr][nc]=1;
            }
          }
      return next;
    }

    function emptyGrid() { return Array.from({length:ROWS},()=>new Array(COLS).fill(0)); }

    let steps = [];
    function buildSteps() {
      steps = [];
      let cur = emptyGrid();
      cur[START[0]][START[1]] = 1;
      steps.push(cur.map(r=>[...r]));
      let prev = null;
      while (JSON.stringify(cur) !== JSON.stringify(prev)) {
        prev = cur.map(r=>[...r]);
        cur = dilate(cur, G);
        steps.push(cur.map(r=>[...r]));
        if (steps.length > 80) break;
      }
    }

    buildSteps();
    let idx = 0, playing = false, timer = null;

    const cv = root.querySelector('#sim-04-cdil_Canvas');
    const ctx = cv.getContext('2d');

    const C = {
      wall:    '#2c2c2a',
      free:    '#eafaf1',
      freeBd:  '#a3e4d7',
      marker:  '#27ae60',
      trail:   '#a3e4d7',
      exit:    '#fef9e7',
      exitBd:  '#f8c471',
      bg:      '#fafaf7',
      text:    '#5e5a4a',
      grid:    '#e4dcc8',
    };

    function draw() {
      const W = cv.width, H = cv.height;
      const cw = W / COLS, ch = H / ROWS;
      ctx.clearRect(0,0,W,H);
      ctx.fillStyle = C.bg;
      ctx.fillRect(0,0,W,H);

      const cur = steps[idx];
      const prev = idx > 0 ? steps[idx-1] : null;

      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const x = c*cw, y = r*ch;
          const isExit = r===EXIT[0]&&c===EXIT[1];

          if (!G[r][c]) {
            ctx.fillStyle = C.wall;
            ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
          } else {
            if (isExit) {
              ctx.fillStyle = C.exit;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.exitBd;
              ctx.lineWidth = 1.5;
              ctx.strokeRect(x+1.5,y+1.5,cw-3,ch-3);
            } else {
              ctx.fillStyle = C.free;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
              ctx.strokeStyle = C.freeBd;
              ctx.lineWidth = 0.5;
              ctx.strokeRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (prev && prev[r][c]) {
              ctx.fillStyle = C.trail;
              ctx.fillRect(x+0.5,y+0.5,cw-1,ch-1);
            }
            if (cur[r][c]) {
              ctx.fillStyle = C.marker;
              ctx.fillRect(x+2,y+2,cw-4,ch-4);
            }
          }
        }
      }

      ctx.strokeStyle = C.grid;
      ctx.lineWidth = 0.5;
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(0,r*ch); ctx.lineTo(W,r*ch); ctx.stroke(); }
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(c*cw,0); ctx.lineTo(c*cw,H); ctx.stroke(); }

      ctx.fillStyle = '#ffffff';
      ctx.font = 'bold 11px monospace';
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      const [sr,sc] = START;
      ctx.fillText('E', (sc+0.5)*cw, (sr+0.5)*ch);
      const [er,ec] = EXIT;
      ctx.fillStyle = '#b9770e';
      ctx.fillText('S', (ec+0.5)*cw, (er+0.5)*ch);

      const reached = steps[idx][EXIT[0]][EXIT[1]];
      let msg = '';
      if (idx === 0) msg = 'Passo 0 — marcador inicial f (entrada)';
      else if (reached) msg = 'Passo ' + idx + ' — saída alcançada! 🎉';
      else msg = 'Passo ' + idx + ' — propagação geodésica';

      root.querySelector('#sim-04-cdil_info').textContent = msg;
      root.querySelector('#sim-04-cdil_btnPrev').disabled = idx === 0;
      root.querySelector('#sim-04-cdil_btnNext').disabled = idx === steps.length-1;
    }

    function step(d) {
      idx = Math.max(0, Math.min(steps.length-1, idx+d));
      draw();
    }

    function reset() {
      stopPlay();
      idx = 0;
      draw();
    }

    function togglePlay() {
      playing ? stopPlay() : startPlay();
    }

    function startPlay() {
      playing = true;
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '⏸ Pausar';
      timer = setInterval(() => {
        if (idx >= steps.length-1) { stopPlay(); return; }
        idx++;
        draw();
      }, 350);
    }

    function stopPlay() {
      playing = false;
      clearInterval(timer);
      root.querySelector('#sim-04-cdil_btnPlay').textContent = '▶ Animar';
    }

    root.querySelector('#sim-04-cdil_btnReset').addEventListener('click', reset);
    root.querySelector('#sim-04-cdil_btnPrev').addEventListener('click', () => step(-1));
    root.querySelector('#sim-04-cdil_btnNext').addEventListener('click', () => step(1));
    root.querySelector('#sim-04-cdil_btnPlay').addEventListener('click', togglePlay);

    draw();
  }

  function tryInitSim04Cdil(){
    var root = document.getElementById('sim-04-cdil');
    if (root) initSim04Cdil(root); else setTimeout(tryInitSim04Cdil, 200);
  }
  tryInitSim04Cdil();
})();
</script>
""")

**Figure 4.7:** Interactive simulator of geodesic dilation: the marker f (green) propagates step by step through the free paths of mask g, without crossing walls.


<figure id="fig-04-sim-04-cdil">
  <img src="imagens/fig-04-sim-04-cdil.png" alt=" Interactive simulator of geodesic dilation: the marker f (green) propagates step by step through the free paths of mask g, without crossing walls. " style="max-width:80%" />
  <figcaption><strong>Figure 4.7:</strong>  Interactive simulator of geodesic dilation: the marker f (green) propagates step by step through the free paths of mask g, without crossing walls. </figcaption>
</figure>

#### 4.3.3.3 Geodesic Erosion

Dually, the **geodesic erosion** of a marker image $f$ under a mask image $g$ is defined by:

<a id="eq-04-cero"></a>
$$
f \ominus_g b = (f \ominus b) \vee g, \tag{4.8}
$$


where $\vee$ represents the pointwise maximum operator.

In this case, the conventional erosion of the marker is followed by a lower constraint imposed by the mask. Thus, no pixel of the result can assume a value lower than the corresponding pixel of the mask.

The classical formulation of geodesic erosion presupposes the dual condition

$$
f \ge g,
$$

so that the mask acts as a lower bound throughout the entire process.

#### 4.3.3.4 Implementation of geodesic erosion

The static function `mm::cero` implements this operator:

```python
staticmethod
def cero(f, g, b=np.zeros((3,3),dtype='uint8'), n=1):
    """Erosão geodésica do marcador f sob a máscara g."""
    y = f.copy()
    for _ in range(n):
        y = np.maximum(mm.ero(y, b), g)
    return y
```

The instruction `np.maximum(mm::ero(y, b), g)` directly implements the expression $(y \ominus b)\vee g$.

As with geodesic dilation, the parameter $n$ defines how many successive geodesic erosions will be computed before returning the final image.

> ### 📝 Relation to morphological reconstruction
>
> The morphological reconstruction presented in the next section is obtained by iteratively applying geodesic dilation (`mm::cdil`) until a fixed point is reached, that is, until no cell changes value between two consecutive iterations. In other words, reconstruction consists of a sequence of successive geodesic dilations that propagate within the mask until no further changes occur.
>
> Dually, it is also possible to define reconstructions based on geodesic erosion through successive applications of `mm::cero`.

#### 4.3.3.5 Example: propagation in a maze via duality

[Figure 4.8](#fig-04-cero-labirinto) illustrates the resolution of the connectivity problem in a maze using morphological duality through the functions `mm::cero` and `mm::suprec`.

Instead of propagating a marker through the free corridors using geodesic dilations, the problem is formulated in the complementary domain. Initially, the original mask is inverted,

$$
g = 1 - g_{orig},
$$

so that the walls now assume value 1 and the corridors value 0. Similarly, the marker is constructed in this same complementary domain, containing a single value 0 at the maze entrance position and value 1 in all other pixels.

Using the cross-shaped structuring element (`mm::secross()`), the geodesic erosion acts on the complemented marker. At each iteration of `mm::cero`, the region connected to the initial marker undergoes successive erosions, while the mask imposes a lower limit that prevents propagation through the maze walls.

The intermediate images show evolution states after different numbers of iterations (`n=5`, `n=12`, and `n=22`). The final result is obtained by the geodesic reconstruction by erosion (`mm::suprec`), which applies successive geodesic erosions until reaching a fixed point, that is, a situation in which no further change occurs between two consecutive iterations.

In the complementary domain, the reconstructed region corresponds exactly to the set of corridors connected to the maze entrance. Thus, connectivity between entrance and exit can be determined directly from the reconstructed image.

In [21]:
%%writefile tmp/fig_04_cero_labirinto.cpp
#define MM_OUT "tmp/fig_04_cero_labirinto.png"
//| label: fig-04-cero-labirinto
//| fig-cap: "Resolução de labirinto no domínio complementar: com máscara e marcador invertidos, *mm::cero* e *mm::suprec* propagam a onda geodésica pelos corredores."
//| echo: true
//| output: true
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
    // Máscara já invertida (255 = parede, 0 = corredor)
    mm::Image g(10, 10);
    unsigned char g_data[10][10] = {
        {255,  0,255,255,255,255,255,255,255,255},
        {255,  0,  0,  0,  0,  0,255,  0,  0,  0},
        {255,255,255,255,255,  0,255,  0,255,  0},
        {255,  0,  0,  0,255,  0,  0,  0,255,  0},
        {255,  0,255,  0,255,255,255,255,255,  0},
        {255,  0,255,  0,  0,  0,  0,  0,  0,  0},
        {255,  0,255,255,255,255,255,255,  0,255},
        {255,  0,  0,  0,  0,  0,  0,255,  0,255},
        {255,255,255,255,255,255,  0,  0,  0,255},
        {255,255,255,255,255,255,255,255,  0,255}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            g.at(y, x) = g_data[y][x];

    mm::Image f(10, 10);
    std::fill(f.data.begin(), f.data.end(), 255);
    f.at(0, 1) = 0;                      // semente (0) no corredor de entrada

    mm::Image B_cruz = mm::secross();

    mm::Image passo_5    = mm::cero(f, g, B_cruz, 5);
    mm::Image passo_12   = mm::cero(f, g, B_cruz, 12);
    mm::Image passo_22   = mm::cero(f, g, B_cruz, 22);
    mm::Image ponto_fixo = mm::suprec(f, g, B_cruz);

    mm::show(std::vector<mm::Image>{g, f, passo_5, passo_12, passo_22, ponto_fixo},
             MM_OUT,
             std::vector<std::string>{"Mascara (~g)", "Marcador (~f)", "n=5", "n=12", "n=22", "~mm.suprec"},
             6);
    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(g, "tmp/fig_04_cero_labirinto_0.png");
mm::write(f, "tmp/fig_04_cero_labirinto_1.png");
mm::write(passo_5, "tmp/fig_04_cero_labirinto_2.png");
mm::write(passo_12, "tmp/fig_04_cero_labirinto_3.png");
mm::write(passo_22, "tmp/fig_04_cero_labirinto_4.png");
mm::write(ponto_fixo, "tmp/fig_04_cero_labirinto_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_cero_labirinto.cpp


In [22]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_cero_labirinto.cpp -o tmp/fig_04_cero_labirinto -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_cero_labirinto \
  && test -f "tmp/fig_04_cero_labirinto.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_cero_labirinto.png"

[1] Mascara (~g)
[2] Marcador (~f)
[3] n=5
[4] n=12
[5] n=22
[6] ~mm.suprec


In [23]:
mm.show(
    [
        mm.read("tmp/fig_04_cero_labirinto_0.png"),
        mm.read("tmp/fig_04_cero_labirinto_1.png"),
        mm.read("tmp/fig_04_cero_labirinto_2.png"),
        mm.read("tmp/fig_04_cero_labirinto_3.png"),
        mm.read("tmp/fig_04_cero_labirinto_4.png"),
        mm.read("tmp/fig_04_cero_labirinto_5.png"),
    ],
    titles=[
        'Mascara (~g)',
        'Marcador (~f)',
        'n=5',
        'n=12',
        'n=22',
        '~mm.suprec',
    ],
    cols=6,
)

<Figure size 4500x750 with 6 Axes>

**Figure 4.8:** Resolução de labirinto no domínio complementar: com máscara e marcador invertidos, *mm::cero* e *mm::suprec* propagam a onda geodésica pelos corredores.


### 4.3.4 Morphological Reconstruction

**Morphological reconstruction** propagates a **marker** image $f$ within a **mask** image $g$, ensuring that the result never exceeds the intensity values imposed by the mask. The fundamental operator that enables this contained propagation is the **geodesic dilation**, defined by [Equation 4.7](#eq-04-cdil).

Reconstruction is obtained by the iterative application of this conditioned dilation. Initially, the marker is bounded by the mask to establish the initial state:

<a id="eq-04-infrec"></a>
$$
X^{(0)} = f \wedge g,
$$

and the subsequent iterations are defined recursively by:

$$
X^{(k)} = (X^{(k-1)} \oplus b) \wedge g.
$$

The sequence grows monotonically until it reaches a fixed point, producing the **morphological reconstruction by dilation** (also known in the literature as *inf-reconstruction*):

$$
R_g^\delta(f) = \lim_{k\to\infty} X^{(k)} = X^{(k)} \quad \text{when} \quad X^{(k)} = X^{(k-1)}. \tag{4.9}
$$


The iterative ascent is halted as soon as stability is achieved, that is, when two consecutive iterations produce matrices with absolutely identical values.

#### 4.3.4.1 Implementation of morphological reconstruction

The didactic routine `mm::infrec` directly implements the iterative fixed-point algorithm. Initially, the initial effective marker $X^{(0)}$ is determined by the operation `np.minimum(f, g)`. To ensure that the verification loop executes the first pass without triggering false premature convergences, the control variable of the previous iteration (`y1`) is initialized filled with a sentinel value outside the data domain (or simply with a matrix that forces the first execution).

```python
def infrec(f, g, b=np.zeros((3,3), dtype='uint8')):
    """Inf-reconstruction: dilates the marker (f ∧ g) until convergence under the mask g."""
    y = np.minimum(f, g)
    # Initialize y1 with impossible values to force entry into the loop
    y1 = np.full_like(f, 256, dtype=np.int16) 
    while not np.array_equal(y, y1):
        y1 = y.copy()
        # Apply geodesic dilation: (y ⊕ b) ∧ g
        y = np.minimum(mm.dil(y, b), g)
    return y.astype('uint8')
```

Inside the `while` loop, the variable `y` stores the current estimate of the reconstruction $X^{(k)}$, while `y1` preserves the image from the immediately preceding stage $X^{(k-1)}$. The conditional control statement `np.minimum(mm::dil(y, b), g)` faithfully translates the theoretical geodesic dilation, where the conventional morphological expansion commanded by OpenCV is immediately "pruned" and limited by the intensity barriers of the mask $g$. The loop ceases when no pixel modification is recorded between steps.

#### 4.3.4.2 Advantages of Morphological Reconstruction

Morphological reconstruction is significantly more robust than conventional opening because it can remove unwanted structures without distorting or altering the morphology of the objects that should be preserved.

While classical opening smooths corners, eliminates tips, and deforms contours due to the rigid geometric imposition of the structuring element, geodesic reconstruction uses the mask to accurately recover the original boundaries and shapes of objects that have connectivity with the original marker.

In intuitive terms, the marker acts as a contagion seed that expands progressively, but only travels through the regions allowed by the mask. Components that have no intersection with the marker will never be reconstructed (being eliminated), while components touched by the seed expand until fully restoring their original geometry.

This discriminatory and conservative behavior is illustrated in [Figure 4.9](#fig-04-reconstrucao-didatica), using the cross-shaped structuring element presented in [Figure 4.3](#fig-04-elemento-cruz).

In [24]:
%%writefile tmp/fig_04_reconstrucao_didatica.cpp
#define MM_OUT "tmp/fig_04_reconstrucao_didatica.png"
// Compile with: g++ -std=c++17 -o program program.cpp -I. -lstdc++ -lm

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

//| label: fig-04-reconstrucao-didatica
//| fig-cap: "*Pipeline* de Reconstrução Morfológica por Dilatação Condicionada: a máscara contém dois objetos, o marcador isola apenas o núcleo do objeto principal, e as iterações reconstroem sua forma exata até a convergência."
//| echo: true
//| output: true

int main() {
    mm::Image g(10, 10);
    unsigned char g_data[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,0,0,0,1,1,0},
        {0,1,1,1,1,0,0,1,1,0},
        {0,1,1,1,1,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,0},
        {0,1,1,1,0,0,0,0,0,0},
        {0,0,1,1,1,0,0,1,0,0},
        {0,0,1,1,1,0,0,1,1,0},
        {0,0,0,1,0,0,0,0,0,0},
        {0,0,0,0,0,0,0,0,0,0}};
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            g.at(y, x) = g_data[y][x] * 255;

    mm::Image B_cruz = mm::secross();
    mm::Image f = mm::ero(g, mm::sebox());  // marcador: núcleo do objeto principal

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = f;
    for (int i = 1; i <= 5; i++) {
        img_atual = mm::cdil(img_atual, g, B_cruz);
        iteracoes.push_back(img_atual);
        titulos.push_back("Dilatação Cond. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_reconstruida = mm::infrec(f, g, B_cruz);

    // Check if reconstruction equals last iteration
    bool equal = true;
    for (int i = 0; i < img_reconstruida.h * img_reconstruida.w * img_reconstruida.channels; i++) {
        if (img_reconstruida.data[i] != iteracoes.back().data[i]) {
            equal = false;
            break;
        }
    }
    std::cout << "✅ Estabilidade na iteração 5: " << (equal ? "True" : "False") << std::endl;

    std::vector<mm::Image> imgs;
    imgs.push_back(g);
    imgs.push_back(f);
    imgs.insert(imgs.end(), iteracoes.begin(), iteracoes.end());
    imgs.push_back(img_reconstruida);

    std::vector<std::string> titles;
    titles.push_back("Máscara (g)");
    titles.push_back("Marcador (f)");
    titles.insert(titles.end(), titulos.begin(), titulos.end());
    titles.push_back("Reconstrução R_g(f)");

    mm::show(imgs, MM_OUT, titles, 8);

    return 0;
}

Overwriting tmp/fig_04_reconstrucao_didatica.cpp


In [25]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_reconstrucao_didatica.cpp -o tmp/fig_04_reconstrucao_didatica -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_reconstrucao_didatica \
  && test -f "tmp/fig_04_reconstrucao_didatica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_reconstrucao_didatica.png"

✅ Estabilidade na iteração 5: True
[1] Máscara (g)
[2] Marcador (f)
[3] Dilatação Cond. (n=1)
[4] Dilatação Cond. (n=2)
[5] Dilatação Cond. (n=3)
[6] Dilatação Cond. (n=4)
[7] Dilatação Cond. (n=5)
[8] Reconstrução R_g(f)


In [26]:
mm.show(mm.read("tmp/fig_04_reconstrucao_didatica.png"), figsize=(18, 3))

<Figure size 2700x450 with 1 Axes>

**Figure 4.9:** *Pipeline* de Reconstrução Morfológica por Dilatação Condicionada: a máscara contém dois objetos, o marcador isola apenas o núcleo do objeto principal, e as iterações reconstroem sua forma exata até a convergência.


### 4.3.5 Hole Filling and Border Removal

Two operators based on morphological reconstruction complete the binary cleaning *pipeline*. Their main characteristics are summarized in [Table 4.3](#tbl-04-clohole-edgeoff).

**Hole filling** (`mm::clohole`) removes cavities completely surrounded by the object, regardless of size, without altering the external contours. The procedure operates on the complement of the image, using as a marker a restriction of the frame to the background:

<a id="eq-04-clohole"></a>
$$
\text{clohole}(f) = \bigl(R_{f^c}^\delta(\text{frame}(f) \wedge f^c)\bigr)^c \tag{4.10}
$$



In operational terms, the external background is first reconstructed, and then complementation is applied to recover the objects with filled holes.

**Border object removal** (`mm::edgeoff`) eliminates all objects that touch the image border, preserving only fully internal components. The marker is obtained by the intersection between the frame and the objects of the image:

<a id="eq-04-edgeoff"></a>
$$
\text{edgeoff}(f) = f \setminus R_f^\delta(\text{frame}(f) \wedge f) \tag{4.11}
$$


<a id="tbl-04-clohole-edgeoff"></a>

**Tabela 4.3:** Comparison between the *clohole* and *edgeoff* operators.

| Operator | Marker | Mask | Effect |
|:---------|:---------|:--------|:-------|
| `mm::clohole` | *frame* restricted to the background ($f^c$) | $f^c$ | Fills internal holes |
| `mm::edgeoff` | *frame* restricted to the object ($f$) | $f$ | Removes objects connected to the border |


The step-by-step evolution of these geodesic transformations can be followed in the figures below. [Figure 4.10](#fig-04-clohole-didatico) illustrates the controlled flooding mechanism of the `mm::clohole` operator, in which reconstruction occurs from the external background and prevents propagation into internal regions not connected to the exterior, resulting in consistent filling of internal cavities. In contrast, [Figure 4.11](#fig-04-edgeoff-didatico) details the dynamics of the `mm::edgeoff` operator, in which only components connected to the border are reconstructed and subsequently removed, preserving exclusively the objects fully contained within the interior of the image.

The connectivity of the geodesic propagation is controlled by the structuring element: `mm::sebox()` (8-neighborhood) includes diagonal connections, whereas `mm::secross()` (4-neighborhood) excludes them. Consequently, the choice of the structuring element affects which components are reached by the reconstruction and, therefore, which will be preserved or removed.

#### 4.3.5.1 Compliance with the implementation

The definitions above are directly aligned with the implementation in `morph.py`, reproduced below:

```python
staticmethod
def clohole(f, b=np.ones((3,3),dtype='uint8')):
    # marcador restrito ao fundo da imagem
    marcador = mm.frame(f, border=1) & mm.neg(f)
    return mm.neg(mm.infrec(marcador, mm.neg(f), b))

staticmethod
def edgeoff(f, b=np.ones((3,3),dtype='uint8')):
    # marcador restrito aos objetos da imagem
    marcador = mm.frame(f, border=1) & f
    return mm.subm(f, mm.infrec(marcador, f, b))
```

These implementations make it explicit that both operators are direct instances of morphological reconstruction by geodesic dilation using `mm::infrec`, differing only in the choice of marker and mask: `clohole` operates on the image complement, while `edgeoff` operates directly in the domain of objects.

In [27]:
%%writefile tmp/fig_04_clohole_didatico.cpp
#define MM_OUT "tmp/fig_04_clohole_didatico.png"
//| label: fig-04-clohole-didatico
//| fig-cap: "*Pipeline* de preenchimento de buracos (*clohole*): o marcador vem da borda da imagem, restrito ao complemento $f^c$. A dilatação geodésica reconstrói o fundo externo; após a complementação, os buracos internos ficam preenchidos."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    mm::Image f(10, 10);
    std::vector<std::vector<int>> f_data = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,1,1,1,0,0,1,0,0},
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,1,1,1,0,1,0,1,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,0,1}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            f.at(y, x) = f_data[y][x] * 255;

    mm::Image B_cruz = mm::secross();
    mm::Image f_c = mm::neg(f);
    mm::Image marcador_ch = mm::frame(f, 1);       // borda externa

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = marcador_ch;
    for (int i = 1; i <= 5; i++) {
        img_atual = mm::cdil(img_atual, f_c, B_cruz);
        iteracoes.push_back(img_atual);
        titulos.push_back("Iter. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_clohole = mm::neg(mm::infrec(marcador_ch, f_c, B_cruz));
    mm::Image clohole_ref = mm::clohole(f);
    bool valid = true;
    for (int i = 0; i < img_clohole.h * img_clohole.w * img_clohole.channels; i++) {
        if (img_clohole.data[i] != clohole_ref.data[i]) {
            valid = false;
            break;
        }
    }
    std::cout << "✅ Validação clohole: " << (valid ? "True" : "False") << "\n";

    std::vector<mm::Image> all_images;
    all_images.push_back(f);
    all_images.push_back(marcador_ch);
    all_images.insert(all_images.end(), iteracoes.begin(), iteracoes.end());
    all_images.push_back(img_clohole);

    std::vector<std::string> all_titles;
    all_titles.push_back("f original");
    all_titles.push_back("Marcador (borda)");
    all_titles.insert(all_titles.end(), titulos.begin(), titulos.end());
    all_titles.push_back("clohole(f)");

    mm::show(all_images, MM_OUT, all_titles, 8);
    return 0;
}

Overwriting tmp/fig_04_clohole_didatico.cpp


In [28]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_clohole_didatico.cpp -o tmp/fig_04_clohole_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_clohole_didatico \
  && test -f "tmp/fig_04_clohole_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_clohole_didatico.png"

✅ Validação clohole: True
[1] f original
[2] Marcador (borda)
[3] Iter. (n=1)
[4] Iter. (n=2)
[5] Iter. (n=3)
[6] Iter. (n=4)
[7] Iter. (n=5)
[8] clohole(f)


In [29]:
mm.show(mm.read("tmp/fig_04_clohole_didatico.png"), figsize=(18, 3))

<Figure size 2700x450 with 1 Axes>

**Figure 4.10:** *Pipeline* de preenchimento de buracos (*clohole*): o marcador vem da borda da imagem, restrito ao complemento $f^c$. A dilatação geodésica reconstrói o fundo externo; após a complementação, os buracos internos ficam preenchidos.


In [30]:
%%writefile tmp/fig_04_edgeoff_didatico.cpp
#define MM_OUT "tmp/fig_04_edgeoff_didatico.png"
// Compile with: g++ -std=c++17 -I. -o program program.cpp -lm

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    //| label: fig-04-edgeoff-didatico
    //| fig-cap: "*Pipeline* of edge structure elimination (*edgeoff*): the marker captures the roots connected to the extremities, the reconstruction delimits these elements and the subtraction preserves only the totally internal objects."
    //| echo: true
    //| output: true

    mm::Image f(10, 10);
    // Initialize f with the given pattern * 255
    unsigned char pattern[10][10] = {
        {0,0,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,0,0,0,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,0,0,1,0,0,1,0,1},
        {0,1,1,1,1,0,0,1,0,0},
        {0,0,0,0,0,0,0,0,0,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,1,1,1,0,1,0,1,0},
        {0,0,1,1,1,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,0,1}
    };
    for (int y = 0; y < 10; y++)
        for (int x = 0; x < 10; x++)
            f.at(y, x) = pattern[y][x] * 255;

    mm::Image B_box = mm::sebox();
    mm::Image marcador_eo = mm::band(mm::frame(f, 1), f);   // borda ∩ f

    std::vector<mm::Image> iteracoes;
    std::vector<std::string> titulos;
    mm::Image img_atual = marcador_eo;
    for (int i = 1; i < 6; i++) {
        img_atual = mm::cdil(img_atual, f, B_box);
        iteracoes.push_back(img_atual);
        titulos.push_back("Iter. (n=" + std::to_string(i) + ")");
    }

    mm::Image img_edgeoff = mm::edgeoff(f, B_box);

    std::vector<mm::Image> show_imgs = {f, marcador_eo};
    show_imgs.insert(show_imgs.end(), iteracoes.begin(), iteracoes.end());
    show_imgs.push_back(img_edgeoff);

    std::vector<std::string> show_titles = {"f original", "Marcador (borda)"};
    show_titles.insert(show_titles.end(), titulos.begin(), titulos.end());
    show_titles.push_back("edgeoff(f)");

    mm::show(show_imgs, MM_OUT, show_titles, 8);

    return 0;
}

Overwriting tmp/fig_04_edgeoff_didatico.cpp


In [31]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_edgeoff_didatico.cpp -o tmp/fig_04_edgeoff_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_edgeoff_didatico \
  && test -f "tmp/fig_04_edgeoff_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_edgeoff_didatico.png"

[1] f original
[2] Marcador (borda)
[3] Iter. (n=1)
[4] Iter. (n=2)
[5] Iter. (n=3)
[6] Iter. (n=4)
[7] Iter. (n=5)
[8] edgeoff(f)


In [32]:
mm.show(mm.read("tmp/fig_04_edgeoff_didatico.png"), figsize=(18, 3))

<Figure size 2700x450 with 1 Axes>

**Figure 4.11:** *Pipeline* de eliminação de estruturas de borda (*edgeoff*): o marcador captura as raízes conectadas às extremidades, a reconstrução delimita esses elementos e a subtração preserva só os objetos totalmente internos.


### 4.3.6 Binary Cleaning Pipeline with CLAHE

Based on the previous analysis — in which CLAHE produced the highest inter-class variance value ($\sigma_B^2 \approx 2.47 \times 10^3$), see [Figure 4.2](#fig-04-otsu-comparacao-histogramas), and the `mm::clohole` operator proved effective in filling internal cavities —, the final segmentation pipeline, illustrated in [Figure 4.12](#fig-04-pipeline-clahe), is structured by the following computational flow:

$$
\text{gray}
\xrightarrow{\text{CLAHE}}
\xrightarrow{\text{Otsu}}
\xrightarrow{\text{open}}
\xrightarrow{\text{clohole}}
\xrightarrow{\text{open}}
\xrightarrow{\text{edgeoff}}
\text{segmentation}
$$

After the `mm::clohole` step, a second morphological opening is applied with a larger structuring element (`mm::sedisk(33)`, a disk of diameter 33). This operation removes small residual regions and artifacts that may have remained after segmentation. In particular, geodesic filling can transform small isolated cavities into components connected to the object, making an additional size-based filtering step convenient. The diameter was chosen so that the coins remain able to contain the structuring element, while significantly smaller components are eliminated.

In this image, no coin is connected to the matrix border. Consequently, applying `mm::edgeoff` does not alter the result obtained after the second opening. Nevertheless, this step is kept in the pipeline for robustness, since in other images there may be partially visible objects or objects connected to the borders, which should be removed before the analysis stage.

> ### 💡 Why the opening after clohole?
>
> The `mm::clohole` operator fills all closed cavities present in the segmented objects. In some situations, small unwanted regions may remain after this step or become connected to the main objects. The subsequent morphological opening removes components smaller than the structuring element, while preserving the coins due to their significantly larger size.

In [33]:
%%writefile tmp/fig_04_pipeline_clahe.cpp
#define MM_OUT "tmp/fig_04_pipeline_clahe.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    // ── Pré-processamento: apenas CLAHE
    mm::Image img_clahe0 = mm::clahe(img_coins_gray, 2.0, 8);

    // ── Etapa 1: Binarização Otsu
    mm::Image img_bin = mm::threshold(img_clahe0);

    // ── Etapa 2: Abertura — remove ruídos brancos de fundo
    mm::Image img_open = mm::open(img_bin, mm::sedisk(9));

    // ── Etapa 3: clohole — fecha todos os buracos internos
    mm::Image img_hole = mm::clohole(img_open);

    // ── Etapa 4: Abertura (kernel grande) — remove artefatos do clohole
    mm::Image img_limpo = mm::open(img_hole, mm::sedisk(33));

    // ── Etapa 5: edgeoff — remove objetos que tocam a borda 
    mm::Image img_final = mm::edgeoff(img_limpo, mm::SE::box(3), 1);

    mm::show(
        std::vector<mm::Image>{img_clahe0, img_bin, img_open,
                               img_hole, img_limpo, img_final},
        MM_OUT,
        std::vector<std::string>{"CLAHE", "Otsu", "Abertura (r=9)",
                                 "clohole", "Abertura (r=33)", "Final (edgeoff)"},
        6
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_final, "tmp/state/img_final_55.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_clahe0, "tmp/fig_04_pipeline_clahe_0.png");
mm::write(img_bin, "tmp/fig_04_pipeline_clahe_1.png");
mm::write(img_open, "tmp/fig_04_pipeline_clahe_2.png");
mm::write(img_hole, "tmp/fig_04_pipeline_clahe_3.png");
mm::write(img_limpo, "tmp/fig_04_pipeline_clahe_4.png");
mm::write(img_final, "tmp/fig_04_pipeline_clahe_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_pipeline_clahe.cpp


In [34]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_pipeline_clahe.cpp -o tmp/fig_04_pipeline_clahe -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_pipeline_clahe \
  && test -f "tmp/fig_04_pipeline_clahe.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_pipeline_clahe.png"

[1] CLAHE
[2] Otsu
[3] Abertura (r=9)
[4] clohole
[5] Abertura (r=33)
[6] Final (edgeoff)


In [35]:
mm.show(
    [
        mm.read("tmp/fig_04_pipeline_clahe_0.png"),
        mm.read("tmp/fig_04_pipeline_clahe_1.png"),
        mm.read("tmp/fig_04_pipeline_clahe_2.png"),
        mm.read("tmp/fig_04_pipeline_clahe_3.png"),
        mm.read("tmp/fig_04_pipeline_clahe_4.png"),
        mm.read("tmp/fig_04_pipeline_clahe_5.png"),
    ],
    titles=[
        'CLAHE',
        'Otsu',
        'Abertura (r=9)',
        'clohole',
        'Abertura (r=33)',
        'Final (edgeoff)',
    ],
    cols=6,
    figsize=(18, 6),
)

<Figure size 2700x900 with 6 Axes>

**Figure 4.12:** *Pipeline* completo de segmentação com CLAHE: Otsu → abertura (r=9) → clohole → abertura (r=33) → edgeoff.


### 4.3.7 Grayscale Morphology

Morphological operators extend naturally to grayscale images. In this formulation, erosion and dilation act directly on the image intensity levels. For flat structuring elements ($b \equiv 0$), erosion corresponds to the **local minimum** and dilation to the **local maximum** within the neighborhood defined by the structuring element.

The intuitive interpretation is straightforward: erosion **darkens** regions by replacing each pixel with the smallest value in its neighborhood, while dilation **lightens** regions by using the largest available value. Combining these operators allows for the construction of transformations capable of enhancing edges, removing illumination trends, and highlighting local structures.

Three derived operators are especially useful:

**Morphological gradient** — highlights edges as the difference between dilation and erosion:

<a id="eq-04-gradiente-morf"></a>
$$
\text{grad}_B(f) = (f \oplus B) - (f \ominus B) \tag{4.12}
$$


***Top-hat*** — highlights bright structures smaller than the structuring element (difference between the original image and its opening):

<a id="eq-04-tophat"></a>
$$
\text{top-hat}_B(f) = f - (f \circ B) \tag{4.13}
$$


***Black-hat*** — highlights dark structures smaller than the structuring element (difference between the closing and the original image):

<a id="eq-04-blackhat"></a>
$$
\text{black-hat}_B(f) = (f \bullet B) - f \tag{4.14}
$$


The *Top-hat* extracts bright details that do not survive the opening, while the *Black-hat* reveals dark details removed by the closing. The morphological gradient, in turn, enhances abrupt intensity transitions, producing a representation similar to that of an edge detector.

To understand the mechanism of these operators at a local level, [Figure 4.13](#fig-04-sim-04-operadores-av) presents an interactive simulator of grayscale morphology. The simulator allows you to freely edit the structuring element, visualize its displacement over the image, and simultaneously track the one-dimensional intensity profile. In this way, it becomes possible to directly observe how erosion selects local minima, how dilation selects local maxima, and how the morphological gradient emerges from the difference between these two operators.

The button located in the upper right corner allows you to toggle between the original grayscale visualization and a pseudocolored representation (*colormap*) for the three gradient types only. The colored version facilitates the visual perception of intensity variations, making the action of the morphological operators on maxima, minima, and local transitions of the image more evident.

The operators presented are available in `morph.py` through the functions `mm::gradm`, `mm::tophat`, and `mm::blackhat`.

In [36]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-operadores-av" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-operadores-av * { box-sizing: border-box; }
  #sim-04-operadores-av canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; }
  #sim-04-operadores-av button { font-size: 11px; padding: 6px 10px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 4px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-operadores-av button:hover { background: #e8dfcf; }
  #sim-04-operadores-av button.gc_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  
  #sim-04-operadores-av button.gc_toggle_btn { background: #ebf4fd; border-color: #2980b9; color: #2980b9; font-weight: 700; }
  #sim-04-operadores-av button.gc_toggle_btn:hover { background: #d4e6fc; }
  
  .gc_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .gc_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 8px; padding: 8px; text-align: center; }
  .gc_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .gc_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  #sim04_seCanvas { cursor: pointer; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎛️ Advanced Mathematical Morphology Simulator</span>
  <button id="sim04_btn_toggle_view" class="gc_toggle_btn" onclick="sim04_toggleViewMode()">
    Visualization: Real Grayscale
  </button>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Barra de Modos de Operação -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:14px; justify-content:center;">
    <button id="sim04_btn_orig" class="gc_active" onclick="sim04_setMode('orig')">Original f</button>
    <button id="sim04_btn_ero" onclick="sim04_setMode('ero')">Erosion</button>
    <button id="sim04_btn_dil" onclick="sim04_setMode('dil')">Dilation</button>
    <button id="sim04_btn_open" onclick="sim04_setMode('open')">Opening (∘)</button>
    <button id="sim04_btn_close" onclick="sim04_setMode('close')">Closing (•)</button>
    <button id="sim04_btn_grad" onclick="sim04_setMode('grad')">Gradient</button>
    <button id="sim04_btn_tophat" onclick="sim04_setMode('tophat')">Top-hat</button>
    <button id="sim04_btn_bhat" onclick="sim04_setMode('bhat')">Black-hat</button>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:grid; grid-template-columns:repeat(auto-fit, minmax(260px, 1fr)); gap:16px; align-items:start; margin-bottom:14px;">
    
    <!-- Canvas Principal -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
      <canvas id="sim04_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Move the mouse to update the 1D profile of the corresponding line
      </div>
    </div>

    <!-- Painel Lateral de Estatísticas, Descrição e Kernel B -->
    <div style="display:flex; flex-direction:column; gap:12px;">
      
      <!-- Estatísticas / Valores do Pixel -->
      <div style="display:grid; grid-template-columns:repeat(2, 1fr); gap:8px;">
        <div class="gc_stat_box"><div class="gc_stat_label">X (col)</div><div id="sim04_sX" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">Y (row)</div><div id="sim04_sY" class="gc_stat_value">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label">f(x,y)</div><div id="sim04_sOrig" class="gc_stat_value" style="color:#27ae60;">—</div></div>
        <div class="gc_stat_box"><div class="gc_stat_label" id="sim04_sLabel">value</div><div id="sim04_sVal" class="gc_stat_value" style="color:#2980b9;">—</div></div>
      </div>

      <!-- Descrição Teórica -->
      <div id="sim04_desc" class="gc_panel" style="font-size:11px; color:#5e5a4a; line-height:1.5; min-height:45px;"></div>

      <!-- Editor do Elemento Estruturante B -->
      <div class="gc_panel" style="text-align:center;">
        <div style="font-size:10.5px; font-weight:700; color:#5e5a4a; margin-bottom:8px; text-transform:uppercase; letter-spacing:.3px;">
          Element B (Click to Edit)
        </div>
        <canvas id="sim04_seCanvas" width="114" height="114" style="margin:0 auto; display:block; border-radius:6px; border:1px solid #e4dcc8; background:#ffffff;"></canvas>
      </div>

      <!-- Fórmulas Auxiliares -->
      <div class="gc_panel" style="font-size:9.5px; font-family:monospace; color:#5e5a4a; line-height:1.5;">
        <strong style="color:#26241d;">Opening (f∘B):</strong> dil(ero(f))<br>
        <strong style="color:#26241d;">Closing (f•B):</strong> ero(dil(f))<br>
        <strong style="color:#26241d;">Gradient:</strong> dil(f) − ero(f)<br>
        <strong style="color:#26241d;">Top-hat:</strong> f − (f∘B)<br>
        <strong style="color:#26241d;">Black-hat:</strong> (f•B) − f
      </div>

    </div>

  </div>

  <!-- Perfil 1D na Parte Inferior -->
  <div style="background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
    <canvas id="sim04_profileCanvas" width="660" height="110" style="border-radius:8px; border:1px solid #e4dcc8; width:100%; height:auto; background:#ffffff;"></canvas>
    <div id="sim04_profileLabel" style="font-size:10.5px; color:#8a8371; margin-top:6px; font-family:monospace;">1D line profile: None (hover over the image)</div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04Advanced(root){
    if (!root || root.dataset.sim04AdvancedInit) return;
    root.dataset.sim04AdvancedInit = "1";

    const ROWS = 40, COLS = 40, CELL = 8, PAD = 6;
    const W = COLS * CELL + PAD * 2, H = ROWS * CELL + PAD * 2;

    const cv = root.querySelector('#sim04_Canvas');
    cv.width = W; cv.height = H;
    const ctx = cv.getContext('2d');

    let gc_viewMode = 'gray';
    let gc_mode = 'orig';
    let gc_hx = -1, gc_hy = 19;

    const f = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
    const cx1 = 13, cy1 = 14, r1 = 9, cx2 = 28, cy2 = 14, r2 = 7, cx3 = 20, cy3 = 28, r3 = 8;
    for (let r=0; r<ROWS; r++) {
      for (let c=0; c<COLS; c++) {
        let v = 30;
        const d1 = Math.hypot(c - cx1, r - cy1), d2 = Math.hypot(c - cx2, r - cy2), d3 = Math.hypot(c - cx3, r - cy3);
        if (d1 < r1) v = Math.round(200 - 90 * (d1 / r1));
        if (d2 < r2) v = Math.max(v, Math.round(180 - 70 * (d2 / r2)));
        if (d3 < r3) v = Math.max(v, Math.round(160 - 50 * (d3 / r3)));
        if (Math.hypot(c - 33, r - 8) < 2.2) v = 230;
        if (Math.hypot(c - 6, r - 32) < 2.2) v = Math.min(v, 20);
        f[r][c] = Math.max(0, Math.min(255, v));
      }
    }

    const B_SIZE = 5;
    const B = Array.from({length: B_SIZE}, () => new Array(B_SIZE).fill(0));
    const rad = 2;
    for (let r=0; r<B_SIZE; r++) {
      for (let c=0; c<B_SIZE; c++) {
        if (Math.hypot(c - rad, r - rad) <= rad) B[r][c] = 1;
      }
    }

    const MORPH_DATA = {};

    function recalculateMorphology() {
      const Boff = [];
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          if (B[r][c]) Boff.push([r - center, c - center]);
        }
      }
      if (Boff.length === 0) Boff.push([0, 0]);

      function e_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mn = 255;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 255;
              if (val < mn) mn = val;
            }
            res[r][c] = mn;
          }
        }
        return res;
      }

      function d_op(img) {
        const res = Array.from({length: ROWS}, ()=>new Array(COLS).fill(0));
        for (let r=0; r<ROWS; r++) {
          for (let c=0; c<COLS; c++) {
            let mx = 0;
            for (const [dr, dc] of Boff) {
              const vr = r + dr, vc = c + dc;
              const val = (vr >= 0 && vr < ROWS && vc >= 0 && vc < COLS) ? img[vr][vc] : 0;
              if (val > mx) mx = val;
            }
            res[r][c] = mx;
          }
        }
        return res;
      }

      function diff(a, b) {
        return Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => Math.max(0, Math.min(255, a[r][c] - b[r][c]))));
      }

      MORPH_DATA.orig = f;
      MORPH_DATA.ero = e_op(f);
      MORPH_DATA.dil = d_op(f);
      MORPH_DATA.open = d_op(MORPH_DATA.ero);
      MORPH_DATA.close = e_op(MORPH_DATA.dil);
      MORPH_DATA.grad = diff(MORPH_DATA.dil, MORPH_DATA.ero);
      MORPH_DATA.tophat = diff(f, MORPH_DATA.open);
      MORPH_DATA.bhat = diff(MORPH_DATA.close, f);
    }

    const MODES_CONFIG = {
      orig: {label: 'f(x,y)', color: '#27ae60', desc: 'Imagem original f.'},
      ero: {label: 'ero(x,y)', color: '#c0392b', desc: 'Erosão: Encolhe estruturas claras de acordo com a geometria de B.'},
      dil: {label: 'dil(x,y)', color: '#2980b9', desc: 'Dilatação: Expande estruturas claras preenchendo falhas.'},
      open: {label: 'open(x,y)', color: '#8e44ad', desc: 'Abertura: Suaviza contornos, elimina pequenos ruídos e picos brilhantes isolados.'},
      close: {label: 'close(x,y)', color: '#d35400', desc: 'Fechamento: Preenche pequenos canais ou buracos escuros interiores.'},
      grad: {label: 'grad(x,y)', color: '#16a085', desc: 'Gradiente morfológico: Destaca as bordas físicas dos objetos.'},
      tophat: {label: 'th(x,y)', color: '#b9770e', desc: 'Top-hat: Isola elementos brilhantes menores que B.'},
      bhat: {label: 'bh(x,y)', color: '#2c3e50', desc: 'Black-hat: Isola fossas escuras ou vales menores que B.'}
    };

    window.sim04_toggleViewMode = function() {
      if (gc_viewMode === 'gray') {
        gc_viewMode = 'colormap';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Falsa Cor';
      } else {
        gc_viewMode = 'gray';
        root.querySelector('#sim04_btn_toggle_view').innerHTML = 'Visualização: Cinza Real';
      }
      gc_draw();
    };

    window.sim04_setMode = function(m) {
      gc_mode = m;
      Object.keys(MODES_CONFIG).forEach(k => {
        const btn = root.querySelector('#sim04_btn_' + k);
        if (btn) btn.classList.toggle('gc_active', k === m);
      });
      const cfg = MODES_CONFIG[m];
      root.querySelector('#sim04_sLabel').textContent = cfg.label;
      root.querySelector('#sim04_sVal').style.color = cfg.color;
      root.querySelector('#sim04_desc').textContent = cfg.desc;
      gc_draw();
      gc_drawProfile();
    };

    function gc_imgToGray(data) {
      const id = ctx.createImageData(W, H);
      for (let r=0; r<ROWS; r++) {
        for (let c=0; c<COLS; c++) {
          const v = data[r][c];
          const x = PAD + c * CELL, y = PAD + r * CELL;
          let rc = v, gc = v, bc = v;

          if (gc_viewMode === 'colormap' && (gc_mode === 'grad' || gc_mode === 'tophat' || gc_mode === 'bhat')) {
            if (v > 0) {
              rc = Math.min(255, v * 7);       
              gc = Math.min(255, v * 3.5);     
              bc = Math.max(40, 255 - v * 4.5); 
            } else { rc = 250; gc = 250; bc = 247; }
          }

          for (let dy=0; dy<CELL; dy++) {
            for (let dx=0; dx<CELL; dx++) {
              const idx = 4 * ((y + dy) * W + (x + dx));
              id.data[idx] = rc; id.data[idx+1] = gc; id.data[idx+2] = bc; id.data[idx+3] = 255;
            }
          }
        }
      }
      return id;
    }

    function gc_draw() {
      const data = MORPH_DATA[gc_mode];
      const id = gc_imgToGray(data);
      ctx.putImageData(id, 0, 0);

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 0.5;
      for (let c=0; c<=COLS; c++) { ctx.beginPath(); ctx.moveTo(PAD + c * CELL, PAD); ctx.lineTo(PAD + c * CELL, PAD + ROWS * CELL); ctx.stroke(); }
      for (let r=0; r<=ROWS; r++) { ctx.beginPath(); ctx.moveTo(PAD, PAD + r * CELL); ctx.lineTo(PAD + COLS * CELL, PAD + r * CELL); ctx.stroke(); }

      if (gc_hy >= 0 && gc_hy < ROWS) {
        ctx.strokeStyle = 'rgba(192, 57, 43, 0.4)'; ctx.lineWidth = 1;
        ctx.beginPath(); ctx.moveTo(PAD, PAD + gc_hy * CELL + CELL / 2); ctx.lineTo(PAD + COLS * CELL, PAD + gc_hy * CELL + CELL / 2); ctx.stroke();
      }

      if (gc_hx >= 0 && gc_hy >= 0) {
        ctx.strokeStyle = 'rgba(41, 128, 185, 0.9)'; ctx.lineWidth = 1.5;
        ctx.strokeRect(PAD + gc_hx * CELL + 0.5, PAD + gc_hy * CELL + 0.5, CELL - 1, CELL - 1);
      }

      ctx.strokeStyle = '#e4dcc8'; ctx.lineWidth = 1;
      ctx.strokeRect(PAD, PAD, COLS * CELL, ROWS * CELL);
    }

    const seCanvas = root.querySelector('#sim04_seCanvas');
    const seCtx = seCanvas.getContext('2d');
    const SE_CELL = 20, SE_PAD = 7;

    function gc_drawSE() {
      seCtx.clearRect(0, 0, 114, 114);
      const center = Math.floor(B_SIZE / 2);
      for (let r=0; r<B_SIZE; r++) {
        for (let c=0; c<B_SIZE; c++) {
          const x = SE_PAD + c * SE_CELL, y = SE_PAD + r * SE_CELL;
          seCtx.fillStyle = B[r][c] ? '#2980b9' : '#fafaf7';
          seCtx.fillRect(x, y, SE_CELL - 2, SE_CELL - 2);
          seCtx.strokeStyle = '#e4dcc8';
          seCtx.strokeRect(x, y, SE_CELL - 2, SE_CELL - 2);
          if (r === center && c === center) {
            seCtx.fillStyle = '#ffffff'; seCtx.font = 'bold 11px monospace';
            seCtx.textAlign = 'center'; seCtx.textBaseline = 'middle';
            seCtx.fillText('★', x + (SE_CELL - 2) / 2, y + (SE_CELL - 2) / 2);
          }
        }
      }
    }

    seCanvas.addEventListener('click', function(e) {
      const rect = seCanvas.getBoundingClientRect();
      const scaleX = seCanvas.width / rect.width;
      const scaleY = seCanvas.height / rect.height;
      const cx = (e.clientX - rect.left) * scaleX;
      const cy = (e.clientY - rect.top) * scaleY;
      const c = Math.floor((cx - SE_PAD) / SE_CELL);
      const r = Math.floor((cy - SE_PAD) / SE_CELL);
      if (c >= 0 && c < B_SIZE && r >= 0 && r < B_SIZE) {
        B[r][c] = B[r][c] ? 0 : 1;
        recalculateMorphology();
        gc_drawSE();
        gc_draw();
        gc_drawProfile();
      }
    });

    function gc_drawProfile() {
      const pc = root.querySelector('#sim04_profileCanvas');
      const pctx = pc.getContext('2d');
      pctx.clearRect(0, 0, 660, 110);
      
      if (gc_hy < 0 || gc_hy >= ROWS) return;

      root.querySelector('#sim04_profileLabel').innerHTML = `<span style='color:#c0392b; font-weight:bold;'>Perfil 1D da Linha ${gc_hy}</span> — Tracejado: Original f(x) | Cor da Aba: Operação Atual`;

      const dataOrig = MORPH_DATA.orig[gc_hy];
      const dataCurrent = MORPH_DATA[gc_mode][gc_hy];
      const currentColor = MODES_CONFIG[gc_mode].color;
      
      const stepX = 660 / (COLS - 1);
      
      function drawLine(arrayData, color, width, isDash = false) {
        pctx.strokeStyle = color; pctx.lineWidth = width;
        pctx.beginPath();
        if (isDash) pctx.setLineDash([4, 4]); else pctx.setLineDash([]);
        for (let c=0; c<COLS; c++) {
          const x = c * stepX;
          const y = 95 - (arrayData[c] / 255) * 85; 
          if (c === 0) pctx.moveTo(x, y); else pctx.lineTo(x, y);
        }
        pctx.stroke();
      }

      pctx.setLineDash([]);
      pctx.strokeStyle = '#e4dcc8'; pctx.lineWidth = 0.5;
      for (let h=0; h<=4; h++) {
        let yVal = 10 + h * 21.25;
        pctx.beginPath(); pctx.moveTo(0, yVal); pctx.lineTo(660, yVal); pctx.stroke();
      }

      if (gc_mode !== 'orig') {
        drawLine(dataOrig, 'rgba(39, 174, 96, 0.4)', 1.5, true);
      }
      drawLine(dataCurrent, currentColor, 2.5, false);

      pctx.setLineDash([]);
      pctx.fillStyle = '#8a8371'; pctx.font = '9px monospace';
      pctx.fillText('Intensidade (255)', 5, 10);
      pctx.fillText('Fundo (0)', 5, 104);
    }

    cv.addEventListener('mousemove', function(e) {
      const rect = cv.getBoundingClientRect();
      const sx = W / rect.width, sy = H / rect.height;
      const c = Math.floor(((e.clientX - rect.left) * sx - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * sy - PAD) / CELL);
      if (c >= 0 && c < COLS && r >= 0 && r < ROWS) {
        gc_hx = c; gc_hy = r;
        root.querySelector('#sim04_sX').textContent = c;
        root.querySelector('#sim04_sY').textContent = r;
        root.querySelector('#sim04_sOrig').textContent = f[r][c];
        root.querySelector('#sim04_sVal').textContent = MORPH_DATA[gc_mode][r][c];
      }
      gc_draw();
      gc_drawProfile();
    });

    recalculateMorphology();
    sim04_setMode('orig');
    gc_drawSE();
  }

  function tryInitSim04Advanced(){
    var root = document.getElementById('sim-04-operadores-av');
    if (root) initSim04Advanced(root); else setTimeout(tryInitSim04Advanced, 200);
  }
  tryInitSim04Advanced();
})();
</script>
""")

**Figure 4.13:** Advanced interactive morphology simulator with editable structuring element and 1D profile.


<figure id="fig-04-sim-04-operadores-av">
  <img src="imagens/fig-04-sim-04-operadores-av.png" alt=" Advanced interactive morphology simulator with editable structuring element and 1D profile. " style="max-width:80%" />
  <figcaption><strong>Figure 4.13:</strong>  Advanced interactive morphology simulator with editable structuring element and 1D profile. </figcaption>
</figure>

[Figure 4.14](#fig-04-morf-gc-histogramas) illustrates the effects of these operators on the coin image and their respective histograms. Note that erosion shifts the distribution toward lower intensities, while dilation shifts it toward higher intensities. The gradient concentrates values in contour regions, and the *Top-hat* and *Black-hat* operators produce histograms strongly concentrated at low intensity levels, as only small local structures are highlighted.

In [37]:
%%writefile tmp/fig_04_morf_gc_histogramas.cpp
#define MM_OUT "tmp/fig_04_morf_gc_histogramas.png"
//| label: fig-04-morf-gc-histogramas
//| fig-cap: "Morfologia em tons de cinza e seus histogramas: erosão, dilatação, gradiente, top-hat e black-hat. Elemento estruturante: disco de diâmetro 19."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
// [pdi:state-io:end]

    mm::Image B = mm::sedisk(19);
    mm::Image img_ero  = mm::ero(img_coins_gray, B);
    mm::Image img_dil  = mm::dil(img_coins_gray, B);
    mm::Image img_grad = mm::gradm(img_coins_gray, B);
    mm::Image img_th   = mm::tophat(img_coins_gray, B);
    mm::Image img_bh   = mm::blackhat(img_coins_gray, B);

    mm::show(
        std::vector<mm::Image>{img_coins_gray, mm::histImg(img_coins_gray), img_ero, mm::histImg(img_ero),
         img_dil, mm::histImg(img_dil), img_grad, mm::histImg(img_grad),
         img_th, mm::histImg(img_th), img_bh, mm::histImg(img_bh)},
        MM_OUT,
        std::vector<std::string>{"Original", "Hist", "Erosão", "Hist", "Dilatação", "Hist",
                "Gradiente", "Hist", "Top-hat", "Hist", "Black-hat", "Hist"},
        4
    );

    return 0;
}

Overwriting tmp/fig_04_morf_gc_histogramas.cpp


In [38]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_morf_gc_histogramas.cpp -o tmp/fig_04_morf_gc_histogramas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_morf_gc_histogramas \
  && test -f "tmp/fig_04_morf_gc_histogramas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_morf_gc_histogramas.png"

[1] Original
[2] Hist
[3] Erosão
[4] Hist
[5] Dilatação
[6] Hist
[7] Gradiente
[8] Hist
[9] Top-hat
[10] Hist
[11] Black-hat
[12] Hist


In [39]:
mm.show(mm.read("tmp/fig_04_morf_gc_histogramas.png"))

<Figure size 1044x784 with 1 Axes>

**Figure 4.14:** Morfologia em tons de cinza e seus histogramas: erosão, dilatação, gradiente, top-hat e black-hat. Elemento estruturante: disco de diâmetro 19.


The morphological operators presented previously will now be used as tools for refinement and marker generation in more advanced segmentation methods, presented below.

## 4.4 Image Segmentation: Fundamentals and Taxonomy

**Image segmentation** consists of dividing the image into regions associated with objects or structures of interest. In DIP, it represents the transition between low-level processing — such as filtering and enhancement — and more advanced analysis stages, such as feature extraction, recognition, and scene interpretation.

Formally, the goal of segmentation is to decompose the complete spatial domain of an image, denoted by $\Omega$, into a partition of subsets $\{R_1, R_2, \ldots, R_n\}$ that simultaneously satisfies the criteria of **completeness** and **disjointness**:

<a id="eq-04-particao"></a>
$$
\bigcup_{i=1}^{n} R_i = \Omega, \qquad R_i \cap R_j = \emptyset \quad \forall\, i \neq j \tag{4.15}
$$


In addition to the completeness and disjointness properties expressed in [Equation 4.15](#eq-04-particao), each subregion $R_i$ must constitute a **homogeneous** domain according to a similarity predicate defined over local properties — intensity, color, or texture — and, simultaneously, be **distinct** from adjacent regions.

Segmentation techniques can be organized into different families. This chapter emphasizes the approaches summarized in [Table 4.4](#tbl-04-segmentacao), based primarily on criteria of intensity, connectivity, and spatial proximity.

<a id="tbl-04-segmentacao"></a>

**Tabela 4.4:** Simplified taxonomy of the main segmentation and refinement approaches studied in this chapter.

| Approach | Segmentation Criterion | Reference Operators |
| :--- | :--- | :--- |
| **Thresholding** | Partitioning of the intensity space | Otsu's criterion, global and local thresholding |
| **Mathematical Morphology** | Spatial relations defined by structuring elements/functions | Erosion, dilation, opening, closing, and reconstruction |
| **Region-Based** | Local homogeneity and spatial connectivity | Connected component labeling, Distance Transform, and *Watershed* |


Up to this point, the practical development has focused on **thresholding**, through the combination of adaptive CLAHE equalization and Otsu's global method. This step was complemented by **morphological reconstruction** operators based on geodesic dilations, implemented by the functions `mm::infrec`, `mm::clohole`, and `mm::edgeoff`, producing a clean binary mask suitable for analysis.

However, in scenarios where distinct objects appear connected in the binary mask — whether by physical contact, partial overlap, or narrow pixel bridges produced by segmentation — thresholding is no longer sufficient to individualize each object. In such cases, multiple objects become part of a single connected component, hindering subsequent measurement and interpretation stages.

To overcome this limitation, the following sections introduce three complementary tools: **Connected Component Labeling**, the **Distance Transform**, and the marker-based ***Watershed*** segmentation algorithm. Together, these techniques make it possible to separate adjacent objects, identify regions individually, and extract consistent geometric descriptors for quantitative analysis.

### 4.4.1 Labeling

**Connected component labeling** is the operator that assigns a unique integer identifier to each set of pixels belonging to the same connected component in a binary image.

> ### 📝 Formal Definition
>
> Given a binary image $f$ and a connectivity relation defined by a structuring element $B$ (typically 4-connectivity or 8-connectivity), the labeling algorithm of [Figure 4.15](#fig-04-sim-alg-rotulagem2) produces an image $g$ in which all pixels belonging to the same connected component receive the same positive integer label, while pixels belonging to distinct components receive different labels.

Connectivity defines which pixels are considered direct neighbors of a pixel $(x,y)$. The most commonly used definitions are:

* **4-Connectivity:** considers only the four orthogonal neighbors (north, south, east, and west).
* **8-Connectivity:** considers the four orthogonal neighbors and the four diagonal ones, totaling eight neighbors.

The choice of connectivity directly influences the formation of connected components and, consequently, the labeling result, as illustrated in [Figure 4.17](#fig-04-rotulacao-didatico). An additional example can be interactively explored in the simulator presented in [Figure 4.16](#fig-04-sim-04-rotulacao).

The implementation in `morph.py` provides two versions of this operator. The function `mm::label0` explicitly reproduces the flood-fill algorithm using a stack and allows controlling connectivity through the adopted structuring element. Meanwhile, `mm::label` delegates the operation to OpenCV's optimized implementation (`mm::label0`). In both cases, the result is a labeled image in which each connected component receives a distinct integer identifier.

In [40]:
# @title { display-mode: "form" }
# Unique prefix to avoid conflicts with other notebook cells
PREFIX = "lbl2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EEEDFE;color:#534AB7}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo de rotulação por flood-fill com pilha — painel interativo com HTML e SVG</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-stack-2" aria-hidden="true"></i></div>
    <div><div class="algo-title">Flood-fill com pilha</div><div class="algo-sub">Rotulagem de componentes conexas</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Criar imagem de saída <em>g</em>, inicializada com <code>zeros</code>, mesma dimensão de <em>f</em>.</div></div>
    <div class="step"><div class="step-num num-teal">2</div><div class="step-text">Inicializar contador de rótulos (cor) <code>cor ← 1</code>.</div></div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Percorrer <em>f</em> em <strong>ordem raster</strong> (coordenadas <code>x</code> e <code>y</code>) até encontrar uma semente: pixel ativo (<code>f[x,y] ≠ 0</code>) ainda não rotulado (<code>g[x,y] = 0</code>).</div></div>
    <div class="step"><div class="step-num num-amber">4</div><div class="step-text">Inserir a semente encontrada na pilha <code>pilha ← [[x,y]]</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">5</div>
      <div class="step-text">Enquanto a pilha contiver coordenadas (<code>while pilha</code>):
        <div class="step-sub">
          <div class="step-sub-item">Desempilhar pixel atual: <code>i, j ← pilha.pop()</code> e atribuir o rótulo: <code>g[i,j] ← cor</code>.</div>
          <div class="step-sub-item">Buscar vizinhos usando o iterador <code>mm._viz(f,b,i,j)</code>. Se o vizinho for ativo no elemento estruturante (<code>bv ≠ 0</code>), ativo na imagem (<code>f[vy,vx] ≠ 0</code>) e não rotulado (<code>g[vy,vx] = 0</code>), empilhá-lo.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">6</div><div class="step-text">Pilha vazia ⟹ Toda a componente conexa atual foi explorada e rotulada com sucesso.</div></div>
    <div class="step"><div class="step-num num-blue">7</div><div class="step-text">Incrementar o rótulo para a próxima componente: <code>cor ← cor + 1</code> e continuar a varredura raster.</div></div>
    <div class="note">A conectividade (4 ou 8 vizinhos) é definida unicamente pela matriz morfológica <code>b</code> passada como parâmetro, alterando os pixels retornados em <code>mm._viz</code>.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Criar matriz de rótulos <code>g</code> preenchida com zeros (fundo). Definir rótulo inicial <code>cor ← 1</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Varredura Raster</div>
      <div class="card-text">Percorrer a matriz bidimensional linha por linha, localizando pixels pertencentes ao objeto que ainda não possuem rótulo.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-amber">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#854F0B">Semente inicial</div>
      <div class="card-text">Ao achar um pixel válido, inicializar a estrutura LIFO de busca: <code>pilha = [[x, y]]</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão por Flood-Fill</div>
      <div class="card-text">Enquanto houver elementos na pilha:</div>
      <div class="card-sub">
        <div class="card-sub-item">Extrair <code>(i, j)</code> via <code>pop()</code> e marcar <code>g[i, j] = cor</code>.</div>
        <div class="card-sub-item">Inspecionar vizinhança geométrica e adicionar novos candidatos à pilha.</div>
      </div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">05</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Próxima Componente</div>
      <div class="card-text">Pilha esvaziada ⟹ Incrementar indexador <code>cor ← cor + 1</code> para diferenciar o próximo objeto isolado.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Alocação Espacial</div><div class="tl-desc"><code>g ← zeros_like(f)</code> e definição do primeiro identificador: <code>cor ← 1</code>.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div><div class="tl-title">Varredura Bidimensional</div><div class="tl-desc">Laços encadeados varrendo as dimensões <code>h</code> e <code>w</code> da imagem.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Descoberta de Objeto</div><div class="tl-desc">Filtro condicional localiza pixel ativo não indexado e cria a <code>pilha</code> semente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04–05</span></div>
    <div>
      <div class="tl-title">Preenchimento por Região (Flood-fill)</div>
      <div class="while-box">
        <div class="while-head">while pilha</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Remover último da pilha <code>(i,j)</code> e aplicar rótulo atual.</div>
          <div class="step-sub-item">Empilhar vizinhos conectados que atendam aos critérios morfológicos de <code>b</code>.</div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">06</span></div>
    <div><div class="tl-title">Fechamento do Objeto</div><div class="tl-desc">Pilha vazia determina o fim do isolamento daquela componente.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">07</span></div>
    <div><div class="tl-title">Atualização do Rótulo</div><div class="tl-desc">Incremento linear: <code>cor ← cor + 1</code>. A varredura raster continua do ponto onde parou.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>label0.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">flood-fill com pilha</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">label0</span>(f, b=np.ones((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>)):
    <span class="cm">"""Rotulagem por flood-fill com pilha."""</span>
    h, w = f.shape
    g = np.zeros(f.shape, dtype=<span class="kw">int</span>)
    cor = <span class="num-lit">1</span>
    <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(h):
        <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(w):
            <span class="kw">if</span> f[x,y] <span class="kw">and not</span> g[x,y]:
                pilha = [[x,y]]
                <span class="kw">while</span> pilha:
                    i,j = pilha.pop(); g[i,j] = cor
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,i,j):
                        <span class="kw">if</span> bv <span class="kw">and</span> f[vy,vx] <span class="kw">and not</span> g[vy,vx]:
                            pilha.append([vy,vx])
                cor += <span class="num-lit">1</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>
<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g = np.zeros(f.shape, dtype=int)</code> — Inicializa a matriz de saída com zeros. Zeros representam o fundo invariável.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">2</div><div class="ann-text"><code>mm._viz(f, b, i, j)</code> — O iterador morfológico avalia a conectividade. Passando <code>B_cruz</code> a busca expande em 4-vizinhança; passando quadrado (<code>ones</code>) expande em 8-vizinhança.</div></div>
  <div class="ann-line"><div class="ann-badge num-amber">3</div><div class="ann-text"><code>pilha.pop()</code> — Remove o último par de coordenadas inserido, caracterizando um comportamento LIFO de busca em profundidade (DFS) para varrer o objeto de forma contígua.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">4</div><div class="ann-text"><code>cor += 1</code> — O incremento ocorre estritamente fora do laço <code>while</code>, garantindo que o mesmo número marque toda a extensão da componente concluída antes de passar para a próxima semente raster.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 820" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>
  
  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>
  
  <rect x="160" y="80" width="240" height="48" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="99" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar saída</text>
  <text x="280" y="116" text-anchor="middle" font-size="12" fill="#085041">g ← zeros(f.shape);  cor ← 1</text>
  
  <rect x="160" y="150" width="240" height="48" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="169" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">varredura raster</text>
  <text x="280" y="186" text-anchor="middle" font-size="12" fill="#3C3489">próximo pixel (x, y) em f</text>
  
  <polygon points="280,224 370,252 280,280 190,252" fill="#F1EFE8" stroke="#B4B2A9" stroke-width="1"/>
  <text x="280" y="248" text-anchor="middle" font-size="11.5" fill="#444441">imagem toda</text>
  <text x="280" y="264" text-anchor="middle" font-size="11.5" fill="#444441">varrida?</text>
  
  <ellipse cx="450" cy="252" rx="52" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="450" y="257" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  
  <polygon points="280,304 380,334 280,364 180,334" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="329" text-anchor="middle" font-size="11.5" fill="#3C3489">f[x,y] ≠ 0  e</text>
  <text x="280" y="344" text-anchor="middle" font-size="11.5" fill="#3C3489">g[x,y] = 0?</text>
  
  <rect x="170" y="388" width="220" height="44" rx="6" fill="#FAEEDA" stroke="#FAC775" stroke-width="1"/>
  <text x="280" y="406" text-anchor="middle" font-size="12" font-weight="500" fill="#854F0B">inserir semente</text>
  <text x="280" y="422" text-anchor="middle" font-size="12" fill="#633806">pilha ← [[x, y]]</text>
  
  <polygon points="280,456 370,486 280,516 190,486" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="491" text-anchor="middle" font-size="11.5" fill="#712B13">pilha vazia?</text>
  
  <rect x="160" y="542" width="240" height="48" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="561" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">i, j ← pilha.pop()</text>
  <text x="280" y="578" text-anchor="middle" font-size="12" fill="#712B13">g[i, j] ← cor</text>
  
  <rect x="150" y="614" width="260" height="60" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="634" text-anchor="middle" font-size="12" font-weight="500" fill="#185FA5">inspecionar vizinhos</text>
  <text x="280" y="650" text-anchor="middle" font-size="12" fill="#0C447C">se ativo e não rotulado</text>
  <text x="280" y="666" text-anchor="middle" font-size="12" fill="#0C447C">→ pilha.append([vy, vx])</text>
  
  <rect x="435" y="466" width="110" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="490" y="482" text-anchor="middle" font-size="12" font-weight="500" fill="#534AB7">cor ← cor + 1</text>
  <text x="490" y="496" text-anchor="middle" font-size="10.5" fill="#3C3489">próximo rótulo</text>

  <line x1="280" y1="58" x2="280" y2="80" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="128" x2="280" y2="150" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="198" x2="280" y2="224" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="370" y1="252" x2="398" y2="252" stroke="#3B6D11" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="384" y="245" text-anchor="middle" font-size="11" fill="#3B6D11">sim</text>
  
  <line x1="280" y1="280" x2="280" y2="304" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 180,334 L 130,334 L 130,174 L 160,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="152" y="327" text-anchor="middle" font-size="11" fill="#534AB7">não</text>
  
  <line x1="280" y1="364" x2="280" y2="388" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <text x="292" y="378" font-size="11" fill="#534AB7">sim</text>
  
  <line x1="280" y1="432" x2="280" y2="456" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="516" x2="280" y2="542" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="292" y="531" font-size="11" fill="#993C1D">não</text>
  
  <line x1="280" y1="590" x2="280" y2="614" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <path d="M 280,674 L 280,694 L 110,694 L 110,486 L 190,486" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="370" y1="486" x2="435" y2="486" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="402" y="479" text-anchor="middle" font-size="11" fill="#993C1D">sim</text>
  
  <path d="M 490,466 L 490,174 L 400,174" fill="none" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-b)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.15:** Flood-fill labeling algorithm with stack.


<figure id="fig-04-sim-alg-rotulagem2">
  <img src="imagens/fig-04-sim-alg-rotulagem2.png" alt=" Flood-fill labeling algorithm with stack. " style="max-width:80%" />
  <figcaption><strong>Figure 4.15:</strong>  Flood-fill labeling algorithm with stack. </figcaption>
</figure>

The helper `_viz` iterates over the structuring window `b` centered at $(i,j)$, generating only the valid neighbors within the image boundaries — the desired connectivity is entirely determined by the shape of `b` passed to the algorithm.

**Didactic example — effect of connectivity:**

In [41]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-rotulacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-rotulacao * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-rotulacao canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-rotulacao button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-rotulacao button:hover { background: #e8dfcf; }
  #sim-04-rotulacao button.rt_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-rotulacao .rt_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .rt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .rt_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .rt_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .rt_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .rt_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🪙 Simulator: Connected Components Labeling</span>
  <span class="rt_pill">flood-fill with stack</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="rt_grid_stats">
    <div class="rt_stat_box">
      <div class="rt_stat_label">Active Pixels</div>
      <div id="rt_statPx" class="rt_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Components</div>
      <div id="rt_statCC" class="rt_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Connectivity</div>
      <div id="rt_statConn" class="rt_stat_value" style="color:#b9770e;">4</div>
    </div>
    <div class="rt_stat_box">
      <div class="rt_stat_label">Raster Step</div>
      <div id="rt_statStep" class="rt_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="rt_ccCanvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Click to toggle pixels · Drag to paint
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:8px; margin-top:10px; justify-content:center;" id="rt_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:220px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Seletor de Conectividade -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Connectivity
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btn4" class="rt_active" onclick="window.rt_setConn(4)" style="flex:1; justify-content:center;">C-4</button>
          <button id="rt_btn8" onclick="window.rt_setConn(8)" style="flex:1; justify-content:center;">C-8</button>
        </div>
        <div id="rt_connDesc" style="font-size:9.5px; color:#8a8371; margin-top:6px; line-height:1.4;">
          4 orthogonal neighbors: N, S, E, W
        </div>
      </div>

      <!-- Seletor de Modo de Visualização -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualization
        </div>
        <div style="display:flex; gap:6px;">
          <button id="rt_btnLabel" class="rt_active" onclick="window.rt_setMode('label')" style="flex:1; justify-content:center;">Labels</button>
          <button id="rt_btnAnim" onclick="window.rt_setMode('anim')" style="flex:1; justify-content:center;">Animated</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="rt_animCtrl" class="rt_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Step by Step
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button onclick="window.rt_animStep(-1)" style="flex:1; justify-content:center;">◀</button>
          <button id="rt_btnPlay" onclick="window.rt_togglePlay()" style="flex:1; justify-content:center;">▶</button>
          <button onclick="window.rt_animStep(1)" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Speed</label>
          <input type="range" id="rt_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Status da Animação -->
      <div id="rt_animStatus" class="rt_panel" style="display:none;">
        <div id="rt_animTitle" style="font-size:11px; font-weight:700; margin-bottom:3px; color:#26241d;">–</div>
        <div id="rt_animDesc" style="font-size:9.5px; color:#8a8371; line-height:1.4;">–</div>
      </div>

      <!-- Exemplos / Presets -->
      <div class="rt_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Preset Examples
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button onclick="window.rt_loadPreset('diagonal')" style="justify-content:flex-start;">■ Diagonal (C-4 vs C-8)</button>
          <button onclick="window.rt_loadPreset('letters')" style="justify-content:flex-start;">A Separated Letters</button>
          <button onclick="window.rt_loadPreset('ring')" style="justify-content:flex-start;">◎ Ring with Hole</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button onclick="window.rt_clearGrid()" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Clear Grid
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04RotulacaoCC(root){
    if (!root || root.dataset.sim04RotulacaoCCInit) return;
    root.dataset.sim04RotulacaoCCInit = "1";

    const rt_COLS = 12, rt_ROWS = 12, rt_CELL = 22, rt_PAD = 10;
    const rt_W = rt_COLS * rt_CELL + rt_PAD * 2, rt_H = rt_ROWS * rt_CELL + rt_PAD * 2;
    const rt_cv = root.querySelector('#rt_ccCanvas');
    rt_cv.width = rt_W; 
    rt_cv.height = rt_H;
    const rt_ctx = rt_cv.getContext('2d');

    const rt_PALETTE = [
      ['#2980b9', '#ebf4fd', '#a9cce3', '#042c53'],
      ['#27ae60', '#eafaf1', '#a3e4d7', '#04342C'],
      ['#b9770e', '#fef5e7', '#f8c471', '#412402'],
      ['#c0392b', '#fdecea', '#f5b7b1', '#4A1B0C'],
      ['#8e44ad', '#f5eef8', '#d7bde2', '#4B1528'],
      ['#16a085', '#e8f8f5', '#a3e4d7', '#0e6251'],
      ['#d35400', '#fbeee6', '#f5cba7', '#7e5109'],
      ['#2c3e50', '#ebedef', '#bdc3c7', '#17202a']
    ];

    let rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
    let rt_connectivity = 4;
    let rt_mode = 'label';
    let rt_painting = false;
    let rt_paintVal = 1;

    let rt_animSteps = [];
    let rt_animIdx = 0;
    let rt_playing = false;
    let rt_playTimer = null;

    function rt_neighbors(r, c, conn) {
      const n = [[r-1,c],[r+1,c],[r,c-1],[r,c+1]];
      if (conn === 8) n.push([r-1,c-1],[r-1,c+1],[r+1,c-1],[r+1,c+1]);
      return n.filter(([nr,nc]) => nr>=0 && nr<rt_ROWS && nc>=0 && nc<rt_COLS);
    }

    function rt_label(g, conn) {
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          if (g[r][cc] && !lbl[r][cc]) {
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      return lbl;
    }

    function rt_buildAnimSteps(g, conn) {
      const steps = [];
      const lbl = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      let c = 1;
      for (let r=0; r<rt_ROWS; r++) {
        for (let cc=0; cc<rt_COLS; cc++) {
          steps.push({type:'raster', r, cc, lbl:lbl.map(a => [...a]), c});
          if (g[r][cc] && !lbl[r][cc]) {
            steps.push({type:'found', r, cc, c, lbl:lbl.map(a => [...a])});
            const stack = [[r,cc]];
            while (stack.length) {
              const [i,j] = stack.pop();
              if (lbl[i][j]) continue;
              lbl[i][j] = c;
              steps.push({type:'fill', i, j, c, lbl:lbl.map(a => [...a])});
              for (const [ni,nj] of rt_neighbors(i, j, conn))
                if (g[ni][nj] && !lbl[ni][nj]) stack.push([ni,nj]);
            }
            c++;
          }
        }
      }
      steps.push({type:'done', lbl:lbl.map(a => [...a]), c:c-1});
      return steps;
    }

    function rt_getColor(c) {
      const p = rt_PALETTE[(c-1) % rt_PALETTE.length];
      return { fill: p[1], stroke: p[0], text: p[0] };
    }

    function rt_drawLabel() {
      const lbl = rt_label(rt_grid, rt_connectivity);
      const numCC = Math.max(0, ...lbl.flat());
      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7'; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = '#8a8371';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('0', x+rt_CELL/2, y+rt_CELL/2);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      const numPx = rt_grid.flat().filter(Boolean).length;
      root.querySelector('#rt_statPx').textContent = numPx;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statConn').textContent = rt_connectivity;
      root.querySelector('#rt_statStep').textContent = '–';
      rt_buildLegend(numCC);
    }

    function rt_drawAnimFrame() {
      if (!rt_animSteps.length) return;
      const step = rt_animSteps[Math.min(rt_animIdx, rt_animSteps.length-1)];
      const lbl = step.lbl;
      const numCC = step.c-1 || 0;

      rt_ctx.clearRect(0, 0, rt_W, rt_H);

      for (let r=0; r<rt_ROWS; r++) {
        for (let c=0; c<rt_COLS; c++) {
          const x = rt_PAD + c * rt_CELL, y = rt_PAD + r * rt_CELL;
          const l = lbl[r][c];
          const active = rt_grid[r][c];

          if (active && l) {
            const col = rt_getColor(l);
            rt_ctx.fillStyle = col.fill; rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = col.stroke; rt_ctx.lineWidth = 1.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.fillStyle = col.text;
            rt_ctx.font = 'bold 11px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText(l, x+rt_CELL/2, y+rt_CELL/2);
          } else if (active) {
            rt_ctx.fillStyle = '#fef5e7';
            rt_ctx.fillRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
            rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 1; rt_ctx.setLineDash([2,2]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
            rt_ctx.setLineDash([]);
            rt_ctx.fillStyle = '#b9770e';
            rt_ctx.font = '9px monospace';
            rt_ctx.textAlign = 'center'; rt_ctx.textBaseline = 'middle';
            rt_ctx.fillText('?', x+rt_CELL/2, y+rt_CELL/2);
          } else {
            rt_ctx.fillStyle = '#fafaf7';
            rt_ctx.strokeStyle = '#e4dcc8'; rt_ctx.lineWidth = 0.5; rt_ctx.setLineDash([]);
            rt_ctx.strokeRect(x+0.5, y+0.5, rt_CELL-1, rt_CELL-1);
          }

          if (r===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(c, x+rt_CELL/2, rt_PAD/2); }
          if (c===0) { rt_ctx.fillStyle='#8a8371'; rt_ctx.font='8px monospace'; rt_ctx.textAlign='center'; rt_ctx.fillText(r, rt_PAD/2, y+rt_CELL/2); }
        }
      }

      if (step.type==='raster' || step.type==='found') {
        const x = rt_PAD + step.cc * rt_CELL, y = rt_PAD + step.r * rt_CELL;
        rt_ctx.strokeStyle = '#c0392b'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }
      if (step.type==='fill') {
        const x = rt_PAD + step.j * rt_CELL, y = rt_PAD + step.i * rt_CELL;
        rt_ctx.strokeStyle = '#b9770e'; rt_ctx.lineWidth = 2; rt_ctx.setLineDash([]);
        rt_ctx.strokeRect(x+1, y+1, rt_CELL-2, rt_CELL-2);
      }

      const descriptions = {
        raster: 'Varredura: (' + step.r + ',' + (step.cc||0) + ') — buscando não rotulado',
        found: 'Pixel em (' + step.r + ',' + (step.cc||0) + ')! Flood-fill: C' + step.c,
        fill: 'Preenchimento: (' + (step.i||0) + ',' + (step.j||0) + ') rotulado C' + step.c,
        done: 'Concluído! ' + step.c + ' componente(s).'
      };
      const titles = { raster: 'Varredura', found: 'Semente!', fill: 'Preenchendo...', done: 'Pronto!' };
      
      root.querySelector('#rt_animTitle').textContent = titles[step.type] || '–';
      root.querySelector('#rt_animDesc').textContent = descriptions[step.type] || '–';
      root.querySelector('#rt_statStep').textContent = (rt_animIdx+1) + '/' + rt_animSteps.length;
      root.querySelector('#rt_statCC').textContent = numCC;
      root.querySelector('#rt_statPx').textContent = rt_grid.flat().filter(Boolean).length;
      rt_buildLegend(numCC);
    }

    function rt_buildLegend(n) {
      const box = root.querySelector('#rt_legendBox');
      box.innerHTML = '';
      if (n === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Nenhum píxel ativo</span>';
        return;
      }
      for (let i=1; i<=n; i++) {
        const p = rt_PALETTE[(i-1) % rt_PALETTE.length];
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + p[1] + '; border:1px solid ' + p[0] + '; display:inline-block"></span><span style="color:' + p[0] + '; font-weight:bold">C' + i + '</span>';
        box.appendChild(span);
      }
    }

    function rt_redraw() {
      if (rt_mode === 'label') rt_drawLabel();
      else rt_drawAnimFrame();
    }

    window.rt_setConn = function(c) {
      rt_connectivity = c;
      root.querySelector('#rt_btn4').classList.toggle('rt_active', c===4);
      root.querySelector('#rt_btn8').classList.toggle('rt_active', c===8);
      root.querySelector('#rt_statConn').textContent = c;
      root.querySelector('#rt_connDesc').textContent = c===4 ? '4 vizinhos ortogonais: N, S, L, O' : '8 vizinhos (inclui diagonais)';
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_setMode = function(m) {
      rt_mode = m;
      window.rt_stopPlay();
      root.querySelector('#rt_btnLabel').classList.toggle('rt_active', m==='label');
      root.querySelector('#rt_btnAnim').classList.toggle('rt_active', m==='anim');
      root.querySelector('#rt_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      root.querySelector('#rt_animStatus').style.display = m==='anim' ? 'block' : 'none';
      if (m==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    };

    window.rt_animStep = function(d) {
      rt_animIdx = Math.max(0, Math.min(rt_animSteps.length-1, rt_animIdx+d));
      rt_drawAnimFrame();
    };

    window.rt_togglePlay = function() {
      if (rt_playing) window.rt_stopPlay(); else window.rt_startPlay();
    };

    window.rt_startPlay = function() {
      rt_playing = true;
      root.querySelector('#rt_btnPlay').textContent = '⏸';
      function tick() {
        if (rt_animIdx >= rt_animSteps.length-1) { window.rt_stopPlay(); return; }
        rt_animIdx++;
        rt_drawAnimFrame();
        const spd = +root.querySelector('#rt_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        rt_playTimer = setTimeout(tick, delay);
      }
      tick();
    };

    window.rt_stopPlay = function() {
      rt_playing = false;
      if (rt_playTimer) { clearTimeout(rt_playTimer); rt_playTimer = null; }
      root.querySelector('#rt_btnPlay').textContent = '▶';
    };

    window.rt_clearGrid = function() {
      rt_grid = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
      rt_animIdx = 0; rt_animSteps = [];
      window.rt_stopPlay();
      rt_redraw();
    };

    const rt_PRESETS = {
      diagonal: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let i=1; i<=5; i++) g[i][i]=1;
        [[2,8],[2,9],[3,8],[3,9],[3,10], [6,1],[6,2],[7,1],[7,2]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      letters: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        [[1,1],[2,1],[3,1],[4,1],[5,1],[5,2],[5,3], [1,5],[1,6],[1,7],[2,5],[2,7],[3,5],[3,7],[4,5],[4,7],[5,5],[5,6],[5,7], [1,9],[1,10],[2,9],[3,9],[4,9],[5,9],[5,10]].forEach(([r,c]) => g[r][c]=1);
        return g;
      },
      ring: () => {
        const g = Array.from({length:rt_ROWS}, () => new Array(rt_COLS).fill(0));
        for (let c=2; c<=9; c++) { g[2][c]=1; g[8][c]=1; }
        for (let r=3; r<=7; r++) { g[r][2]=1; g[r][9]=1; }
        for (let c=4; c<=7; c++) { g[4][c]=1; g[6][c]=1; }
        for (let r=5; r<=5; r++) { g[r][4]=1; g[r][7]=1; }
        return g;
      }
    };

    window.rt_loadPreset = function(name) {
      rt_grid = rt_PRESETS[name]();
      rt_animIdx = 0; window.rt_stopPlay();
      if (rt_mode==='anim') rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity);
      rt_redraw();
    };

    function rt_cellAt(e) {
      const rect = rt_cv.getBoundingClientRect();
      const sx = rt_W / rect.width, sy = rt_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - rt_PAD) / rt_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - rt_PAD) / rt_CELL);
      return {r, c};
    }

    rt_cv.addEventListener('mousedown', function(e){
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS) return;
      rt_painting = true;
      rt_paintVal = rt_grid[r][c] ? 0 : 1;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });

    rt_cv.addEventListener('mousemove', function(e){
      if (!rt_painting) return;
      const {r, c} = rt_cellAt(e);
      if (r<0 || r>=rt_ROWS || c<0 || c>=rt_COLS || rt_grid[r][c] === rt_paintVal) return;
      rt_grid[r][c] = rt_paintVal;
      if (rt_mode==='anim') { rt_animIdx=0; rt_animSteps=rt_buildAnimSteps(rt_grid, rt_connectivity); }
      rt_redraw();
    });
    
    document.addEventListener('mouseup', () => rt_painting = false);

    rt_grid = rt_PRESETS.diagonal();
    rt_drawLabel();
  }

  function tryInitSim04RotulacaoCC(){
    var root = document.getElementById('sim-04-rotulacao');
    if (root) initSim04RotulacaoCC(root); else setTimeout(tryInitSim04RotulacaoCC, 200);
  }
  tryInitSim04RotulacaoCC();
})();
</script>
""")

**Figure 4.16:** Interactive simulator for connected component labeling: visualization of flood-fill expansion, 4-connectivity and 8-connectivity.


<figure id="fig-04-sim-04-rotulacao">
  <img src="imagens/fig-04-sim-04-rotulacao.png" alt=" Interactive simulator for connected component labeling: visualization of flood-fill expansion, 4-connectivity and 8-connectivity. " style="max-width:80%" />
  <figcaption><strong>Figure 4.16:</strong>  Interactive simulator for connected component labeling: visualization of flood-fill expansion, 4-connectivity and 8-connectivity. </figcaption>
</figure>

In [42]:
%%writefile tmp/fig_04_rotulacao_didatico.cpp
#define MM_OUT "tmp/fig_04_rotulacao_didatico.png"
// Compile: g++ -std=c++17 -o program program.cpp -I. $(pkg-config --cflags --libs opencv4) -DMM_USE_OPENCV
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <set>

//| label: fig-04-rotulacao-didatico
//| fig-cap: "Efeito da conectividade na rotulação (mm::label0): pixels diagonalmente adjacentes formam componentes distintas em conectividade-4 e se fundem em conectividade-8."
//| echo: true
//| output: true

int main() {
    // Build f from the numpy array literal (10x10, *255)
    mm::Image f(10, 10);
    {
        int raw[10][10] = {
            {0,0,0,0,0,0,0,0,0,0},
            {0,1,0,0,0,0,0,0,0,0},
            {0,0,1,0,0,0,0,0,0,0},
            {0,0,0,1,0,0,1,1,0,0},
            {0,0,0,0,0,0,1,1,0,0},
            {0,0,0,0,0,0,0,0,0,0},
            {0,1,1,1,0,0,0,0,0,0},
            {0,1,0,1,0,0,0,1,0,0},
            {0,1,1,1,0,0,0,0,1,0},
            {0,0,0,0,0,0,0,0,0,0}
        };
        for (int y = 0; y < 10; y++)
            for (int x = 0; x < 10; x++)
                f.at(y, x) = (unsigned char)(raw[y][x] * 255);
    }

    mm::Image B4 = mm::secross();   // conectividade-4
    mm::Image B8 = mm::sebox();     // conectividade-8

    mm::Image lbl4 = mm::label0(f, B4);
    mm::Image lbl8 = mm::label0(f, B8);

    int n4 = 0, n8 = 0;
    // max() equivalent: scan for max value
    for (int i = 0; i < (int)lbl4.data.size(); i++) if (lbl4.data[i] > n4) n4 = lbl4.data[i];
    for (int i = 0; i < (int)lbl8.data.size(); i++) if (lbl8.data[i] > n8) n8 = lbl8.data[i];

    std::cout << "Componentes C4: " << n4 << "  |  C8: " << n8 << "\n";

    // norm_label as a lambda
    auto norm_label = [](const mm::Image& lbl, int mx) -> mm::Image {
        mm::Image out = lbl;
        for (int y = 0; y < lbl.h; y++) {
            for (int x = 0; x < lbl.w; x++) {
                if (lbl.at(y, x))
                    out.at(y, x) = (unsigned char)((int)lbl.at(y, x) * 255 / mx);
            }
        }
        return out;
    };

    int mx4 = std::max(n4, 1);
    int mx8 = std::max(n8, 1);
    mm::Image n4_img = norm_label(lbl4, mx4);
    mm::Image n8_img = norm_label(lbl8, mx8);

    mm::show(
        std::vector<mm::Image>{f, n4_img, n8_img},
        MM_OUT,
        std::vector<std::string>{
            "f original",
            "C4 (" + std::to_string(n4) + " comp.)",
            "C8 (" + std::to_string(n8) + " comp.)"
        },
        3
    );

    return 0;
}

Overwriting tmp/fig_04_rotulacao_didatico.cpp


In [43]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_rotulacao_didatico.cpp -o tmp/fig_04_rotulacao_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_rotulacao_didatico \
  && test -f "tmp/fig_04_rotulacao_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_rotulacao_didatico.png"

Componentes C4: 7  |  C8: 4
[1] f original
[2] C4 (7 comp.)
[3] C8 (4 comp.)


In [44]:
mm.show(mm.read("tmp/fig_04_rotulacao_didatico.png"))

<Figure size 784x450 with 1 Axes>

**Figure 4.17:** Efeito da conectividade na rotulação (*mm::label0*): pixels diagonalmente adjacentes formam componentes distintas em conectividade-4 e se fundem em conectividade-8.


### 4.4.2 Distance Transform

The **Distance Transform** (DT) is an operator that, applied to a binary image $f$, produces a grayscale image $D$ in which each pixel belonging to the object ($f(x,y)\neq 0$) receives as its value the geometric distance to the nearest background pixel ($f(x',y')=0$):

$$
D(x,y) = \min_{(x',y') \,:\, f(x',y')=0} \; d\bigl((x,y),\,(x',y')\bigr)
$$

where $d(\cdot,\cdot)$ is a distance metric — typically the Euclidean distance ($L_2$). The result is a topographic representation of the objects: pixels located in the interior assume high values, whereas pixels near the borders exhibit low distance values. The **local maxima** of $D$ correspond to the points farthest from the object boundary, often near their geometric centers or centers of maximal inscription — a property particularly useful for the automatic generation of markers in the *watershed* algorithm.

> ### 📝 Formal definition using erosions
>
> The DT also admits an iterative morphological interpretation, according to the algorithm in [Figure 4.18](#fig-04-sim-alg-distancia2). Consider a structuring function $b$ whose central value is zero and whose neighbors have negative costs associated with the displacement. By applying successive erosions with this particular structuring function, the pixel values of the objects (which must assume the maximum possible distance in the image) are progressively reduced according to the costs defined by $b$. The accumulated value of this propagation then comes to represent the distance to the background according to the metric induced by the structuring function.

This interpretation is implemented in `mm::dist1()`, which accumulates successive erosions using the operation `mm::ero1()`. In turn, `mm::dist()` delegates the Euclidean distance computation to the optimized OpenCV operator `mm::dist(f)`, where `f` is the input binary image, the L2 distance specifies the Euclidean metric ($L_2$), and `5` indicates the use of a 5×5 mask to approximate the distance with high precision.

The `dist1` function produces a discrete distance transform whose metric is determined by the geometry and weights of the structuring function used. For example, using a cross-shaped structuring function with unit cost for the four orthogonal neighbors yields the ***Manhattan*** distance ($L_1$). Other choices of neighborhood and weights induce different metrics. In turn, `mm::dist()` computes an efficient approximation of the Euclidean distance ($L_2$).

Because it requires successive erosions over the entire image, the `dist1` approach has a significantly higher computational cost than `mm::dist()`, and it is employed in this book mainly for didactic purposes and to highlight the relationship between mathematical morphology and distance transforms.

In [45]:
# @title { display-mode: "form" }
# Exclusive prefix to avoid conflict with other notebook cells
PREFIX = "dist2"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#E6F1FB;color:#185FA5}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
.metric-elem{{display:inline-grid;grid-template-columns:repeat(3,16px);grid-template-rows:repeat(3,16px);gap:2px;margin-top:6px}}
.mc{{width:16px;height:16px;border-radius:2px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:500}}
.mc-on{{background:#E6F1FB;color:#185FA5}}
.mc-off{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}
.mc-ctr{{background:#185FA5;color:#fff}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Transformada de distância por erosões numéricas sucessivas — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-ripple" aria-hidden="true"></i></div>
    <div><div class="algo-title">Transformada de distância por erosão numérica</div><div class="algo-sub">Propagação matemática de distâncias via elemento estruturante com pesos</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Inicializar a imagem de trabalho fazendo uma cópia da original: <code>g ← f.copy()</code>. Os pixels de fundo (0) servem como fontes de distância nula.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Entrar em um laço infinito de erosões com pesos (ponto fixo):
        <div class="step-sub">
          <div class="step-sub-item">Salvar estado anterior: <code>f ← g.copy()</code>.</div>
          <div class="step-sub-item">Erodir: <code>g ← ero1(g, b)</code>, aplicando a subtração local de pesos e computando o valor mínimo para cada vizinhança.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-blue">3</div><div class="step-text">Verificar convergência: se <code>f</code> for idêntica a <code>g</code> (<code>array_equal</code>), a frente de onda de distâncias se estabilizou. Romper o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Retornar a matriz modificada <code>g</code> contendo o mapa exato de distâncias.</div></div>
    <div class="note">Nesta abordagem morfológica numérica, não há incremento artificial ou contador. A distância propaga-se de fora para dentro porque a erosão contínua puxa o valor <code>0</code> do fundo e o decrementa matematicamente (subtraindo os pesos negativos como <code>-1</code>), fazendo com que os valores escalem radialmente.</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Cruz — L₁ (Manhattan)</div>
    <div class="metric-formula">B_cruz [y,x]</div>
    <div class="metric-desc">Pesos: Centro=0, Lados=-1, Cantos=-inf</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#854F0B">Erosão de Cinzas</div>
    <div class="metric-formula">f[vy,vx] - bv</div>
    <div class="metric-desc">Subtrai o peso e busca o valor mínimo local</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Convergência</div>
    <div class="metric-formula">f == g</div>
    <div class="metric-desc">Para quando nenhum pixel muda de valor</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Clonar imagem de entrada: <code>g ← f.copy()</code>. O objeto possui intensidade alta (255) e o fundo possui intensidade 0.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Mapeamento Local (ero1)</div>
      <div class="card-text">Para cada coordenada <code>(y, x)</code>, buscar o mínimo valor da operação <code>f[vy, vx] - bv</code> aplicada à sua vizinhança estruturante.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Loop Iterativo</div>
      <div class="card-text">Atualizar sequencialmente: <code>f = g.copy()</code> seguido de <code>g = ero1(g, b)</code>. Os valores nulos propagam-se para o interior do objeto.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Critério de Parada</div>
      <div class="card-text">Se <code>np.array_equal(f, g)</code>, significa que o mapa de distâncias atingiu o equilíbrio estável e a propagação terminou.</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Cópia de Trabalho</div><div class="tl-desc">Prepara a matriz inicial `g`.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Loop de Erosão de Escala</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Guarda estado: <code>f ← g.copy()</code></div>
          <div class="step-sub-item">Aplica erosão com pesos: <code>g ← ero1(g, b)</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Estabilização Espacial</div><div class="tl-desc">Condição de parada acionada assim que <code>np.array_equal(f, g)</code> se torna verdadeiro.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">04</span></div>
    <div><div class="tl-title">Retorno Numérico</div><div class="tl-desc">Retorna <code>g</code> contendo as distâncias calculadas pela subtração cumulativa dos pesos.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>morph_dist.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Erosão numérica iterativa</span>
  </div>
  <div class="code-body">
<pre><span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">ero1</span>(f, b):
    g = np.empty_like(f)
    <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
            g[y,x] = <span class="num-lit">255</span>
            <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f,b,y,x):
                <span class="kw">if</span> np.isinf(bv): <span class="kw">continue</span> 
                val = int(f[vy,vx]) - int(bv)
                <span class="kw">if</span> g[y,x] > val: 
                    g[y,x] = max(0, val)
    <span class="kw">return</span> g

<span class="kw">@staticmethod</span>
<span class="kw">def</span> <span class="fn">dist1</span>(f, b):
    g = f.copy()
    <span class="kw">while</span> <span class="fn">True</span>:
        f = g.copy()
        g = mm.ero1(g, b)
        <span class="kw">if</span> np.array_equal(f, g): 
            <span class="kw">break</span>
    <span class="kw">return</span> g</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>g[y,x] = 255</code> — Inicializa o elemento com o valor máximo antes de computar o operador de mínimo da erosão.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>f[vy,vx] - bv</code> — Subtrai o peso associado da vizinhança. Como os pesos da cruz externa são negativos (ex: <code>-1</code>), a operação torna-se uma adição matemática (<code>f[vy,vx] - (-1) = f[vy,vx] + 1</code>) propagando a distância a partir das bordas zeradas.</div></div>
  <div class="ann-line"><div class="ann-badge num-blue">3</div><div class="ann-text"><code>np.array_equal(f, g)</code> — Critério de convergência exato por estabilização de ponto fixo.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 560 720" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:560px;display:block">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="150" y="85" width="260" height="46" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">inicializar mapa</text>
  <text x="280" y="120" text-anchor="middle" font-size="12" fill="#085041">g ← f.copy()</text>

  <rect x="150" y="165" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="184" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">salvar estado anterior</text>
  <text x="280" y="200" text-anchor="middle" font-size="12" fill="#712B13">f ← g.copy()</text>

  <rect x="150" y="245" width="260" height="46" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="264" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">executar erosão com pesos</text>
  <text x="280" y="280" text-anchor="middle" font-size="12" fill="#712B13">g ← ero1(g, b)</text>

  <polygon points="280,325 390,355 280,385 170,355" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="351" text-anchor="middle" font-size="12" fill="#0C447C">array_equal(f, g)</text>
  <text x="280" y="367" text-anchor="middle" font-size="12" fill="#0C447C">estabilizou?</text>

  <line x1="170" y1="355" x2="100" y2="355" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="355" x2="100" y2="188" stroke="#993C1D" stroke-width="1.2"/>
  <line x1="100" y1="188" x2="150" y2="188" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="120" y="345" text-anchor="middle" font-size="11" fill="#993C1D">não</text>

  <line x1="280" y1="385" x2="280" y2="430" stroke="#0F6E56" stroke-width="1.2" marker-end="url(#arr-g)"/>
  <text x="295" y="405" font-size="11" fill="#0F6E56">sim</text>

  <rect x="150" y="430" width="260" height="46" rx="6" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="458" text-anchor="middle" font-size="12" font-weight="500" fill="#27500A">retornar g</text>

  <ellipse cx="280" cy="525" rx="50" ry="20" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="529" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>
  <line x1="280" y1="476" x2="280" y2="505" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="131" x2="280" y2="165" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="211" x2="280" y2="245" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="291" x2="280" y2="325" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>

  <text x="280" y="600" text-anchor="middle" font-size="11.5" fill="#888780" font-style="italic">Ponto Fixo Numérico: A distância emerge da propagação matemática do valor zero.</text>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.18:** Distance Transform Algorithm.


<figure id="fig-04-sim-alg-distancia2">
  <img src="imagens/fig-04-sim-alg-distancia2.png" alt=" Distance Transform Algorithm. " style="max-width:80%" />
  <figcaption><strong>Figure 4.18:</strong>  Distance Transform Algorithm. </figcaption>
</figure>

[Figure 4.19](#fig-04-sim-04-distancia) presents an interactive simulator for the TD: by positioning the cursor over different pixels of the object, it is possible to observe, in real time, the distance value associated with that position—that is, the distance to the nearest background pixel. [Figure 4.20](#fig-04-distancia-didatico) presents a practical example of this execution in a Python environment.

In [46]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-distancia" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-distancia * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-distancia canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-distancia button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-distancia button:hover { background: #e8dfcf; }
  #sim-04-distancia button.sim04_td_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-distancia .sim04_td_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim04_td_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim04_td_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .sim04_td_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim04_td_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim04_td_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .sim04_td_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 120px; margin: 6px auto; }
  .sim04_td_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .sim04_td_se_btn.sim04_td_inf { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🗺️ Simulator: Distance Transform (DT)</span>
  <span class="sim04_td_pill">Borders at +∞ (144)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="sim04_td_grid_stats">
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Active Pixels</div>
      <div id="sim04_td_statPx" class="sim04_td_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Max Distance</div>
      <div id="sim04_td_statMax" class="sim04_td_stat_value" style="color:#2980b9;">0</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Current Metric</div>
      <div id="sim04_td_statMetric" class="sim04_td_stat_value" style="color:#b9770e; font-size:11px;">L∞ (Chebyshev)</div>
    </div>
    <div class="sim04_td_stat_box">
      <div class="sim04_td_stat_label">Iteration (k)</div>
      <div id="sim04_td_statStep" class="sim04_td_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas e Legenda -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="sim04_td_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Click to toggle pixels · Drag to paint
      </div>
      <div style="display:flex; flex-wrap:wrap; gap:6px; margin-top:10px; align-items:center; justify-content:center;" id="sim04_td_legendBox"></div>
    </div>

    <!-- Painel de Controles Lateral -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Editor do Elemento Estruturante (b) -->
      <div class="sim04_td_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Structuring Element (b)
        </div>
        <div style="font-size:9.5px; color:#8a8371; margin-bottom:6px; text-align:left;">Click to change weights:</div>
        
        <div class="sim04_td_se_grid">
          <button id="sim04_td_se_0_0" class="sim04_td_se_btn sim04_td_active" data-se="0,0">-1</button>
          <button id="sim04_td_se_0_1" class="sim04_td_se_btn sim04_td_active" data-se="0,1">-1</button>
          <button id="sim04_td_se_0_2" class="sim04_td_se_btn sim04_td_active" data-se="0,2">-1</button>
          
          <button id="sim04_td_se_1_0" class="sim04_td_se_btn sim04_td_active" data-se="1,0">-1</button>
          <button id="sim04_td_se_1_1" class="sim04_td_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>0</button>
          <button id="sim04_td_se_1_2" class="sim04_td_se_btn sim04_td_active" data-se="1,2">-1</button>
          
          <button id="sim04_td_se_2_0" class="sim04_td_se_btn sim04_td_active" data-se="2,0">-1</button>
          <button id="sim04_td_se_2_1" class="sim04_td_se_btn sim04_td_active" data-se="2,1">-1</button>
          <button id="sim04_td_se_2_2" class="sim04_td_se_btn sim04_td_active" data-se="2,2">-1</button>
        </div>
        
        <div style="display:flex; flex-wrap:wrap; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">L1 (Cross)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">L∞ (Square)</button>
          <button data-se-preset="chamfer" style="flex:1; font-size:9.5px; min-width:90px;">Chamfer 3-4</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualization
        </div>
        <div style="display:flex; gap:6px;">
          <button id="sim04_td_btnLabel" class="sim04_td_active" data-mode="label" style="flex:1; justify-content:center;">Final (DT)</button>
          <button id="sim04_td_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animated</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="sim04_td_animCtrl" class="sim04_td_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Step-by-Step Propagation
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="sim04_td_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Speed</label>
          <input type="range" id="sim04_td_speedSlider" min="1" max="10" value="5" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos f -->
      <div class="sim04_td_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Preset Examples
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="square" style="justify-content:flex-start;">■ Solid Square</button>
          <button data-preset="lshape" style="justify-content:flex-start;">╚ L Shape</button>
          <button data-preset="ring" style="justify-content:flex-start;">◎ Ring with Hole</button>
          <button data-preset="corner0" style="justify-content:flex-start;">↘ Background at (0,0)</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Clear Grid
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04TD(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const td_COLS = 12, td_ROWS = 12, td_CELL = 22, td_PAD = 10;
    const td_W = td_COLS * td_CELL + td_PAD * 2, td_H = td_ROWS * td_CELL + td_PAD * 2;
    const td_cv = root.querySelector('#sim04_td_Canvas');
    td_cv.width = td_W; 
    td_cv.height = td_H;
    const td_ctx = td_cv.getContext('2d');
    
    const td_MAX_DIST = 144;

    const td_COLORS = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72', '#5b2c6f', '#4a235a'];
    function td_getColor(d, maxD) {
      if(d === 0) return { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' };
      if(d >= td_MAX_DIST) return { fill: '#fdecea', text: '#c0392b', stroke: '#f5b7b1' };
      
      let ratio = maxD > 1 ? (d - 1) / (maxD - 1) : 0;
      let idx = Math.max(0, Math.min(td_COLORS.length - 1, Math.floor(ratio * (td_COLORS.length - 1))));
      
      const bg = td_COLORS[idx];
      const text = (idx > 4) ? '#ffffff' : '#26241d';
      return { fill: bg, text: text, stroke: '#2980b9' };
    }

    let td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
    
    const td_VALS = [-1, -2, -3, -4, -5, -6, -7, -8, -9, -Infinity];
    let td_b = [
      [-1, -1, -1],
      [-1,  0, -1],
      [-1, -1, -1]
    ];
    
    let td_mode = 'label';
    let td_painting = false;
    let td_paintVal = 1;

    let td_animSteps = [];
    let td_animIdx = 0;
    let td_playing = false;
    let td_playTimer = null;

    function td_compute() {
      let f = td_grid;
      let g = Array.from({length: td_ROWS}, (_, r) => 
        Array.from({length: td_COLS}, (_, c) => f[r][c] ? td_MAX_DIST : 0)
      );
      let steps = [];
      let k = 0;

      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(g), eroded: Array.from({length:td_ROWS}, ()=>new Array(td_COLS).fill(0)), k: k });

      let activePx = f.flat().reduce((a,b)=>a+b, 0);
      if(activePx === 0) return { D: g, steps, max: 0 };

      while (true) {
        k++;
        let nextG = Array.from({length: td_ROWS}, () => new Array(td_COLS).fill(0));
        let changed = false;
        let erodedCells = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));

        for (let r=0; r<td_ROWS; r++) {
          for (let c=0; c<td_COLS; c++) {
            if (f[r][c] === 1) {
              let min_val = g[r][c];
              
              for (let dr=-1; dr<=1; dr++) {
                for (let dc=-1; dc<=1; dc++) {
                  let b_val = td_b[dr+1][dc+1];
                  if (b_val !== -Infinity && !(dr===0 && dc===0)) {
                    let nr = r + dr, nc = c + dc;
                    let neighbor_val = td_MAX_DIST;
                    
                    if (nr >= 0 && nr < td_ROWS && nc >= 0 && nc < td_COLS) {
                      neighbor_val = g[nr][nc];
                    }
                    
                    let val = neighbor_val - b_val; 
                    if (val < min_val) {
                      min_val = val;
                    }
                  }
                }
              }
              nextG[r][c] = min_val;
              if (min_val !== g[r][c]) {
                changed = true;
                erodedCells[r][c] = 1;
              }
            } else {
              nextG[r][c] = 0;
            }
          }
        }
        
        if (!changed) {
          steps.push({ g: copy(nextG), eroded: erodedCells, k: k, done: true });
          break;
        }
        
        steps.push({ g: copy(nextG), eroded: erodedCells, k: k });
        g = nextG;
      }
      
      let maxD = 0;
      for(let r=0; r<td_ROWS; r++){
        for(let c=0; c<td_COLS; c++){
           if(f[r][c] === 1 && g[r][c] < td_MAX_DIST && g[r][c] > maxD) maxD = g[r][c];
        }
      }
      return { D: g, steps, max: maxD };
    }

    function td_drawFinal() {
      const res = td_compute();
      const D = res.D;
      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            const col = td_getColor(d, res.max);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
          } else {
            td_ctx.fillStyle = '#fafaf7'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statMax').textContent = res.max;
      root.querySelector('#sim04_td_statStep').textContent = 'Finalizado';
      td_buildLegend(res.D, res.max);
    }

    function td_drawAnimFrame() {
      if (!td_animSteps.length) return;
      const step = td_animSteps[Math.min(td_animIdx, td_animSteps.length-1)];
      const D = step.g;
      const eroded = step.eroded;
      
      let currentMaxD = 0;
      for(let r=0; r<td_ROWS; r++) {
        for(let c=0; c<td_COLS; c++) {
          if(td_grid[r][c] && D[r][c] < td_MAX_DIST && D[r][c] > currentMaxD) currentMaxD = D[r][c];
        }
      }

      td_ctx.clearRect(0, 0, td_W, td_H);

      for (let r=0; r<td_ROWS; r++) {
        for (let c=0; c<td_COLS; c++) {
          const x = td_PAD + c * td_CELL, y = td_PAD + r * td_CELL;
          const d = D[r][c];
          const active = td_grid[r][c];

          if (active) {
            let maxValArray = td_animSteps[td_animSteps.length-1].g.flat().filter(v=>v<td_MAX_DIST);
            const globalMax = maxValArray.length > 0 ? Math.max(...maxValArray) : 1;
            const col = td_getColor(d, globalMax);
            td_ctx.fillStyle = col.fill; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            
            if(eroded[r][c] === 1) {
               td_ctx.strokeStyle = '#b9770e'; td_ctx.lineWidth = 2; td_ctx.setLineDash([2,2]);
               td_ctx.fillStyle = 'rgba(185, 119, 14, 0.15)'; td_ctx.fillRect(x+1, y+1, td_CELL-2, td_CELL-2);
            } else {
               td_ctx.strokeStyle = col.stroke; td_ctx.lineWidth = 1; td_ctx.setLineDash([]);
            }
            
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = col.text;
            td_ctx.font = d >= td_MAX_DIST ? 'bold 9px monospace' : 'bold 11px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText(d >= td_MAX_DIST ? '144' : d, x+td_CELL/2, y+td_CELL/2);
            td_ctx.setLineDash([]);
          } else {
            td_ctx.fillStyle = '#fafaf7';
            td_ctx.strokeStyle = '#e4dcc8'; td_ctx.lineWidth = 0.5; td_ctx.setLineDash([]);
            td_ctx.strokeRect(x+0.5, y+0.5, td_CELL-1, td_CELL-1);
            td_ctx.fillStyle = '#8a8371';
            td_ctx.font = '9px monospace';
            td_ctx.textAlign = 'center'; td_ctx.textBaseline = 'middle';
            td_ctx.fillText('0', x+td_CELL/2, y+td_CELL/2);
          }

          if (r===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(c, x+td_CELL/2, td_PAD/2); }
          if (c===0) { td_ctx.fillStyle='#8a8371'; td_ctx.font='8px monospace'; td_ctx.textAlign='center'; td_ctx.fillText(r, td_PAD/2, y+td_CELL/2); }
        }
      }

      root.querySelector('#sim04_td_statMax').textContent = currentMaxD;
      root.querySelector('#sim04_td_statPx').textContent = td_grid.flat().filter(Boolean).length;
      root.querySelector('#sim04_td_statStep').textContent = step.done ? 'Concluído' : step.k;
      td_buildLegend(D, currentMaxD);
    }

    function td_buildLegend(D_matrix, maxD) {
      const box = root.querySelector('#sim04_td_legendBox');
      box.innerHTML = '';
      
      let uniqueD = new Set();
      D_matrix.forEach(row => row.forEach(val => { 
        if(val > 0 && val < td_MAX_DIST) uniqueD.add(val); 
      }));
      let sortedD = Array.from(uniqueD).sort((a,b) => a-b);
      
      if (sortedD.length === 0) {
        box.innerHTML = '<span style="font-size:9.5px; color:#8a8371;">Sem propagação visível</span>';
        return;
      }
      
      let displayArr = sortedD;
      if (sortedD.length > 12) {
          displayArr = sortedD.filter((_, i) => i === 0 || i === sortedD.length-1 || i%Math.ceil(sortedD.length/10) === 0);
      }
      
      box.innerHTML = '<span style="font-size:9.5px; color:#8a8371; width:100%;">Cores (exceto 144):</span>';
      
      displayArr.forEach(d => {
        const col = td_getColor(d, maxD);
        const span = document.createElement('span');
        span.style.cssText = 'display:flex; align-items:center; gap:4px; font-size:9.5px; color:#5e5a4a;';
        span.innerHTML = '<span style="width:12px; height:12px; border-radius:3px; background:' + col.fill + '; border:1px solid ' + col.stroke + '; display:inline-block"></span><span>' + d + '</span>';
        box.appendChild(span);
      });
    }

    function td_updateSEUI() {
      let typeName = "Personalizada";
      let isCross = td_b[0][1]===-1 && td_b[1][0]===-1 && td_b[1][2]===-1 && td_b[2][1]===-1 && td_b[0][0]===-Infinity && td_b[0][2]===-Infinity && td_b[2][0]===-Infinity && td_b[2][2]===-Infinity;
      let isSquare = td_b.every((r, i) => r.every((v, j) => (i===1 && j===1) ? true : v===-1));
      let isChamfer = td_b[0][1]===-3 && td_b[1][0]===-3 && td_b[1][2]===-3 && td_b[2][1]===-3 && td_b[0][0]===-4 && td_b[0][2]===-4 && td_b[2][0]===-4 && td_b[2][2]===-4;

      if (isCross) typeName = "L1 (City-Block)";
      else if (isSquare) typeName = "L∞ (Chebyshev)";
      else if (isChamfer) typeName = "Chamfer 3-4";

      root.querySelector('#sim04_td_statMetric').textContent = typeName;

      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#sim04_td_se_' + r + '_' + c);
          const val = td_b[r][c];
          
          if(val !== -Infinity) {
            btn.textContent = String(val);
            btn.className = 'sim04_td_se_btn sim04_td_active';
          } else {
            btn.textContent = '-∞';
            btn.className = 'sim04_td_se_btn sim04_td_inf';
          }
        }
      }
    }

    function td_toggleSE(r, c) {
      if(r===1 && c===1) return;
      
      let currentVal = td_b[r][c];
      let idx = td_VALS.indexOf(currentVal);
      let nextVal = td_VALS[(idx + 1) % td_VALS.length];
      
      td_b[r][c] = nextVal;
      td_updateSEUI();
      td_refresh();
    };

    function td_setPresetSE(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) td_b[r][c] = -Infinity;
      td_b[1][1] = 0;

      if (type === 'cross') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -1;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) if(!(r===1 && c===1)) td_b[r][c] = -1;
      } else if (type === 'chamfer') {
        td_b[0][1] = td_b[1][0] = td_b[1][2] = td_b[2][1] = -3;
        td_b[0][0] = td_b[0][2] = td_b[2][0] = td_b[2][2] = -4;
      }
      
      td_updateSEUI();
      td_refresh();
    };

    function td_refresh() {
      if (td_mode === 'anim') {
        td_animSteps = td_compute().steps;
        td_animIdx = 0;
        td_drawAnimFrame();
      } else {
        td_drawFinal();
      }
    }

    function td_setMode(m) {
      td_mode = m;
      td_stopPlay();
      root.querySelector('#sim04_td_btnLabel').classList.toggle('sim04_td_active', m==='label');
      root.querySelector('#sim04_td_btnAnim').classList.toggle('sim04_td_active', m==='anim');
      root.querySelector('#sim04_td_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      td_refresh();
    }

    function td_animStep(d) {
      td_animIdx = Math.max(0, Math.min(td_animSteps.length-1, td_animIdx+d));
      td_drawAnimFrame();
    }

    function td_togglePlay() {
      if (td_playing) td_stopPlay(); else td_startPlay();
    }

    function td_startPlay() {
      td_playing = true;
      root.querySelector('#sim04_td_btnPlay').textContent = '⏸';
      function tick() {
        if (td_animIdx >= td_animSteps.length-1) { td_stopPlay(); return; }
        td_animIdx++;
        td_drawAnimFrame();
        const spd = +root.querySelector('#sim04_td_speedSlider').value;
        const delay = Math.round(1200 - spd * 100);
        td_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function td_stopPlay() {
      td_playing = false;
      if (td_playTimer) { clearTimeout(td_playTimer); td_playTimer = null; }
      root.querySelector('#sim04_td_btnPlay').textContent = '▶';
    }

    function td_clearGrid() {
      td_grid = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
      td_stopPlay();
      td_refresh();
    }

    const td_PRESETS = {
      square: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) for(let c=2; c<=9; c++) g[r][c] = 1;
        return g;
      },
      lshape: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let r=2; r<=9; r++) { g[r][2]=1; g[r][3]=1; g[r][4]=1; }
        for(let c=5; c<=9; c++) { g[7][c]=1; g[8][c]=1; g[9][c]=1; }
        return g;
      },
      ring: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(0));
        for(let c=2; c<=9; c++) { g[2][c]=1; g[3][c]=1; g[8][c]=1; g[9][c]=1; }
        for(let r=4; r<=7; r++) { g[r][2]=1; g[r][3]=1; g[r][8]=1; g[r][9]=1; }
        return g;
      },
      corner0: () => {
        const g = Array.from({length:td_ROWS}, () => new Array(td_COLS).fill(1));
        g[0][0] = 0;
        return g;
      }
    };

    function td_loadPreset(name) {
      td_grid = td_PRESETS[name]();
      td_stopPlay();
      td_refresh();
    }

    function td_cellAt(e) {
      const rect = td_cv.getBoundingClientRect();
      const sx = td_W / rect.width, sy = td_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - td_PAD) / td_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - td_PAD) / td_CELL);
      return {r, c};
    }

    td_cv.addEventListener('mousedown', function(e){
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS) return;
      td_painting = true;
      td_paintVal = td_grid[r][c] ? 0 : 1;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });

    td_cv.addEventListener('mousemove', function(e){
      if (!td_painting) return;
      const {r, c} = td_cellAt(e);
      if (r<0 || r>=td_ROWS || c<0 || c>=td_COLS || td_grid[r][c] === td_paintVal) return;
      td_grid[r][c] = td_paintVal;
      td_refresh();
    });
    
    document.addEventListener('mouseup', () => td_painting = false);

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        td_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        td_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') td_clearGrid();
        if(act === 'prev') td_animStep(-1);
        if(act === 'next') td_animStep(1);
        if(act === 'play') td_togglePlay();
      });
    });

    td_grid = td_PRESETS.corner0();
    td_updateSEUI();
    td_refresh();
  }

  function tryInitSim04TD(){
    var root = document.getElementById('sim-04-distancia');
    if (root) initSim04TD(root); else setTimeout(tryInitSim04TD, 200);
  }
  tryInitSim04TD();
})();
</script>
""")

**Figure 4.19:** Interactive simulator of the Distance Transform (DT) iterative via grayscale erosion. Pixels outside the image assume the maximum value (144), propagating costs from the internal background.


<figure id="fig-04-sim-04-distancia">
  <img src="imagens/fig-04-sim-04-distancia.png" alt=" Interactive simulator of the Distance Transform (DT) iterative via grayscale erosion. Pixels outside the image assume the maximum value (144), propagating costs from the internal background. " style="max-width:80%" />
  <figcaption><strong>Figure 4.19:</strong>  Interactive simulator of the Distance Transform (DT) iterative via grayscale erosion. Pixels outside the image assume the maximum value (144), propagating costs from the internal background. </figcaption>
</figure>

In [47]:
%%writefile tmp/fig_04_distancia_didatico.cpp
#define MM_OUT "tmp/fig_04_distancia_didatico.png"
//| label: fig-04-distancia-didatico
//| fig-cap: "Transformada de Distância em imagem binária 10×10. Esquerda: original (*foreground* = 255). Centro: mm.dist1 iterativa (erosões com cruz). Direita: mm.dist (L2)."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
    mm::Image f(10, 10);
    std::fill(f.data.begin(), f.data.end(), 255);
    f.at(0, 0) = 0;

    mm::SE B_cruz{{mm::SE_OUT, -1, mm::SE_OUT}, {-1, 0, -1}, {mm::SE_OUT, -1, mm::SE_OUT}};

    mm::Image d_iter = mm::dist1(f, B_cruz);
    mm::Image d_l2   = mm::dist(f);

    int max_d_iter = 0;
    int max_d_l2 = 0;
    for (int i = 0; i < d_iter.h * d_iter.w; ++i) {
        max_d_iter = std::max(max_d_iter, (int)d_iter.data[i]);
        max_d_l2 = std::max(max_d_l2, (int)d_l2.data[i]);
    }

    std::cout << "Máx. dist1 (erosões) : " << max_d_iter << " px\n";
    std::cout << "Máx. dist  (L2)      : " << max_d_l2 << " px\n";

    mm::show(
        std::vector<mm::Image>{f, d_iter, d_l2},
        MM_OUT,
        std::vector<std::string>{"f original", "mm.dist1 (erosões com cruz)", "mm.dist (L2)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(f, "tmp/fig_04_distancia_didatico_0.png");
mm::write(d_iter, "tmp/fig_04_distancia_didatico_1.png");
mm::write(d_l2, "tmp/fig_04_distancia_didatico_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_distancia_didatico.cpp


In [48]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_distancia_didatico.cpp -o tmp/fig_04_distancia_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_distancia_didatico \
  && test -f "tmp/fig_04_distancia_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_distancia_didatico.png"

Máx. dist1 (erosões) : 18 px
Máx. dist  (L2)      : 13 px
[1] f original
[2] mm.dist1 (erosões com cruz)
[3] mm.dist (L2)


In [49]:
mm.show(
    [
        mm.read("tmp/fig_04_distancia_didatico_0.png"),
        mm.read("tmp/fig_04_distancia_didatico_1.png"),
        mm.read("tmp/fig_04_distancia_didatico_2.png"),
    ],
    titles=[
        'f original',
        'mm.dist1 (erosões com cruz)',
        'mm.dist (L2)',
    ],
    cols=3,
    axis=True,
    figsize=(12, 4),
)

<Figure size 1800x600 with 3 Axes>

**Figure 4.20:** Transformada de Distância em imagem binária 10×10. Esquerda: original (*foreground* = 255). Centro: mm.dist1 iterativa (erosões com cruz). Direita: mm.dist (L2).


Annotating numerical values directly over the pixels allows verifying how `dist1` propagates distances according to the metric induced by the structuring function used. In the case of the cross element with unit cost, the obtained values correspond to the *Manhattan* distance ($L_1$). Although `dist1` and `mm::dist` produce distinct numerical values because they adopt different metrics, both transforms preserve the topographical structure of the objects, causing their maxima to occur in similar central regions. This property justifies the use of `mm::dist` in practical applications, due to its high computational efficiency.

### 4.4.3 Euclidean Distance Transform in four steps

The simulator in [Figure 4.21](#fig-04-sim-04-tde) implements the EDT algorithm from Lotufo (2001) in two stages. In the first stage, the function `edt1` performs a one-dimensional vertical transformation sequentially (*in-place*), traversing each column in *raster* (↓) and *anti-raster* (↑) order to compute distances in the vertical direction (two steps: South and North). In the second stage, the function `edt2` uses this result as input and performs horizontal propagation using queues: for each row of the matrix, two priority queues, `Eq` and `Wq`, are initialized by traversing the column indices in opposite directions (`Eq` from `W-1` to `1`, `Wq` from `2` to `W`), so that each updated pixel immediately enqueues its neighbors for reprocessing within the same round (two more steps: East and West). This queue mechanism allows successive erosions with increasing odd weights (`b = 1, 3, 5, ...`, incremented at each iteration of the outer loop) without propagation getting stuck on outdated values, since each round resolves the horizontal dependency chain completely before the next increment of `b`. The convergence of this propagation produces the Euclidean Distance Transform across the entire matrix, combining the vertical information obtained in `edt1` with the queue-based horizontal propagation performed in `edt2`.

In [50]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-tde" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-tde * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-tde canvas { display: block; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; margin: 0 auto; }
  #sim-04-tde button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all .15s ease; display: inline-flex; align-items: center; gap: 5px; font-weight: 600; }
  #sim-04-tde button:hover { background: #e8dfcf; }
  #sim-04-tde button.sim04_edt_act { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-tde .sim04_edt_smtxt { font-size: 10.5px; color: #8a8371; line-height: 1.4; }
  #sim-04-tde .sim04_edt_mono { font-family: monospace; }
  #sim-04-tde .sim04_edt_statcard { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 8px 10px; text-align: center; min-width: 0; }
  #sim-04-tde .sim04_edt_statlabel { font-size: 9.5px; text-transform: uppercase; letter-spacing: .04em; color: #8a8371; margin-bottom: 2px; font-weight: 700; }
  #sim-04-tde .sim04_edt_statval { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim04_edt_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📐 EDT² 2D Corrected · 4×4 Matrix</span>
  <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Exact Convergence</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição Teórica -->
  <div class="sim04_edt_panel" style="margin-bottom:14px;">
    <div style="font-size:11.5px; font-weight:700; color:#26241d; margin-bottom:2px;">Step 1: edt1 Vertical In-place · Step 2: edt2 Horizontal In-place</div>
    <div class="sim04_edt_smtxt">Uses the same raster/anti-raster structure described in the paper for exact distance propagation (paper example, p. 103).</div>
  </div>

  <!-- Cards de Estatísticas -->
  <div style="display:grid; grid-template-columns:repeat(3, 1fr); gap:10px; margin-bottom:14px;">
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Current Step</div><div class="sim04_edt_statval" id="sim04_edt_sStep" style="color:#c0392b;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Phase</div><div class="sim04_edt_statval" id="sim04_edt_sPhase" style="color:#2980b9; font-size:12px;">–</div></div>
    <div class="sim04_edt_statcard"><div class="sim04_edt_statlabel">Current b</div><div class="sim04_edt_statval" id="sim04_edt_sB" style="color:#b9770e;">–</div></div>
  </div>

  <!-- Canvas da Matriz 4x4 -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim04_edt_cvs"></canvas>
  </div>

  <!-- Barra de Ações -->
  <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:12px; align-items:center; justify-content:center;">
    <button id="sim04_edt_btnFinal" class="sim04_edt_act" data-action="final">Final Result</button>
    <button id="sim04_edt_btnStep" data-action="step">Step by Step</button>
    <div style="width:1px; background:#e4dcc8; height:20px;"></div>
    <button data-action="clear" style="border-color:#f5b7b1; color:#c0392b; background:#fdecea;">× Clear</button>
  </div>

  <!-- Painel de Passo a Passo (Controles de Animação) -->
  <div id="sim04_edt_stepCtrl" style="display:none;" class="sim04_edt_panel">
    <div style="display:flex; gap:6px; align-items:center; flex-wrap:wrap;">
      <button data-action="prev">◀ Previous</button>
      <button id="sim04_edt_btnPlay" data-action="play">▶ Play</button>
      <button data-action="next">Next ▶</button>
      <span class="sim04_edt_smtxt" style="margin-left:4px; font-weight:700;">Speed:</span>
      <input type="range" id="sim04_edt_speedSlider" min="1" max="10" value="5" step="1" style="width:70px; cursor:pointer;">
      <span id="sim04_edt_stepLabel" class="sim04_edt_mono" style="font-size:10.5px; color:#5e5a4a; flex:1; text-align:right; min-width:0; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;"></span>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim04EdtConvergencia(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const COLS=4, ROWS=4, CELL=50, PAD=20, INF=9999;
    const cv = root.querySelector('#sim04_edt_cvs');
    const CTX = cv.getContext('2d');

    cv.width = COLS * CELL + PAD * 2;
    cv.height = ROWS * CELL + PAD * 2;

    let grid = [
      [1, 1, 0, 1],
      [1, 1, 1, 1],
      [1, 0, 1, 1],
      [0, 1, 1, 1]
    ];

    let mode='final';
    let steps=[], stepIdx=0;
    let playing=false, playTimer=null;

    function getCol(d) {
      if(d === 0) return { fill: 'transparent', txt: '#8a8371', bdr: '#e4dcc8' };
      if(d >= INF) return { fill: '#fdecea', txt: '#c0392b', bdr: '#f5b7b1' };
      const colors = ['#ebf4fd', '#a9cce3', '#73c6b6', '#52be80', '#27ae60', '#2980b9', '#1b4f72'];
      let idx = Math.min(colors.length - 1, Math.floor(d / 1.5));
      return { fill: colors[idx], txt: idx > 3 ? '#ffffff' : '#26241d', bdr: colors[Math.min(idx+1, colors.length-1)] };
    }

    function drawMatrix(matrix, opts={}) {
      CTX.clearRect(0, 0, cv.width, cv.height);
      for(let r=0; r<ROWS; r++) {
        for(let c=0; c<COLS; c++) {
          const px = PAD + c * CELL; const py = PAD + r * CELL;
          const d = matrix[r][c]; const col = getCol(d);

          CTX.fillStyle = col.fill; CTX.fillRect(px+1, py+1, CELL-2, CELL-2);

          const isCur = opts.cursor && opts.cursor.r === r && opts.cursor.c === c;
          const isChg = opts.changed && opts.changed[r][c];

          CTX.strokeStyle = isCur ? '#c0392b' : isChg ? '#b9770e' : col.bdr;
          CTX.lineWidth = isCur ? 3 : isChg ? 2.5 : 0.5;
          CTX.strokeRect(px+0.5, py+0.5, CELL-1, CELL-1);

          CTX.fillStyle = col.txt; CTX.font = 'bold 14px monospace';
          CTX.textAlign = 'center'; CTX.textBaseline = 'middle';
          CTX.fillText(d >= INF ? '∞' : String(d), px + CELL/2, py + CELL/2);

          if (r === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('c'+c, px+CELL/2, PAD/2); }
          if (c === 0) { CTX.fillStyle='#8a8371'; CTX.font='9px monospace'; CTX.fillText('r'+r, PAD/2, py+CELL/2); }
        }
      }
    }

    function copyMat(m) { return m.map(row => [...row]); }

    function computeEdt1(gInit) {
      const all = [];
      let g = copyMat(gInit);
      all.push({ phase: 'edt1_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt1 · Matriz de entrada inicializada' });

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=1; r<ROWS; r++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r-1][c] < INF) {
            if(g[r][c] > g[r-1][c] + b) {
              g[r][c] = g[r-1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') = f(' + (r-1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 raster ↓ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let c=0; c<COLS; c++) {
        let b = 1;
        for(let r=ROWS-2; r>=0; r--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r+1][c] < INF) {
            if(g[r][c] > g[r+1][c] + b) {
              g[r][c] = g[r+1][c] + b; chg[r][c] = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') = f(' + (r+1) + ',' + c + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt1_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt1 anti-raster ↑ · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }
      return { steps: all, g: copyMat(g) };
    }

    function computeEdt2(gIn) {
      const all = [];
      let g = copyMat(gIn);
      all.push({ phase: 'edt2_init', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'edt2 · Estado inicial f_v antes da propagação horizontal' });

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=1; c<COLS; c++) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c-1] < INF) {
            if(g[r][c] > g[r][c-1] + b) {
              g[r][c] = g[r][c-1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 raster → · f(' + r + ',' + c + ') = f(' + r + ',' + (c-1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_raster', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 raster → · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      for(let r=0; r<ROWS; r++) {
        let b = 1;
        for(let c=COLS-2; c>=0; c--) {
          let chg = Array.from({length:ROWS}, () => new Array(COLS).fill(0));
          if(grid[r][c] && g[r][c+1] < INF) {
            if(g[r][c] > g[r][c+1] + b) {
              g[r][c] = g[r][c+1] + b; chg[r][c] = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: chg, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') = f(' + r + ',' + (c+1) + ') + b(' + b + ') = ' + g[r][c] });
              b += 2;
            } else {
              b = 1;
              all.push({ phase: 'edt2_anti', g: copyMat(g), cursor: {r, c}, b, changed: null, label: 'edt2 anti-raster ← · f(' + r + ',' + c + ') já menor · b=1' });
            }
          } else { b = 1; }
        }
      }

      all.push({ phase: 'edt2_done', g: copyMat(g), cursor: null, b: 0, changed: null, label: 'Convergência final atingida!' });
      return { steps: all, g: copyMat(g) };
    }

    function computeAll() {
      const gInit = Array.from({length: ROWS}, (_, r) => Array.from({length: COLS}, (_, c) => grid[r][c] ? INF : 0));
      const r1 = computeEdt1(gInit); const r2 = computeEdt2(r1.g);
      return [...r1.steps, ...r2.steps];
    }

    function setMode(m) {
      mode = m; stopPlay();
      root.querySelector('#sim04_edt_btnFinal').className = m === 'final' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_btnStep').className = m === 'step' ? 'sim04_edt_act' : '';
      root.querySelector('#sim04_edt_stepCtrl').style.display = m === 'step' ? 'block' : 'none';

      if(m === 'step') {
        steps = computeAll(); stepIdx = 0; drawFrame();
      } else {
        const finalSteps = computeAll();
        drawMatrix(finalSteps[finalSteps.length - 1].g);
        root.querySelector('#sim04_edt_sStep').textContent = 'Final';
        root.querySelector('#sim04_edt_sPhase').textContent = 'Concluído';
        root.querySelector('#sim04_edt_sB').textContent = '–';
      }
    }

    function drawFrame() {
      if(!steps.length) return;
      const s = steps[Math.min(stepIdx, steps.length - 1)];
      drawMatrix(s.g, { cursor: s.cursor, changed: s.changed });
      root.querySelector('#sim04_edt_sStep').textContent = (stepIdx + 1) + '/' + steps.length;
      root.querySelector('#sim04_edt_sPhase').textContent = s.phase;
      root.querySelector('#sim04_edt_sB').textContent = s.b > 0 ? s.b : '–';
      root.querySelector('#sim04_edt_stepLabel').textContent = s.label;
    }

    function goStep(d) {
      stepIdx = Math.max(0, Math.min(steps.length - 1, stepIdx + d));
      drawFrame();
    }

    function togglePlay() { playing ? stopPlay() : startPlay(); }
    function startPlay() {
      playing = true; root.querySelector('#sim04_edt_btnPlay').textContent = '⏸ Pausar';
      function tick() {
        if(stepIdx >= steps.length - 1) { stopPlay(); return; }
        stepIdx++; drawFrame();
        playTimer = setTimeout(tick, Math.round(1300 - (+root.querySelector('#sim04_edt_speedSlider').value) * 110));
      }
      tick();
    }
    function stopPlay() { playing = false; if(playTimer) clearTimeout(playTimer); const b = root.querySelector('#sim04_edt_btnPlay'); if(b) b.textContent = '▶ Play'; }

    function clearGrid() { grid = Array.from({length: ROWS}, () => new Array(COLS).fill(0)); refresh(); }
    function refresh() { stopPlay(); if(mode === 'step') { steps = computeAll(); stepIdx = Math.min(stepIdx, steps.length - 1); drawFrame(); } else { setMode('final'); } }

    cv.addEventListener('mousedown', e => {
      const rect = cv.getBoundingClientRect();
      const c = Math.floor(((e.clientX - rect.left) * (cv.width / rect.width) - PAD) / CELL);
      const r = Math.floor(((e.clientY - rect.top) * (cv.height / rect.height) - PAD) / CELL);
      if(r >= 0 && r < ROWS && c >= 0 && c < COLS) { grid[r][c] = grid[r][c] ? 0 : 1; refresh(); }
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'final') setMode('final');
        if(act === 'step') setMode('step');
        if(act === 'clear') clearGrid();
        if(act === 'prev') goStep(-1);
        if(act === 'next') goStep(1);
        if(act === 'play') togglePlay();
      });
    });

    setMode('final');
  }

  function tryInit(){
    var root = document.getElementById('sim-04-tde');
    if (root) initSim04EdtConvergencia(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
""")

**Figure 4.21:** Interactive 2D simulator (4x4) with strict synchronization of horizontal propagation queues (b) to obtain the exact convergence described in the paper.


<figure id="fig-04-sim-04-tde">
  <img src="imagens/fig-04-sim-04-tde.png" alt=" Interactive 2D simulator (4x4) with strict synchronization of horizontal propagation queues (b) to obtain the exact convergence described in the paper. " style="max-width:80%" />
  <figcaption><strong>Figure 4.21:</strong>  Interactive 2D simulator (4x4) with strict synchronization of horizontal propagation queues (b) to obtain the exact convergence described in the paper. </figcaption>
</figure>

#### 4.4.3.1 Geodesic Distance Transform

The geodesic distance transform associates each pixel with the smallest distance to a marker, under the constraint imposed by a mask. Thus, propagation occurs exclusively through allowed pixels, preserving the connectivity of the domain.

[Figure 4.22](#fig-04-gdist-menor-caminho) illustrates this process in a maze: (a) the mask `g`; (b) the geodesic distance `D1` calculated from the entrance; (c) the distance `D2` calculated from the exit; and (d) the minimum path obtained from these two transforms.

The optimal path is determined by the sum of the distances (`D1 + D2`). The pixels belonging to the minimal trajectory are those for which this sum assumes its smallest value, defining a connection between entrance and exit with minimum geodesic length.

This principle allows solving mazes without the need to explicitly explore all possible routes. The solution emerges directly from the propagation of distances in a restricted domain. This approach is particularly relevant in highly complex mazes, such as those constructed from quasicrystalline structures and Hamiltonian cycles described by Singh (2024). In Zampirolli (2025), this same formalism is employed to solve a complex maze; below, the method is illustrated in a simplified version of the problem.

In [51]:
%%writefile tmp/fig_04_gdist_menor_caminho.cpp
#define MM_OUT "tmp/fig_04_gdist_menor_caminho.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    //| label: fig-04-gdist-menor-caminho
    //| fig-cap: "Menor caminho geodésico em um labirinto. As distâncias geodésicas são calculadas a partir da entrada e da saída usando mm.gdist. Os pixels cujo somatório das duas distâncias é igual à distância mínima entre os marcadores pertencem a um caminho ótimo."
    //| echo: true
    //| output: true

    // 1 = corredor, 0 = parede
    mm::Image g(10, 10);
    int g_data[10][10] = {
        {0,1,0,0,0,0,0,0,0,0},
        {0,1,1,1,1,1,0,1,1,1},
        {0,0,0,0,0,1,0,1,0,1},
        {0,1,1,1,0,1,1,1,0,1},
        {0,1,0,1,0,0,0,0,0,1},
        {0,1,0,1,1,1,1,1,1,1},
        {0,1,0,0,0,0,0,0,1,0},
        {0,1,1,1,1,1,1,0,1,0},
        {0,0,0,0,0,0,1,1,1,0},
        {0,0,0,0,0,0,0,0,1,0}
    };
    for (int y = 0; y < 10; y++) {
        for (int x = 0; x < 10; x++) {
            g.at(y, x) = g_data[y][x];
        }
    }

    // marcador da entrada
    mm::Image entrada(10, 10);
    entrada.at(0, 1) = 1;

    // marcador da saída
    mm::Image saida(10, 10);
    saida.at(9, 8) = 1;

    // distâncias geodésicas
    mm::Image D1 = mm::gdist(g, entrada);
    mm::Image D2 = mm::gdist(g, saida);

    // soma das distâncias
    mm::Image S = mm::addm(D1, D2);

    // menor valor válido da soma
    int dmin = 255;
    for (int i = 0; i < S.h * S.w; i++) {
        if (S.data[i] > 0 && S.data[i] < dmin) {
            dmin = S.data[i];
        }
    }

    // pixels pertencentes a um caminho ótimo
    mm::Image caminho(10, 10);
    for (int y = 0; y < 10; y++) {
        for (int x = 0; x < 10; x++) {
            caminho.at(y, x) = (S.at(y, x) == dmin) ? 255 : 0;
        }
    }

    std::cout << "Distância geodésica mínima: " << dmin << std::endl;

    std::vector<std::string> titles = {
        "Labirinto",
        "Distância da Entrada",
        "Distância da Saída",
        "Menor Caminho\n(d=" + std::to_string(dmin) + ")"
    };
    mm::show(
        std::vector<mm::Image>{g, D1, D2, caminho},
        MM_OUT,
        titles,
        4
    );

    return 0;
}

Overwriting tmp/fig_04_gdist_menor_caminho.cpp


In [52]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_gdist_menor_caminho.cpp -o tmp/fig_04_gdist_menor_caminho -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_gdist_menor_caminho \
  && test -f "tmp/fig_04_gdist_menor_caminho.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_gdist_menor_caminho.png"

Distância geodésica mínima: 15
[1] Labirinto
[2] Distância da Entrada
[3] Distância da Saída
[4] Menor Caminho
(d=15)


In [53]:
mm.show(mm.read("tmp/fig_04_gdist_menor_caminho.png"), figsize=(14, 4))

<Figure size 2100x600 with 1 Axes>

**Figure 4.22:** Menor caminho geodésico em um labirinto. As distâncias geodésicas são calculadas a partir da entrada e da saída usando mm.gdist. Os pixels cujo somatório das duas distâncias é igual à distância mínima entre os marcadores pertencem a um caminho ótimo.


### 4.4.4 Segmentation by *Watershed*

The ***Watershed*** algorithm interprets a grayscale image as a topographic surface, where high values correspond to mountains and low values correspond to valleys or catchment basins. In the context of marker-based segmentation, the maxima of the Distance Transform are often used to identify internal regions of objects, providing reliable seeds for the flooding process.

Segmentation is then performed through a conceptual simulation of progressive flooding from these markers. As the basins associated with different seeds expand, neighboring regions eventually come into contact. At that point, virtual barriers, known as *watershed lines*, are constructed, which then delimit the objects in the scene. This mechanism allows for the separation of adjacent or partially overlapping objects, even when they form a single connected component after thresholding.

The didactic implementation presented in this chapter initially explores the concept of region growing confined by a binary mask, as detailed in the interactive algorithm of [Figure 4.23](#fig-04-sim-alg-watershed2).

> ### 📝 Didactic version versus classical implementation
>
> The `mm::watershed0` function does not implement the classical *watershed* algorithm. Its purpose is to illustrate, in a simplified manner, the propagation of markers through region growing, allowing one to visualize how different seeds compete for the occupation of available space. The growth is delimited by a binary support mask and monitored by a stagnation control, producing a result similar to a Voronoi partition restricted to the geometry of the input objects.
>
> The `mm::watershed` function, in turn, uses the optimized OpenCV implementation (`mm::watershed`), which performs flooding over a topographic surface defined by the input image. In this case, the propagation of markers is influenced by pixel values, causing separation lines to form naturally over the ridges of the relief.

In [54]:
# @title { display-mode: "form" }
# Exclusive prefix to avoid conflict with other notebook cells
PREFIX = "wat0"

from IPython.display import HTML
HTML(f'''
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:var(--font-sans)}}
.{PREFIX}-tabs{{display:flex;gap:6px;margin-bottom:20px;flex-wrap:wrap}}
.{PREFIX}-tab{{padding:6px 14px;border-radius:20px;font-size:13px;cursor:pointer;border:0.5px solid var(--color-border-secondary);background:var(--color-background-primary);color:var(--color-text-secondary);transition:all .15s;white-space:nowrap}}
.{PREFIX}-tab.active{{background:var(--color-text-primary);color:var(--color-background-primary);border-color:transparent}}
.{PREFIX}-panel{{display:none}}.{PREFIX}-panel.active{{display:block}}

.algo-wrap{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.algo-header{{padding:14px 20px;border-bottom:0.5px solid var(--color-border-tertiary);display:flex;align-items:center;gap:10px}}
.algo-icon{{width:32px;height:32px;border-radius:var(--border-radius-md);display:flex;align-items:center;justify-content:center;font-size:16px;background:#EBF7F2;color:#0F6E56}}
.algo-title{{font-size:14px;font-weight:500;color:var(--color-text-primary);text-align:left}}
.algo-sub{{font-size:12px;color:var(--color-text-secondary);margin-top:1px;text-align:left}}
.algo-body{{padding:20px;text-align:left}}

.step{{display:flex;gap:12px;margin-bottom:14px;align-items:flex-start}}
.step:last-child{{margin-bottom:0}}
.step-num{{min-width:24px;height:24px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0;margin-top:1px}}
.step-text{{font-size:13.5px;line-height:1.65;color:var(--color-text-primary);text-align:left}}
.step-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 6px;border-radius:4px;color:var(--color-text-primary)}}
.step-sub{{margin-top:8px;border-left:2px solid var(--color-border-secondary);padding-left:12px;display:flex;flex-direction:column;gap:5px}}
.step-sub-item{{font-size:13px;color:var(--color-text-secondary);line-height:1.55;text-align:left}}
.step-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.while-box{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:11px 13px;margin-top:8px;text-align:left}}
.while-head{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.06em;margin-bottom:7px}}
.note{{margin-top:16px;padding-top:12px;border-top:0.5px solid var(--color-border-tertiary);font-size:12px;color:var(--color-text-secondary);font-style:italic;line-height:1.5;text-align:left}}

.num-teal{{background:#E1F5EE;color:#0F6E56}}
.num-purple{{background:#EEEDFE;color:#534AB7}}
.num-amber{{background:#FAEEDA;color:#854F0B}}
.num-coral{{background:#FAECE7;color:#993C1D}}
.num-blue{{background:#E6F1FB;color:#185FA5}}
.num-gray{{background:var(--color-background-secondary);color:var(--color-text-secondary)}}

.card-step{{display:flex;border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);overflow:hidden;margin-bottom:8px}}
.card-step:last-child{{margin-bottom:0}}
.card-badge{{min-width:48px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:500;flex-shrink:0}}
.card-content{{padding:10px 14px;flex:1;text-align:left}}
.card-label{{font-size:10px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;margin-bottom:3px}}
.card-text{{font-size:13.5px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.card-text code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}
.card-sub{{margin-top:6px;display:flex;flex-direction:column;gap:3px}}
.card-sub-item{{font-size:12.5px;color:var(--color-text-secondary);padding-left:10px;border-left:2px solid var(--color-border-secondary);line-height:1.5;text-align:left}}
.card-sub-item code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 4px;border-radius:3px;color:var(--color-text-primary)}}

.tl{{position:relative;padding:4px 0 4px 36px;text-align:left}}
.tl-line{{position:absolute;left:11px;top:20px;bottom:20px;width:1.5px;background:var(--color-border-secondary);border-radius:2px}}
.tl-step{{display:flex;gap:0;margin-bottom:18px;position:relative}}
.tl-step:last-child{{margin-bottom:0}}
.tl-dot-wrap{{position:absolute;left:-36px;top:2px;display:flex;flex-direction:column;align-items:center;gap:3px}}
.tl-dot{{width:14px;height:14px;border-radius:50%;border:2px solid var(--color-border-secondary);background:var(--color-background-primary);transition:all .2s;z-index:1}}
.tl-step:hover .tl-dot{{background:var(--color-text-primary);border-color:var(--color-text-primary)}}
.tl-num{{font-size:9px;color:var(--color-text-secondary);font-weight:500;letter-spacing:.04em}}
.tl-title{{font-size:13.5px;font-weight:500;color:var(--color-text-primary);margin-bottom:3px;text-align:left}}
.tl-desc{{font-size:13px;color:var(--color-text-secondary);line-height:1.6;text-align:left}}
.tl-desc code{{font-family:var(--font-mono);font-size:12px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.code-wrap{{background:var(--color-background-secondary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-lg);overflow:hidden}}
.code-bar{{background:var(--color-background-secondary);padding:10px 16px;display:flex;align-items:center;justify-content:space-between;border-bottom:0.5px solid var(--color-border-tertiary)}}
.code-lang{{font-size:11px;font-weight:500;color:var(--color-text-secondary);text-transform:uppercase;letter-spacing:.08em}}
.code-body{{padding:18px 20px;overflow-x:auto;text-align:left}}
.code-body pre{{font-family:var(--font-mono);font-size:13px;line-height:1.75;color:var(--color-text-primary);margin:0;white-space:pre;text-align:left}}
.kw{{color:#7C3AED}} .fn{{color:#0369A1}} .cm{{color:#6B7280;font-style:italic}} .st{{color:#059669}} .num-lit{{color:#DC2626}}

.ann-line{{display:flex;align-items:flex-start;gap:10px;margin-bottom:8px;padding:10px 12px;background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);text-align:left}}
.ann-badge{{min-width:20px;height:20px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:500;flex-shrink:0;margin-top:1px}}
.ann-text{{font-size:13px;line-height:1.6;color:var(--color-text-primary);text-align:left}}
.ann-text code{{font-family:var(--font-mono);font-size:11.5px;background:var(--color-background-secondary);padding:1px 5px;border-radius:3px;color:var(--color-text-primary)}}

.metric-grid{{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;margin-top:16px}}
.metric-card{{background:var(--color-background-primary);border:0.5px solid var(--color-border-tertiary);border-radius:var(--border-radius-md);padding:12px 14px;text-align:left}}
.metric-title{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.07em;margin-bottom:6px}}
.metric-formula{{font-family:var(--font-mono);font-size:12px;color:var(--color-text-primary);margin-bottom:4px}}
.metric-desc{{font-size:12px;color:var(--color-text-secondary);line-height:1.5}}
</style>

<h2 class="sr-only" style="position:absolute;left:-9999px">Algoritmo Didático do Watershed Limitado por Máscara — painel interativo</h2>

<div class="{PREFIX}-tabs" id="{PREFIX}-tabs-container">
  <div class="{PREFIX}-tab" data-idx="0"><i class="ti ti-list-numbers" aria-hidden="true"></i> Passo a passo</div>
  <div class="{PREFIX}-tab" data-idx="1"><i class="ti ti-cards" aria-hidden="true"></i> Cards</div>
  <div class="{PREFIX}-tab" data-idx="2"><i class="ti ti-timeline" aria-hidden="true"></i> Linha do tempo</div>
  <div class="{PREFIX}-tab" data-idx="3"><i class="ti ti-code" aria-hidden="true"></i> Código Python</div>
  <div class="{PREFIX}-tab active" data-idx="4"><i class="ti ti-git-branch" aria-hidden="true"></i> Fluxograma</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-0">
<div class="algo-wrap">
  <div class="algo-header">
    <div class="algo-icon"><i class="ti ti-bucket" aria-hidden="true"></i></div>
    <div><div class="algo-title">Crescimento de Regiões Confinado por Máscara</div><div class="algo-sub">Inundação concorrente com restrição geométrica de suporte e sincronização síncrona por malha</div></div>
  </div>
  <div class="algo-body">
    <div class="step"><div class="step-num num-teal">1</div><div class="step-text">Rotular os marcadores sementes em <code>f</code> via <code>mm.label0(f, b)</code>, instanciar a malha dinâmica <code>g ← f.copy()</code> e binarizar a <code>mask</code>.</div></div>
    <div class="step">
      <div class="step-num num-coral">2</div>
      <div class="step-text">
        Enquanto houver pixels não rotulados (<code>while True</code>), reiniciar o controle de atividade <code>mudou ← False</code> e varrer a imagem:
        <div class="step-sub">
          <div class="step-sub-item">Identificar se a coordenada atual é um vazio contido no escopo: <code>g[x,y] == 0 and mask[x,y]</code>.</div>
          <div class="step-sub-item">Avaliar a vizinhança estrutural em <code>mm._viz(f, b, x, y)</code> baseada no estado síncrono estável <code>f</code>.</div>
          <div class="step-sub-item">Se um vizinho possuir rótulo dominante (<code>g[x,y] < f[vy,vx]</code>), a célula em <code>g</code> absorve esse identificador e marca-se <code>mudou ← True</code>.</div>
        </div>
      </div>
    </div>
    <div class="step"><div class="step-num num-purple">3</div><div class="step-text">Verificar ponto fixo: caso uma varredura completa não expanda nenhuma fronteira (<code>not mudou</code>), interrompe-se o laço (<code>break</code>).</div></div>
    <div class="step"><div class="step-num num-blue">4</div><div class="step-text">Atualizar o estado de referência de forma síncrona para a próxima iteração: <code>f ← g.copy()</code>.</div></div>
    <div class="step"><div class="step-num num-blue">5</div><div class="step-text">Se <code>op == 'region'</code>, retornar o mapa de bacias <code>g</code>; caso contrário, extrair as cristas divisórias via <code>mm.gradm(g)</code>.</div></div>
    <div class="note">A sincronização <code>f = g.copy()</code> ao final de cada ciclo impede o crescimento assimétrico ou dependente da ordem da varredura raster (propagação em estilo Jacobi).</div>
  </div>
</div>

<div class="metric-grid">
  <div class="metric-card">
    <div class="metric-title" style="color:#0F6E56">Escopo Geométrico</div>
    <div class="metric-formula">mask[x,y] > 0</div>
    <div class="metric-desc">Restrição binária rígida impedindo o avanço periférico de rótulos.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#993C1D">Estabilização</div>
    <div class="metric-formula">if not mudou: break</div>
    <div class="metric-desc">Evita loops infinitos interrompendo ao saturar o domínio da máscara.</div>
  </div>
  <div class="metric-card">
    <div class="metric-title" style="color:#185FA5">Mapeamento Jacobi</div>
    <div class="metric-formula">f = g.copy()</div>
    <div class="metric-desc">Sincronização em bloco após inspeção de todas as coordenadas.</div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-1">
  <div class="card-step">
    <div class="card-badge num-teal">01</div>
    <div class="card-content">
      <div class="card-label" style="color:#0F6E56">Inicialização</div>
      <div class="card-text">Geração dos identificadores iniciais pelo mapeamento de componentes conexas e binarização da máscara de suporte.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-coral">02</div>
    <div class="card-content">
      <div class="card-label" style="color:#993C1D">Expansão Concorrente</div>
      <div class="card-text">Varredura 2D inspecionando vazios internos autorizados. A malha de trabalho <code>g</code> absorve os rótulos lidos da referência estável <code>f</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-purple">03</div>
    <div class="card-content">
      <div class="card-label" style="color:#534AB7">Ponto Fixo Local</div>
      <div class="card-text">A flag <code>mudou</code> monitora mudanças estruturais. Se nenhuma frente avançar, o laço de inundação é finalizado via <code>break</code>.</div>
    </div>
  </div>
  <div class="card-step">
    <div class="card-badge num-blue">04</div>
    <div class="card-content">
      <div class="card-label" style="color:#185FA5">Sincronização e Saída</div>
      <div class="card-text">Atualização em bloco do estado referencial. A saída pode ser moldada como partições regionais ou linhas de cristas (linhas de watershed).</div>
    </div>
  </div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-2">
<div class="tl">
  <div class="tl-line"></div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">01</span></div>
    <div><div class="tl-title">Condicionamento Prévio</div><div class="tl-desc">Rotulagem preliminar de marcadores e isolamento booleano do domínio.</div></div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">02</span></div>
    <div>
      <div class="tl-title">Laço Síncrono Iterativo</div>
      <div class="while-box">
        <div class="while-head">while True</div>
        <div class="step-sub" style="border-color:var(--color-border-tertiary)">
          <div class="step-sub-item">Redefinição de flag: <code>mudou = False</code></div>
          <div class="step-sub-item">Crescimento condicional: se <code>g[x,y] == 0</code> e estiver na máscara, expande lendo <code>f</code></div>
          <div class="step-sub-item">Controle de estabilidade: <code>if not mudou: break</code></div>
          <div class="step-sub-item">Atualização síncrona: <code>f = g.copy()</code></div>
        </div>
      </div>
    </div>
  </div>
  <div class="tl-step">
    <div class="tl-dot-wrap"><div class="tl-dot"></div><span class="tl-num">03</span></div>
    <div><div class="tl-title">Extração Topológica</div><div class="tl-desc">Retorno condicional das bacias preenchidas ou cálculo morfológico do gradiente de transição.</div></div>
  </div>
</div>
</div>

<div class="{PREFIX}-panel" id="{PREFIX}-p-3">
<div class="code-wrap">
  <div class="code-bar">
    <span class="code-lang"><i class="ti ti-brand-python" aria-hidden="true" style="font-size:14px;vertical-align:-2px;margin-right:5px"></i>mm_watershed.py</span>
    <span style="font-size:11px;color:var(--color-text-secondary)">Algoritmo com Restrição de Máscara</span>
  </div>
  <div class="code-body">
<pre><span class="kw">def</span> <span class="fn">watershed0</span>(f, mask=<span class="fn">None</span>, b=np.zeros((<span class="num-lit">3</span>,<span class="num-lit">3</span>),dtype=<span class="st">'uint8'</span>), op=<span class="st">'region'</span>):
    f = mm.label0(f, b)
    g = f.copy()
    mask = np.ones_like(f) <span class="kw">if</span> mask <span class="kw">is</span> <span class="fn">None</span> <span class="kw">else</span> (mask > <span class="num-lit">0</span>)
    
    <span class="kw">while</span> <span class="fn">True</span>:
        mudou = <span class="fn">False</span>
        <span class="kw">for</span> x <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">0</span>]):
            <span class="kw">for</span> y <span class="kw">in</span> <span class="fn">range</span>(f.shape[<span class="num-lit">1</span>]):
                <span class="kw">if</span> g[x,y] == <span class="num-lit">0</span> <span class="kw">and</span> mask[x,y]:
                    <span class="kw">for</span> vy,vx,bv <span class="kw">in</span> mm._viz(f, b, x, y):
                        <span class="kw">if</span> bv <span class="kw">and g[x,y] < f[vy,vx]</span>: 
                            g[x,y] = f[vy,vx]
                            mudou = <span class="fn">True</span>
        <span class="kw">if</span> <span class="kw">not</span> mudou: 
            <span class="kw">break</span>
        f = g.copy()
        
    <span class="kw">return</span> g <span class="kw">if</span> op == <span class="st">'region'</span> <span class="kw">else</span> mm.gradm(g, mm.secross())</pre>
  </div>
</div>

<div style="margin-top:16px;display:flex;flex-direction:column;gap:8px;text-align:left">
  <div class="ann-line"><div class="ann-badge num-teal">1</div><div class="ann-text"><code>mask = (mask &gt; 0)</code> — Converte a imagem de suporte informada para um mapa Booleano indexável.</div></div>
  <div class="ann-line"><div class="ann-badge num-coral">2</div><div class="ann-text"><code>g[x,y] == 0 and mask[x,y]</code> — Filtro ativo: pixels fora da máscara (fundo zero) são ignorados de imediato, confinando as frentes de expansão.</div></div>
  <div class="ann-line"><div class="ann-badge num-purple">3</div><div class="ann-text"><code>if not mudou: break</code> — Mecanismo de escape. Quando todos os espaços internos permitidos forem preenchidos ou estabilizados contra a barreira, o laço aborta de forma limpa.</div></div>
</div>
</div>

<div class="{PREFIX}-panel active" id="{PREFIX}-p-4">
<svg viewBox="0 0 580 840" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:580px;display:block;margin:0 auto">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#888780"/></marker>
    <marker id="arr-b" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#185FA5"/></marker>
    <marker id="arr-r" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#993C1D"/></marker>
    <marker id="arr-g" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#0F6E56"/></marker>
    <marker id="arr-p" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M1,1 L1,5 L7,3 Z" fill="#534AB7"/></marker>
  </defs>

  <ellipse cx="280" cy="36" rx="60" ry="22" fill="#E1F5EE" stroke="#0F6E56" stroke-width="1"/>
  <text x="280" y="41" text-anchor="middle" font-size="13" font-weight="500" fill="#085041">início</text>

  <rect x="130" y="85" width="300" height="52" rx="6" fill="#E1F5EE" stroke="#9FE1CB" stroke-width="1"/>
  <text x="280" y="104" text-anchor="middle" font-size="12" font-weight="500" fill="#0F6E56">rotular sementes e normalizar máscara</text>
  <text x="280" y="121" text-anchor="middle" font-size="11" fill="#085041">f ← mm.label0(f, b) ; g ← f.copy() ; mask ← mask > 0</text>

  <polygon points="280,166 390,196 280,226 170,196" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="201" text-anchor="middle" font-size="12" fill="#0C447C">True?</text>

  <rect x="150" y="256" width="260" height="40" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="281" text-anchor="middle" font-size="12" fill="#712B13">reiniciar ciclo: mudou ← False</text>

  <rect x="120" y="326" width="320" height="72" rx="6" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="345" text-anchor="middle" font-size="12" font-weight="500" fill="#993C1D">varredura espacial e expansão</text>
  <text x="280" y="364" text-anchor="middle" font-size="11.5" fill="#712B13">se g[x,y] == 0 e mask[x,y] e f[vy,vx] > g[x,y]:</text>
  <text x="280" y="382" text-anchor="middle" font-size="11.5" font-weight="500" fill="#712B13">g[x,y] ← f[vy,vx] ; mudou ← True</text>

  <polygon points="280,430 380,460 280,490 180,460" fill="#FAECE7" stroke="#F0997B" stroke-width="1"/>
  <text x="280" y="465" text-anchor="middle" font-size="11.5" fill="#712B13">not mudou?</text>

  <rect x="150" y="520" width="260" height="40" rx="6" fill="#E6F1FB" stroke="#85B7EB" stroke-width="1"/>
  <text x="280" y="545" text-anchor="middle" font-size="12" fill="#0C447C">sincronizar malha de ref: f ← g.copy()</text>

  <polygon points="280,600 390,630 280,660 170,630" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="280" y="635" text-anchor="middle" font-size="12" fill="#3C3489">op == 'region'?</text>

  <rect x="70" y="695" width="170" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="155" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar g ( bacias )</text>

  <rect x="320" y="695" width="200" height="40" rx="6" fill="#EEEDFE" stroke="#AFA9EC" stroke-width="1"/>
  <text x="420" y="719" text-anchor="middle" font-size="11.5" fill="#3C3489">retornar mm.gradm(g)</text>

  <ellipse cx="280" cy="790" rx="55" ry="22" fill="#EAF3DE" stroke="#97C459" stroke-width="1"/>
  <text x="280" y="795" text-anchor="middle" font-size="13" font-weight="500" fill="#27500A">fim</text>

  <line x1="280" y1="58" x2="280" y2="85" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  <line x1="280" y1="137" x2="280" y2="166" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
  
  <line x1="280" y1="226" x2="280" y2="256" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>
  <text x="295" y="240" font-size="11" fill="#185FA5">sim</text>

  <line x1="280" y1="296" x2="280" y2="326" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <line x1="280" y1="398" x2="280" y2="430" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  
  <line x1="280" y1="490" x2="280" y2="520" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="295" y="504" font-size="11" fill="#993C1D">não</text>

  <path d="M 150,540 L 45,540 L 45,196 L 170,196" fill="none" stroke="#185FA5" stroke-width="1.2" marker-end="url(#arr-b)"/>

  <path d="M 380,460 L 460,460 L 460,580 L 280,580 L 280,600" fill="none" stroke="#993C1D" stroke-width="1.2" marker-end="url(#arr-r)"/>
  <text x="420" y="450" text-anchor="middle" font-size="11" fill="#993C1D">sim (break)</text>

  <path d="M 390,196 L 500,196 L 500,580 L 280,580" fill="none" stroke="#185FA5" stroke-width="1.2"/>
  <text x="445" y="186" font-size="11" fill="#185FA5">não</text>

  <line x1="170" y1="630" x2="155" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="155" y1="630" x2="155" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="140" y="620" font-size="11" fill="#534AB7">sim</text>

  <line x1="390" y1="630" x2="420" y2="630" stroke="#534AB7" stroke-width="1.2"/>
  <line x1="420" y1="630" x2="420" y2="695" stroke="#534AB7" stroke-width="1.2" marker-end="url(#arr-p)"/>
  <text x="402" y="620" font-size="11" fill="#534AB7">não</text>

  <line x1="155" y1="735" x2="155" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="155" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="735" x2="420" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="420" y1="755" x2="280" y2="755" stroke="#888780" stroke-width="1.2"/>
  <line x1="280" y1="755" x2="280" y2="768" stroke="#888780" stroke-width="1.2" marker-end="url(#arr)"/>
</svg>
</div>

<script>
(function() {{
  var container = document.getElementById('{PREFIX}-tabs-container');
  if (!container) return;
  
  var tabs = container.querySelectorAll('.{PREFIX}-tab');
  var panels = [
    document.getElementById('{PREFIX}-p-0'),
    document.getElementById('{PREFIX}-p-1'),
    document.getElementById('{PREFIX}-p-2'),
    document.getElementById('{PREFIX}-p-3'),
    document.getElementById('{PREFIX}-p-4')
  ];

  tabs.forEach(function(tab) {{
    tab.onclick = function() {{
      var idx = parseInt(this.getAttribute('data-idx'));
      
      tabs.forEach(function(t) {{ t.classList.remove('active'); }});
      panels.forEach(function(p) {{ if(p) p.classList.remove('active'); }});
      
      this.classList.add('active');
      if(panels[idx]) panels[idx].classList.add('active');
    }};
  }});
}})();
</script>
''')

**Figure 4.23:** Didactic *Watershed* Algorithm by Region Growing Limited by Mask.


<figure id="fig-04-sim-alg-watershed2">
  <img src="imagens/fig-04-sim-alg-watershed2.png" alt=" Didactic *Watershed* Algorithm by Region Growing Limited by Mask. " style="max-width:80%" />
  <figcaption><strong>Figure 4.23:</strong>  Didactic *Watershed* Algorithm by Region Growing Limited by Mask. </figcaption>
</figure>

[Figure 4.24](#fig-04-sim-04-watershed) presents an iterative simulator that illustrates the propagation of markers across the region of interest. Each marker acts as a flooding source that expands its area of influence until it encounters regions originating from other seeds. In the OpenCV algorithm, pixels belonging to the dividing lines are identified by the value `-1`, representing the boundaries between adjacent watersheds.

In [55]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-04-watershed" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-04-watershed * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-04-watershed canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; cursor: crosshair; background: #ffffff; }
  #sim-04-watershed button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; display: inline-flex; align-items: center; justify-content: center; gap: 5px; transition: all 0.15s ease; font-weight: 600; }
  #sim-04-watershed button:hover { background: #e8dfcf; }
  #sim-04-watershed button.ws_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-04-watershed .ws_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .ws_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .ws_grid_stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin-bottom: 14px; }
  .ws_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .ws_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .ws_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  
  .ws_se_grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 4px; max-width: 110px; margin: 6px auto; }
  .ws_se_btn { padding: 6px!important; font-size: 10px!important; font-family: monospace; font-weight: 700; }
  .ws_se_btn.ws_off { color: #8a8371!important; background: #fafaf7!important; border-color: #e4dcc8!important; font-weight: normal; }
  
  .ws_tool_grid { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 6px; }
  .ws_color_dot { width: 10px; height: 10px; border-radius: 2px; display: inline-block; border: 1px solid rgba(0,0,0,0.2); }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">💧 Simulator: Watershed Segmentation</span>
  <span class="ws_pill">Propagation with Structuring Element (b)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Estatísticas -->
  <div class="ws_grid_stats">
    <div class="ws_stat_box">
      <div class="ws_stat_label">Area (Mask)</div>
      <div id="ws_statPx" class="ws_stat_value" style="color:#8a8371;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Markers</div>
      <div id="ws_statSeeds" class="ws_stat_value" style="color:#27ae60;">0</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Flooding</div>
      <div id="ws_statFill" class="ws_stat_value" style="color:#2980b9;">0%</div>
    </div>
    <div class="ws_stat_box">
      <div class="ws_stat_label">Iteration (k)</div>
      <div id="ws_statStep" class="ws_stat_value" style="color:#c0392b;">–</div>
    </div>
  </div>

  <!-- Layout Principal em 2 Colunas -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; align-items:flex-start;">
    
    <!-- Canvas -->
    <div style="flex:2; min-width:260px; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px; text-align:center;">
      <canvas id="ws_Canvas" style="margin:0 auto;"></canvas>
      <div style="font-size:10px; color:#8a8371; margin-top:8px;">
        🖱️ Drag to draw/erase the mask or seeds
      </div>
    </div>

    <!-- Painel Lateral de Ferramentas -->
    <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
      
      <!-- Ferramentas de Desenho -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Tools
        </div>
        <div class="ws_tool_grid">
          <button id="ws_tool_mask" class="ws_active" data-tool="mask">
            <span class="ws_color_dot" style="background:#e4dcc8;"></span> Mask
          </button>
          <button id="ws_tool_erase" data-tool="erase">
            <span class="ws_color_dot" style="background:#fafaf7; border-style:dashed;"></span> Erase
          </button>
          <button id="ws_tool_s2" data-tool="s2">
            <span class="ws_color_dot" style="background:#2980b9;"></span> Seed 1
          </button>
          <button id="sim04_ws_tool_s3" data-tool="s3">
            <span class="ws_color_dot" style="background:#27ae60;"></span> Seed 2
          </button>
          <button id="sim04_ws_tool_s4" data-tool="s4" style="grid-column: span 2;">
            <span class="ws_color_dot" style="background:#b9770e;"></span> Seed 3
          </button>
        </div>
      </div>

      <!-- Elemento Estruturante (b) -->
      <div class="ws_panel" style="text-align:center;">
        <div style="font-size:11px; font-weight:700; margin-bottom:2px; text-align:left; color:#5e5a4a;">
          Structuring Element (b)
        </div>
        <div class="ws_se_grid">
          <button id="ws_se_0_0" class="ws_se_btn ws_off" data-se="0,0">0</button>
          <button id="ws_se_0_1" class="ws_se_btn ws_active" data-se="0,1">1</button>
          <button id="sim04_ws_se_0_2" class="ws_se_btn ws_off" data-se="0,2">0</button>
          
          <button id="sim04_ws_se_1_0" class="ws_se_btn ws_active" data-se="1,0">1</button>
          <button class="ws_se_btn" style="background:#e4dcc8; cursor:not-allowed; color:#8a8371;" disabled>1</button>
          <button id="sim04_ws_se_1_2" class="ws_se_btn ws_active" data-se="1,2">1</button>
          
          <button id="sim04_ws_se_2_0" class="ws_se_btn ws_off" data-se="2,0">0</button>
          <button id="sim04_ws_se_2_1" class="ws_se_btn ws_active" data-se="2,1">1</button>
          <button id="sim04_ws_se_2_2" class="ws_se_btn ws_off" data-se="2,2">0</button>
        </div>
        <div style="display:flex; gap:4px; margin-top:8px;">
          <button data-se-preset="cross" style="flex:1; font-size:9.5px;">Cross (C-4)</button>
          <button data-se-preset="square" style="flex:1; font-size:9.5px;">Square (C-8)</button>
        </div>
      </div>

      <!-- Modo de Visualização -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Visualization
        </div>
        <div style="display:flex; gap:6px;">
          <button id="ws_btnLabel" class="ws_active" data-mode="label" style="flex:1; justify-content:center;">Final</button>
          <button id="ws_btnAnim" data-mode="anim" style="flex:1; justify-content:center;">Animated</button>
        </div>
      </div>

      <!-- Controles de Animação -->
      <div id="ws_animCtrl" class="ws_panel" style="display:none;">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Step-by-Step Flooding
        </div>
        <div style="display:flex; gap:6px; margin-bottom:8px;">
          <button data-action="prev" style="flex:1; justify-content:center;">◀</button>
          <button id="ws_btnPlay" data-action="play" style="flex:1; justify-content:center;">▶</button>
          <button data-action="next" style="flex:1; justify-content:center;">▶▶</button>
        </div>
        <div style="display:flex; align-items:center; gap:8px;">
          <label style="font-size:9.5px; color:#8a8371; font-weight:700;">Speed</label>
          <input type="range" id="ws_speedSlider" min="1" max="10" value="6" style="flex:1; max-width:160px; height:4px; cursor:pointer;">
        </div>
      </div>

      <!-- Exemplos Iniciais -->
      <div class="ws_panel">
        <div style="font-size:11px; font-weight:700; margin-bottom:6px; color:#5e5a4a;">
          Initial Examples
        </div>
        <div style="display:flex; flex-direction:column; gap:5px;">
          <button data-preset="moedas" style="justify-content:flex-start;">🪙 Touching Coins</button>
          <button data-preset="celulas" style="justify-content:flex-start;">🦠 Cluster of 3</button>
        </div>
      </div>

      <!-- Botão de Limpeza -->
      <button data-action="clear" style="justify-content:center; border-color:#f5b7b1; color:#c0392b; background:#fdecea;">
        🗑️ Clear All
      </button>

    </div>

  </div>

</div>
</div>

<script>
(function(){
  function initSim04Watershed(root){
    if (!root || root.dataset.initDone) return;
    root.dataset.initDone = "1";

    const ws_COLS = 16, ws_ROWS = 14, ws_CELL = 20, ws_PAD = 10;
    const ws_W = ws_COLS * ws_CELL + ws_PAD * 2, ws_H = ws_ROWS * ws_CELL + ws_PAD * 2;
    const ws_cv = root.querySelector('#ws_Canvas');
    ws_cv.width = ws_W; 
    ws_cv.height = ws_H;
    const ws_ctx = ws_cv.getContext('2d');

    const ws_COLORS = {
      0:  { fill: 'transparent', text: '#8a8371', stroke: '#e4dcc8' },
      1:  { fill: '#fafaf7',     text: '#8a8371', stroke: '#e4dcc8' },
      2:  { fill: '#ebf4fd',     text: '#2980b9', stroke: '#a9cce3' },
      3:  { fill: '#eafaf1',     text: '#27ae60', stroke: '#a3e4d7' },
      4:  { fill: '#fef5e7',     text: '#b9770e', stroke: '#f8c471' }
    };

    let ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    let ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
    
    let ws_b = [
      [false, true,  false],
      [true,  true,  true],
      [false, true,  false]
    ];
    
    let ws_currentTool = 'mask'; 
    let ws_mode = 'label';
    let ws_painting = false;

    let ws_animSteps = [];
    let ws_animIdx = 0;
    let ws_playing = false;
    let ws_playTimer = null;

    function ws_compute() {
      let steps = [];
      
      let current = Array.from({length:ws_ROWS}, (_, r) => 
        Array.from({length:ws_COLS}, (_, c) => {
          if (ws_markers[r][c] > 1) return ws_markers[r][c];
          if (ws_grid[r][c] === 1) return 1;
          return 0;
        })
      );
      
      const copy = arr => arr.map(row => [...row]);
      steps.push({ g: copy(current), k: 0 });

      let k = 0;
      while (true) {
        k++;
        let nextG = copy(current);
        let changed = false;

        for (let r = 0; r < ws_ROWS; r++) {
          for (let c = 0; c < ws_COLS; c++) {
            if (current[r][c] === 1) {
              let labels = new Set();
              
              for (let dr = -1; dr <= 1; dr++) {
                for (let dc = -1; dc <= 1; dc++) {
                  if (dr === 0 && dc === 0) continue;
                  if (ws_b[dr+1][dc+1]) { 
                    let nr = r + dr, nc = c + dc;
                    if (nr >= 0 && nr < ws_ROWS && nc >= 0 && nc < ws_COLS) {
                      let val = current[nr][nc];
                      if (val > 1) labels.add(val);
                    }
                  }
                }
              }

              if (labels.size === 1) {
                nextG[r][c] = [...labels][0];
                changed = true;
              } else if (labels.size > 1) {
                let labelsArr = [...labels];
                nextG[r][c] = labelsArr[Math.floor(Math.random() * labelsArr.length)];
                changed = true;
              }
            }
          }
        }

        if (!changed) {
          steps.push({ g: copy(nextG), k: k, done: true });
          break;
        }
        steps.push({ g: copy(nextG), k: k });
        current = nextG;
      }
      
      return { final: current, steps };
    }

    function ws_drawState(matrix, stepInfo) {
      ws_ctx.clearRect(0, 0, ws_W, ws_H);
      let area = 0, filled = 0, numSeeds = new Set();

      for (let r=0; r<ws_ROWS; r++) {
        for (let c=0; c<ws_COLS; c++) {
          const x = ws_PAD + c * ws_CELL, y = ws_PAD + r * ws_CELL;
          const val = matrix[r][c];
          
          if (ws_grid[r][c] === 1) area++;
          if (ws_markers[r][c] > 1) numSeeds.add(ws_markers[r][c]);
          if (val > 1) filled++;

          const col = ws_COLORS[val] || ws_COLORS[0];
          ws_ctx.fillStyle = col.fill; 
          ws_ctx.fillRect(x+1, y+1, ws_CELL-2, ws_CELL-2);
          
          if (val > 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
            ws_ctx.fillStyle = col.text;
            ws_ctx.font = 'bold 10px monospace';
            ws_ctx.textAlign = 'center'; ws_ctx.textBaseline = 'middle';
            ws_ctx.fillText('S' + (val-1), x+ws_CELL/2, y+ws_CELL/2);
          } else if (val === 1) {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 1; ws_ctx.setLineDash([2,2]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          } else {
            ws_ctx.strokeStyle = col.stroke; ws_ctx.lineWidth = 0.5; ws_ctx.setLineDash([]);
            ws_ctx.strokeRect(x+0.5, y+0.5, ws_CELL-1, ws_CELL-1);
          }

          if (r===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(c, x+ws_CELL/2, ws_PAD/2); }
          if (c===0) { ws_ctx.fillStyle='#8a8371'; ws_ctx.font='8px monospace'; ws_ctx.textAlign='center'; ws_ctx.fillText(r, ws_PAD/2, y+ws_CELL/2); }
        }
      }

      let pct = area === 0 ? 0 : Math.round((filled / area) * 100);
      root.querySelector('#ws_statPx').textContent = area;
      root.querySelector('#ws_statSeeds').textContent = numSeeds.size;
      root.querySelector('#ws_statFill').textContent = pct + '%';
      
      if (stepInfo) {
        root.querySelector('#ws_statStep').textContent = stepInfo.done ? 'Concluído' : stepInfo.k;
      } else {
        root.querySelector('#ws_statStep').textContent = 'Finalizado';
      }
    }

    function ws_updateSEUI() {
      for (let r=0; r<3; r++) {
        for (let c=0; c<3; c++) {
          if(r===1 && c===1) continue;
          const btn = root.querySelector('#ws_se_' + r + '_' + c);
          if(!btn) continue;
          if(ws_b[r][c]) {
            btn.textContent = '1';
            btn.className = 'ws_se_btn ws_active';
          } else {
            btn.textContent = '0';
            btn.className = 'ws_se_btn ws_off';
          }
        }
      }
    }

    window.ws_toggleSE = function(r, c) {
      if(r===1 && c===1) return;
      ws_b[r][c] = !ws_b[r][c];
      ws_updateSEUI();
      ws_refresh();
    };

    window.ws_setPresetSE = function(type) {
      for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = false;
      ws_b[1][1] = true;
      if (type === 'cross') {
        ws_b[0][1] = ws_b[1][0] = ws_b[1][2] = ws_b[2][1] = true;
      } else if (type === 'square') {
        for (let r=0; r<3; r++) for (let c=0; c<3; c++) ws_b[r][c] = true;
      }
      ws_updateSEUI();
      ws_refresh();
    };

    function ws_refresh() {
      const res = ws_compute();
      if (ws_mode === 'anim') {
        ws_animSteps = res.steps;
        ws_animIdx = 0;
        ws_drawState(ws_animSteps[0].g, ws_animSteps[0]);
      } else {
        ws_drawState(res.final, null);
      }
    }

    function ws_setTool(t) {
      ws_currentTool = t;
      ['mask','erase','s2','s3','s4'].forEach(id => {
        const el = root.querySelector('#ws_tool_' + id);
        if (el) el.classList.toggle('ws_active', id === t);
      });
    }

    function ws_setMode(m) {
      ws_mode = m;
      ws_stopPlay();
      root.querySelector('#ws_btnLabel').classList.toggle('ws_active', m==='label');
      root.querySelector('#ws_btnAnim').classList.toggle('ws_active', m==='anim');
      root.querySelector('#ws_animCtrl').style.display = m==='anim' ? 'block' : 'none';
      ws_refresh();
    }

    function ws_animStep(d) {
      ws_animIdx = Math.max(0, Math.min(ws_animSteps.length-1, ws_animIdx+d));
      ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
    }

    function ws_togglePlay() {
      if (ws_playing) ws_stopPlay(); else ws_startPlay();
    }

    function ws_startPlay() {
      if (ws_animSteps.length === 0) return;
      ws_playing = true;
      root.querySelector('#ws_btnPlay').textContent = '⏸';
      function tick() {
        if (ws_animIdx >= ws_animSteps.length-1) { ws_stopPlay(); return; }
        ws_animIdx++;
        ws_drawState(ws_animSteps[ws_animIdx].g, ws_animSteps[ws_animIdx]);
        const spd = +root.querySelector('#ws_speedSlider').value;
        const delay = Math.round(1100 - spd * 100);
        ws_playTimer = setTimeout(tick, delay);
      }
      tick();
    }

    function ws_stopPlay() {
      ws_playing = false;
      if (ws_playTimer) { clearTimeout(ws_playTimer); ws_playTimer = null; }
      root.querySelector('#ws_btnPlay').textContent = '▶';
    }

    function ws_clearGrid() {
      ws_grid = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_markers = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
      ws_stopPlay();
      ws_refresh();
    }

    const ws_PRESETS = {
      moedas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=3; r<=10; r++) for(let c=2; c<=7; c++) g[r][c] = 1;
        g[3][2]=0; g[3][7]=0; g[10][2]=0; g[10][7]=0; 
        for(let r=4; r<=11; r++) for(let c=6; c<=12; c++) g[r][c] = 1;
        g[4][6]=0; g[4][12]=0; g[11][6]=0; g[11][12]=0;
        m[6][4] = 2;
        m[8][9] = 3;
        return {g, m};
      },
      celulas: () => {
        let g = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        let m = Array.from({length:ws_ROWS}, () => new Array(ws_COLS).fill(0));
        for(let r=2; r<=12; r++) for(let c=3; c<=12; c++) {
           if((r>=3 && r<=11) || (c>=4 && c<=11)) g[r][c] = 1;
        }
        g[2][3]=g[2][12]=g[12][3]=g[12][12]=0;
        m[4][6] = 2;
        m[10][5] = 3;
        m[7][10] = 4;
        return {g, m};
      }
    };

    function ws_loadPreset(name) {
      const {g, m} = ws_PRESETS[name]();
      ws_grid = g; ws_markers = m;
      ws_stopPlay();
      ws_refresh();
    }

    function ws_cellAt(e) {
      const rect = ws_cv.getBoundingClientRect();
      const sx = ws_W / rect.width, sy = ws_H / rect.height;
      const raw = e.touches ? e.touches[0] : e;
      const c = Math.floor(((raw.clientX - rect.left) * sx - ws_PAD) / ws_CELL);
      const r = Math.floor(((raw.clientY - rect.top) * sy - ws_PAD) / ws_CELL);
      return {r, c};
    }

    function ws_applyTool(r, c) {
      if (ws_currentTool === 'mask') {
        ws_grid[r][c] = 1;
        if (ws_markers[r][c] > 0) ws_markers[r][c] = 0;
      } else if (ws_currentTool === 'erase') {
        ws_grid[r][c] = 0;
        ws_markers[r][c] = 0;
      } else {
        let seedVal = parseInt(ws_currentTool.replace('s',''));
        ws_grid[r][c] = 1;
        ws_markers[r][c] = seedVal;
      }
    }

    ws_cv.addEventListener('mousedown', function(e){
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_painting = true;
      ws_applyTool(r, c);
      ws_refresh();
    });

    ws_cv.addEventListener('mousemove', function(e){
      if (!ws_painting) return;
      const {r, c} = ws_cellAt(e);
      if (r<0 || r>=ws_ROWS || c<0 || c>=ws_COLS) return;
      ws_applyTool(r, c);
      ws_refresh();
    });
    
    document.addEventListener('mouseup', () => ws_painting = false);

    root.querySelectorAll('[data-tool]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setTool(this.getAttribute('data-tool'));
      });
    });

    root.querySelectorAll('[data-se]').forEach(btn => {
      btn.addEventListener('click', function() {
        const parts = this.getAttribute('data-se').split(',');
        ws_toggleSE(parseInt(parts[0], 10), parseInt(parts[1], 10));
      });
    });

    root.querySelectorAll('[data-se-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setPresetSE(this.getAttribute('data-se-preset'));
      });
    });

    root.querySelectorAll('[data-mode]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_setMode(this.getAttribute('data-mode'));
      });
    });

    root.querySelectorAll('[data-preset]').forEach(btn => {
      btn.addEventListener('click', function() {
        ws_loadPreset(this.getAttribute('data-preset'));
      });
    });

    root.querySelectorAll('[data-action]').forEach(btn => {
      btn.addEventListener('click', function() {
        const act = this.getAttribute('data-action');
        if(act === 'clear') ws_clearGrid();
        if(act === 'prev') ws_animStep(-1);
        if(act === 'next') ws_animStep(1);
        if(act === 'play') ws_togglePlay();
      });
    });

    ws_updateSEUI();
    ws_loadPreset('moedas');
  }

  function tryInitSim04Watershed(){
    var root = document.getElementById('sim-04-watershed');
    if (root) initSim04Watershed(root); else setTimeout(tryInitSim04Watershed, 200);
  }
  tryInitSim04Watershed();
})();
</script>
""")

**Figure 4.24:** Interactive simulator of the *Watershed* algorithm by morphological propagation. Draw the mask, position the markers and adjust the Structuring Element to observe the flooding. When basins meet simultaneously, the tie is resolved by randomly assuming one of the regions.


<figure id="fig-04-sim-04-watershed">
  <img src="imagens/fig-04-sim-04-watershed.png" alt=" Interactive simulator of the *Watershed* algorithm by morphological propagation. Draw the mask, position the markers and adjust the Structuring Element to observe the flooding. When basins meet simultaneously, the tie is resolved by randomly assuming one of the regions. " style="max-width:80%" />
  <figcaption><strong>Figure 4.24:</strong>  Interactive simulator of the *Watershed* algorithm by morphological propagation. Draw the mask, position the markers and adjust the Structuring Element to observe the flooding. When basins meet simultaneously, the tie is resolved by randomly assuming one of the regions. </figcaption>
</figure>

#### Morphological *Watershed* Pipeline

Marker-based *watershed* typically integrates into a broader segmentation workflow. In real images, preprocessing steps are often required to enhance contrast, reduce noise, and generate reliable markers. This complete workflow is summarized in [Table 4.5](#tbl-04-watershed-pipeline-real).

<a id="tbl-04-watershed-pipeline-real"></a>

**Tabela 4.5:** Complete marker-based *watershed* pipeline for real images.

| Step | Operation | Purpose |
| --- | --- | --- |
| 1 | CLAHE + Smoothing | Contrast enhancement and noise reduction |
| 2 | Thresholding | Initial separation between object and background |
| 3 | Opening/Closing | Removal of noise and small imperfections |
| 4 | Mask Dilation | Identification of Sure Background |
| 5 | Distance Transform + Threshold | Identification of Sure Foreground |
| 6 | Uncertain Region | Difference between Sure Background and Sure Foreground |
| 7 | `mm::watershed` | Propagation of markers through the uncertain region |


To focus exclusively on the concepts of Distance Transform, markers, and topographical flooding, the example in [Figure 4.25](#fig-04-watershed-didatico) uses a synthetic binary image and adopts a simplified workflow, summarized in [Table 4.6](#tbl-04-watershed-pipeline-simples).

<a id="tbl-04-watershed-pipeline-simples"></a>

**Tabela 4.6:** Simplified pipeline used in the didactic example of [Figure 4.25](#fig-04-watershed-didatico).

| Step | Operation | Purpose |
| --- | --- | --- |
| 1 | Distance Transform | Construction of the topographical surface |
| 2 | DT Threshold | Extraction of markers (Sure Foreground) |
| 3 | Dilation | Determination of Sure Background |
| 4 | Uncertain Region | Difference between background and markers |
| 5 | `mm::watershed` | Propagation of markers and generation of boundaries |


In [56]:
%%writefile tmp/fig_04_watershed_didatico.cpp
#define MM_OUT "tmp/fig_04_watershed_didatico.png"
//| label: fig-04-watershed-didatico
//| fig-cap: "*Pipeline* *watershed* delimitado por máscara em imagem binária 20×20."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
    // 1. Synthetic image and Distance Transform
    mm::Image f_sint(20, 20);
    f_sint = mm::circle(f_sint, 6, 10, 5, 255, -1);
    f_sint = mm::circle(f_sint, 14, 10, 5, 255, -1);
    mm::Image dist = mm::dist(f_sint);

    // 2. Markers (distance peaks)
    mm::Image m(20, 20);
    double maxDist = 0;
    for (int y = 0; y < dist.h; y++)
        for (int x = 0; x < dist.w; x++)
            maxDist = std::max(maxDist, (double)dist.at(y, x));
    for (int y = 0; y < dist.h; y++)
        for (int x = 0; x < dist.w; x++)
            m.at(y, x) = (dist.at(y, x) > 0.8 * maxDist) ? 255 : 0;

    // 3. Conditional Watershed0 execution (m=markers first, mask=f_sint)
    mm::Image w_reg  = mm::watershed0(m, f_sint, "region");
    mm::Image w_line = mm::watershed0(m, f_sint, "line");

    //w_reg  = mm::watershedB(m, mask=f_sint, op='region')
    //w_line = mm::watershedB(m, mask=f_sint, op='line')

    w_reg  = mm::watershed(m, f_sint, "region");
    w_line = mm::watershed(m, f_sint, "line");

    // 4. Displaying the results
    mm::show({f_sint, dist, m, w_reg, w_line}, MM_OUT, {"Original", "Distância", "Marcadores", "Regiões", "Linhas"}, 5);
    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(f_sint, "tmp/fig_04_watershed_didatico_0.png");
mm::write(dist, "tmp/fig_04_watershed_didatico_1.png");
mm::write(m, "tmp/fig_04_watershed_didatico_2.png");
mm::write(w_reg, "tmp/fig_04_watershed_didatico_3.png");
mm::write(w_line, "tmp/fig_04_watershed_didatico_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_watershed_didatico.cpp


In [57]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_watershed_didatico.cpp -o tmp/fig_04_watershed_didatico -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_watershed_didatico \
  && test -f "tmp/fig_04_watershed_didatico.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_watershed_didatico.png"

[1] Original
[2] Distância
[3] Marcadores
[4] Regiões
[5] Linhas


In [58]:
mm.show(
    [
        mm.read("tmp/fig_04_watershed_didatico_0.png"),
        mm.read("tmp/fig_04_watershed_didatico_1.png"),
        mm.read("tmp/fig_04_watershed_didatico_2.png"),
        mm.read("tmp/fig_04_watershed_didatico_3.png"),
        mm.read("tmp/fig_04_watershed_didatico_4.png"),
    ],
    titles=[
        'Original',
        'Distância',
        'Marcadores',
        'Regiões',
        'Linhas',
    ],
    cols=5,
    figsize=(16, 4),
)

<Figure size 2400x600 with 5 Axes>

**Figure 4.25:** *Pipeline* *watershed* delimitado por máscara em imagem binária 20×20.


**Application to overlapping coins:**

In [59]:
%%writefile tmp/fig_04_watershed_moedas.cpp
#define MM_OUT "tmp/fig_04_watershed_moedas.png"
// Compile with C++17: g++ -std=c++17 -o program program.cpp -I. $(pkg-config --cflags --libs opencv4)
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <set>
#include <numeric>
#include <algorithm>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    mm::Image img_base = img_coins_gray;

    // 1. Simulate overlapping/connected coins (using base binary mask)
    mm::Image img_sobrepostas = mm::dil(img_final, mm::sebox(40));

    // 2. Morphological opening to clean noise
    mm::Image opening = mm::open(img_sobrepostas, mm::SE::box(2));

    // 3. Distance Transform
    mm::Image dist = mm::dist(opening);
    mm::Image dist_vis;
    double max_dist = 0;
    for (int y = 0; y < dist.h; y++) {
        for (int x = 0; x < dist.w; x++) {
            max_dist = std::max(max_dist, (double)dist.at(y, x));
        }
    }
    if (max_dist > 0) {
        dist_vis = mm::Image(dist.h, dist.w);
        for (int y = 0; y < dist.h; y++) {
            for (int x = 0; x < dist.w; x++) {
                dist_vis.at(y, x) = (unsigned char)(255 * ((double)dist.at(y, x) / max_dist));
            }
        }
    } else {
        dist_vis = dist;
    }

    // 4. Safe peaks (Coin markers)
    mm::Image picos(dist.h, dist.w);
    for (int y = 0; y < dist.h; y++) {
        for (int x = 0; x < dist.w; x++) {
            if (dist.at(y, x) > 0.5 * max_dist) {
                picos.at(y, x) = 255;
            }
        }
    }

    // 5. Watershed execution (adjusted to use new signature)
    // We pass 'opening' directly as mask, since it delimits the expansion scope of the coins
    mm::Image ws_region = mm::watershed(picos, opening, "region");
    mm::Image ws_line = mm::watershed(picos, opening, "line");

    // 6. Object counting (Ignores background 0)
    std::set<int> unique_labels;
    for (int y = 0; y < ws_region.h; y++) {
        for (int x = 0; x < ws_region.w; x++) {
            int val = ws_region.at(y, x);
            if (val > 0) unique_labels.insert(val);
        }
    }
    std::vector<int> labels(unique_labels.begin(), unique_labels.end());
    std::cout << "Objects detected: " << labels.size() << "\n";

    // 7. Final Annotation
    cv::Mat img_base_cv(img_base.h, img_base.w, CV_8UC1, img_base.data.data());
    cv::Mat img_annotated_cv;
    cv::cvtColor(img_base_cv, img_annotated_cv, cv::COLOR_GRAY2BGR);

    int idx = 0;
    for (int label_id : labels) {
        // Create mask for this label
        mm::Image mask_reg(ws_region.h, ws_region.w);
        long long mask_sum = 0;
        long long sum_y = 0, sum_x = 0;
        int count = 0;
        for (int y = 0; y < ws_region.h; y++) {
            for (int x = 0; x < ws_region.w; x++) {
                if (ws_region.at(y, x) == label_id) {
                    mask_reg.at(y, x) = 255;
                    mask_sum += 1;
                    sum_y += y;
                    sum_x += x;
                    count++;
                }
            }
        }
        if (mask_sum < 500) continue;

        int cy = (count > 0) ? (int)(sum_y / count) : 0;
        int cx = (count > 0) ? (int)(sum_x / count) : 0;

        cv::putText(img_annotated_cv, std::to_string(idx + 1), cv::Point(cx - 25, cy + 20),
                    cv::FONT_HERSHEY_SIMPLEX, 3.2, cv::Scalar(0, 255, 0), 8, cv::LINE_AA);
        idx++;
    }

    // Draw separation lines in red
    mm::Image mask_ann = mm::dil(ws_line, mm::sebox(5));  // 11x11 box kernel
    cv::Mat img_annotated_final(img_annotated_cv.rows, img_annotated_cv.cols, CV_8UC3, img_annotated_cv.data);
    for (int y = 0; y < mask_ann.h; y++) {
        for (int x = 0; x < mask_ann.w; x++) {
            if (mask_ann.at(y, x) > 0) {
                img_annotated_final.at<cv::Vec3b>(y, x) = cv::Vec3b(0, 0, 255);
            }
        }
    }

    // Convert back to mm::Image for display
    mm::Image img_annotated(img_annotated_cv.rows, img_annotated_cv.cols, 3);
    std::memcpy(img_annotated.data.data(), img_annotated_cv.data, img_annotated.data.size());

    // Display
    mm::show(std::vector<mm::Image>{img_base, img_sobrepostas, dist_vis, picos, ws_region, img_annotated},
             MM_OUT,
             std::vector<std::string>{"Original", "Sobrepostas", "Distância", "Marcadores", "Watershed", "Anotado"},
             3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_base, "tmp/fig_04_watershed_moedas_0.png");
mm::write(img_sobrepostas, "tmp/fig_04_watershed_moedas_1.png");
mm::write(dist_vis, "tmp/fig_04_watershed_moedas_2.png");
mm::write(picos, "tmp/fig_04_watershed_moedas_3.png");
mm::write(ws_region, "tmp/fig_04_watershed_moedas_4.png");
mm::write(img_annotated, "tmp/fig_04_watershed_moedas_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_watershed_moedas.cpp


In [60]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_watershed_moedas.cpp -o tmp/fig_04_watershed_moedas -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_watershed_moedas \
  && test -f "tmp/fig_04_watershed_moedas.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_watershed_moedas.png"

Objects detected: 12


[1] Original
[2] Sobrepostas
[3] Distância
[4] Marcadores
[5] Watershed
[6] Anotado


In [61]:
mm.show(
    [
        mm.read("tmp/fig_04_watershed_moedas_0.png"),
        mm.read("tmp/fig_04_watershed_moedas_1.png"),
        mm.read("tmp/fig_04_watershed_moedas_2.png"),
        mm.read("tmp/fig_04_watershed_moedas_3.png"),
        mm.read("tmp/fig_04_watershed_moedas_4.png"),
        mm.read("tmp/fig_04_watershed_moedas_5.png"),
    ],
    titles=[
        'Original',
        'Sobrepostas',
        'Distância',
        'Marcadores',
        'Watershed',
        'Anotado',
    ],
    cols=3,
    rows=2,
    figsize=(12, 8),
)

<Figure size 1800x1200 with 6 Axes>

**Figure 4.26:** *Pipeline* *Watershed* para separação de moedas sobrepostas: da máscara binária dilatada até os contornos finais anotados sobre a imagem original.


## 4.5 Shape Component Extraction and Descriptors

After segmentation and morphological refinement, the next step consists of identifying each object present in the image individually and extracting its geometric properties. This stage is fundamental for measurement, classification, and pattern recognition tasks.

A **connected component** is a maximal set of pixels belonging to the object that remain mutually connected according to a previously defined connectivity relation (4- or 8-connectivity). After labeling, each component receives a unique identifier, allowing its characteristics to be analyzed individually.

OpenCV offers two complementary approaches for this analysis, summarized in [Table 4.7](#tbl-04-componentes-vs-contornos).

<a id="tbl-04-componentes-vs-contornos"></a>

**Tabela 4.7:** Comparison between approaches based on connected components and contours.

| | `connectedComponentsWithStats` | `findContours` |
|---|---|---|
| **Returns** | label per pixel and statistics per component | sequence of points describing the border |
| **Direct descriptors** | area, bounding box, and centroid | perimeter, shape, and hierarchy |
| **Objects in contact** | tends to merge connected regions | tends to produce a single external contour |
| **Typical use** | counting, filtering, and labeling | geometric analysis and shape descriptors |


### 4.5.1 Labeling and Component Statistics

`mm::label0` assigns a label to each connected component. From the labeled image, the **area** (pixel count) and the **bounding box** of each object are obtained by scanning ([Figure 4.27](#fig-04-componentes)). OpenCV's `connectedComponentsWithStats`, which returns these statistics ready-made, and the colored annotations on the image are handled in the Python track.

In [62]:
%%writefile tmp/fig_04_componentes.cpp
#define MM_OUT "tmp/fig_04_componentes.png"
//| label: fig-04-componentes
//| fig-cap: "Componentes conexos extraídos após o pipeline CLAHE → Otsu → limpeza morfológica. Cada objeto é colorido com cor distinta e anotado com sua área em pixels."
//| echo: true
//| output: true

#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    // 1. Labeling and statistics
    // Get dimensions from the final image
    int h = img_final.h;
    int w = img_final.w;

    // Wrap the final image as a cv::Mat for connectedComponentsWithStats
    cv::Mat imgMat(h, w, CV_8UC1, img_final.data.data());

    // Use OpenCV's connected components with statistics
    cv::Mat labelsMat, statsMat, centroidsMat;
    int n = cv::connectedComponentsWithStats(imgMat, labelsMat, statsMat, centroidsMat, 8);

    // Copy labels back to a single-channel mm::Image
    mm::Image labels(h, w);
    std::memcpy(labels.data.data(), labelsMat.data, h * w);

    // 2. Component coloring — FIXED palette (generated once with
    // np.random.seed(4)) so the C++ track reproduces color by color
    // without depending on numpy's generator. If n exceeds len(PALETA), cycles.
    std::vector<std::vector<int>> PALETA = {
        {0, 0, 0}, {224, 82, 193}, {233, 155, 73}, {190, 247, 244},
        {103, 94, 179}, {51, 154, 59}, {137, 96, 232}, {250, 243, 205},
        {100, 141, 208}, {228, 187, 163}, {202, 108, 214}, {131, 105, 104},
        {86, 153, 81}
    };

    // Build the colored image: for each pixel, pick color[labels[y,x] % PALETA.size()]
    mm::Image img_colored(h, w, 3);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int label = labels.at(y, x);
            int idx = label % (int)PALETA.size();
            for (int c = 0; c < 3; c++) {
                img_colored.at(y, x, c) = (unsigned char)PALETA[idx][c];
            }
        }
    }
    // Background (label 0) is black
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            if (labels.at(y, x) == 0) {
                for (int c = 0; c < 3; c++) img_colored.at(y, x, c) = 0;
            }
        }
    }

    // Copy for annotation
    mm::Image img_annotated = img_colored;

    // Wrap annotated as a cv::Mat for putText
    cv::Mat annotatedMat(h, w, CV_8UC3, img_annotated.data.data());

    // 3. Descriptor table
    std::cout << "Componentes detectados (excluindo fundo): " << (n - 1) << "\n";
    std::cout << "\n" << std::setw(4) << "ID" << std::setw(8) << "Área"
              << std::setw(6) << "cx" << std::setw(6) << "cy"
              << std::setw(6) << "w" << std::setw(6) << "h" << "\n";
    std::cout << "------------------------------------------" << "\n";

    for (int i = 1; i < n; i++) {
        int area = statsMat.at<int>(i, cv::CC_STAT_AREA);
        int left = statsMat.at<int>(i, cv::CC_STAT_LEFT);
        int top  = statsMat.at<int>(i, cv::CC_STAT_TOP);
        int w_i  = statsMat.at<int>(i, cv::CC_STAT_WIDTH);
        int h_i  = statsMat.at<int>(i, cv::CC_STAT_HEIGHT);
        int cx   = (int)(centroidsMat.at<double>(i, 0));
        int cy   = (int)(centroidsMat.at<double>(i, 1));

        std::cout << std::setw(4) << i << std::setw(8) << area
                  << std::setw(6) << cx << std::setw(6) << cy
                  << std::setw(6) << w_i << std::setw(6) << h_i << "\n";

        // Draw label with black outline then red text
        std::string text = std::to_string(i) + ": " + std::to_string(area);
        cv::putText(annotatedMat, text, cv::Point(cx - 150, cy + 18),
                    cv::FONT_HERSHEY_SIMPLEX, 2.0, cv::Scalar(0, 0, 0), 10, cv::LINE_AA);
        cv::putText(annotatedMat, text, cv::Point(cx - 150, cy + 18),
                    cv::FONT_HERSHEY_SIMPLEX, 2.0, cv::Scalar(0, 0, 255), 5, cv::LINE_AA);
    }

    // Copy annotated Mat back into mm::Image
    std::memcpy(img_annotated.data.data(), annotatedMat.data, h * w * 3);

    // Display results
    mm::show(std::vector<mm::Image>{img_coins_gray, img_final, img_colored, img_annotated},
             MM_OUT,
             std::vector<std::string>{"Original", "Segmentação Final", "Componentes Conexos", "Áreas Anotadas"},
             4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_coins_gray, "tmp/fig_04_componentes_0.png");
mm::write(img_final, "tmp/fig_04_componentes_1.png");
mm::write(img_colored, "tmp/fig_04_componentes_2.png");
mm::write(img_annotated, "tmp/fig_04_componentes_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_componentes.cpp


In [63]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_componentes.cpp -o tmp/fig_04_componentes -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_componentes \
  && test -f "tmp/fig_04_componentes.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_componentes.png"

Componentes detectados (excluindo fundo): 12

  ID   Área    cx    cy     w     h
------------------------------------------
   1  231969  1484   280   553   542
   2  158967   917   250   453   468
   3  139229   250   312   433   419
   4  175550   857   821   474   465
   5  222882  1442   889   539   531
   6  210043   316   977   527   516
   7  343376   934  1559   667   665
   8  213641  1555  1574   530   515
   9  147880   328  1543   432   438
  10  187280   363  2111   487   492
  11  150110  1487  2215   433   444
  12  215387   932  2292   528   522


[1] Original
[2] Segmentação Final
[3] Componentes Conexos
[4] Áreas Anotadas


In [64]:
mm.show(
    [
        mm.read("tmp/fig_04_componentes_0.png"),
        mm.read("tmp/fig_04_componentes_1.png"),
        mm.read("tmp/fig_04_componentes_2.png"),
        mm.read("tmp/fig_04_componentes_3.png"),
    ],
    titles=[
        'Original',
        'Segmentação Final',
        'Componentes Conexos',
        'Áreas Anotadas',
    ],
    cols=4,
    figsize=(18, 6),
)

<Figure size 2700x900 with 4 Axes>

**Figure 4.27:** Componentes conexos extraídos após o *pipeline* CLAHE → Otsu → limpeza morfológica. Cada objeto é colorido com cor distinta e anotado com sua área em pixels.


### 4.5.2 Shape Descriptors

Descriptors based on **vector contour** — perimeter (`arcLength`), polygon area (`contourArea`), circularity, moments, and polygonal approximation (`approxPolyDP`) — depend on boundary tracing (`findContours`), which is absent in `morph.hpp`. They remain only in the Python track. In the C++ track, the **morphological gradient** (`mm::gradm`) highlights the contour of each object ([Figure 4.28](#fig-04-contornos)).

In [65]:
%%writefile tmp/fig_04_contornos.cpp
#define MM_OUT "tmp/fig_04_contornos.png"
// Compile: g++ -std=c++17 -o contornos contornos.cpp -I. -lopencv_core -lopencv_imgproc -lopencv_highgui
#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <iomanip>
#include <vector>
#include <cmath>
#include <cstring>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    //| label: fig-04-contornos
    //| fig-cap: "Contornos extraídos com *findContours*. Cada moeda é anotada com sua circularidade — valores próximos de 1 confirmam forma circular."
    //| echo: true
    //| output: true

    // Bridge mm::Image to cv::Mat (same buffer, no copy)
    cv::Mat img_final_cv(img_final.h, img_final.w, CV_8UC1, img_final.data.data());
    cv::Mat img_coins_gray_cv(img_coins_gray.h, img_coins_gray.w, CV_8UC1, img_coins_gray.data.data());

    std::vector<std::vector<cv::Point>> contornos;
    cv::findContours(img_final_cv, contornos, cv::RETR_EXTERNAL, cv::CHAIN_APPROX_SIMPLE);

    cv::Mat img_contornos, img_circulares;
    cv::cvtColor(img_coins_gray_cv, img_contornos, cv::COLOR_GRAY2BGR);
    img_circulares = img_contornos.clone();

    std::cout << "Contornos detectados: " << contornos.size() << "\n";
    std::cout << std::setw(4) << "ID" << std::setw(8) << "Área" 
              << std::setw(10) << "Perímetro" << std::setw(14) << "Circularidade" << "\n";
    std::cout << std::string(42, '-') << "\n";

    for (size_t idx = 0; idx < contornos.size(); ++idx) {
        int i = static_cast<int>(idx) + 1;
        const auto& cnt = contornos[idx];

        double area  = cv::contourArea(cnt);
        double perim = cv::arcLength(cnt, true);
        double circ  = (perim > 0) ? (4 * M_PI * area / (perim * perim)) : 0;
        cv::Moments m = cv::moments(cnt);
        int cx = (m.m00 > 0) ? static_cast<int>(m.m10 / m.m00) : 0;
        int cy = (m.m00 > 0) ? static_cast<int>(m.m01 / m.m00) : 0;

        std::cout << std::setw(4) << i << std::setw(8) << std::fixed << std::setprecision(0) << area
                  << std::setw(10) << std::setprecision(1) << perim
                  << std::setw(14) << std::setprecision(3) << circ << "\n";

        std::vector<std::vector<cv::Point>> cnt_list = {cnt};
        cv::drawContours(img_contornos,  cnt_list, -1, cv::Scalar(0, 255, 0), 3);
        cv::drawContours(img_circulares, cnt_list, -1, cv::Scalar(0, 255, 0), 3);

        std::vector<std::pair<cv::Scalar, int>> cores_espessuras = {
            {cv::Scalar(0, 0, 0), 8}, 
            {cv::Scalar(255, 0, 0), 3}
        };
        for (const auto& [cor, esp] : cores_espessuras) {
            char texto[32];
            std::snprintf(texto, sizeof(texto), "%.2f", circ);
            cv::putText(img_circulares, texto, cv::Point(cx - 80, cy + 15),
                        cv::FONT_HERSHEY_SIMPLEX, 3.6, cor, esp, cv::LINE_AA);
        }
    }

    // Copy results back to mm::Image
    mm::Image img_contornos_mm(img_contornos.rows, img_contornos.cols, 3);
    std::memcpy(img_contornos_mm.data.data(), img_contornos.data, img_contornos_mm.data.size());

    mm::Image img_circulares_mm(img_circulares.rows, img_circulares.cols, 3);
    std::memcpy(img_circulares_mm.data.data(), img_circulares.data, img_circulares_mm.data.size());

    mm::show(std::vector<mm::Image>{img_final, img_contornos_mm, img_circulares_mm},
             MM_OUT,
             std::vector<std::string>{"Segmentação Final", "Contornos", "Circularidade"},
             3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_final, "tmp/fig_04_contornos_0.png");
mm::write(img_contornos, "tmp/fig_04_contornos_1.png");
mm::write(img_circulares, "tmp/fig_04_contornos_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_contornos.cpp


In [66]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_contornos.cpp -o tmp/fig_04_contornos -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_contornos \
  && test -f "tmp/fig_04_contornos.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_contornos.png"

Contornos detectados: 12
  ID   ÁreaPerímetro Circularidade
------------------------------------------
   1  214626    1769.6         0.861
   2  149480    1465.1         0.875
   3  186570    1654.9         0.856
   4  147252    1458.6         0.870
   5  212886    1756.3         0.867


   6  342424    2232.5         0.863
   7  209282    1767.7         0.842
   8  222108    1809.7         0.852
   9  174854    1616.4         0.841
  10  138602    1471.5         0.804
  11  158306    1538.7         0.840
  12  231181    1847.4         0.851


[1] Segmentação Final
[2] Contornos
[3] Circularidade


In [67]:
mm.show(
    [
        mm.read("tmp/fig_04_contornos_0.png"),
        mm.read("tmp/fig_04_contornos_1.png"),
        mm.read("tmp/fig_04_contornos_2.png"),
    ],
    titles=[
        'Segmentação Final',
        'Contornos',
        'Circularidade',
    ],
    cols=3,
    figsize=(18, 6),
)

<Figure size 2700x900 with 3 Axes>

**Figure 4.28:** Contornos extraídos com *findContours*. Cada moeda é anotada com sua circularidade — valores próximos de 1 confirmam forma circular.


### 4.5.3 Connection with Modern Object Detection

The descriptors extracted in the previous sections — especially *bounding boxes*, centroids, areas, and shape measures — establish a natural bridge between classical morphological segmentation and modern object detection systems. Although the techniques studied in this chapter use operations on pixels and segmented regions, many of the representations produced are directly compatible with the formats employed in contemporary computer vision models.

Deep learning-based detectors, such as the YOLO (*You Only Look Once*) family (REDMON, 2016), operate directly on color images and produce, for each detected object, a *bounding box* described by the center $(cx,cy)$ and the dimensions $(w,h)$, as well as a class and a confidence score. This representation shares the same basic geometric structure obtained by `connectedComponentsWithStats`, although it is produced by a learned model rather than by explicit segmentation.

[Figure 4.29](#fig-04-bbox) illustrates how *bounding boxes* obtained by morphology can be exported in the YOLO format to compose datasets used in training or evaluating detectors.

In [68]:
%%writefile tmp/fig_04_bbox.cpp
#define MM_OUT "tmp/fig_04_bbox.png"
// Compile with: g++ -std=c++17 -o program program.cpp -lopencv_core -lopencv_imgproc -lopencv_highgui -I/usr/include/opencv4 $(pkg-config --cflags --libs opencv4)

#include "morph.hpp"
#include <opencv2/opencv.hpp>
#include <iostream>
#include <iomanip>
#include <string>
#include <vector>
#include <fstream>
#include <cstring>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_coins_gray = mm::_read_state("tmp/state/img_coins_gray_8.png");
mm::Image img_final = mm::_read_state("tmp/state/img_final_55.png");
// [pdi:state-io:end]

    // Recalculates labels/statistics from the final segmentation
    // Connected components analysis using OpenCV
    cv::Mat img_final_cv(img_final.h, img_final.w, CV_8UC1, img_final.data.data());

    cv::Mat labels, stats, centroids;
    int n = cv::connectedComponentsWithStats(img_final_cv, labels, stats, centroids, 8);

    int H_img = img_coins_gray.h;
    int W_img = img_coins_gray.w;

    // Convert grayscale to BGR for drawing
    cv::Mat img_bbox_cv;
    cv::Mat img_coins_cv(img_coins_gray.h, img_coins_gray.w, CV_8UC1, img_coins_gray.data.data());
    cv::cvtColor(img_coins_cv, img_bbox_cv, cv::COLOR_GRAY2BGR);

    int CLASSE = 0;   // 0 = coin (only category in this example)

    std::cout << std::setw(4) << "cls" << std::setw(9) << "cx_n" 
              << std::setw(9) << "cy_n" << std::setw(9) << "w_n" 
              << std::setw(9) << "h_n" << "  <- YOLO format" << std::endl;
    std::cout << std::string(54, '-') << std::endl;

    std::vector<std::string> yolo_lines;
    for (int i = 1; i < n; i++) {
        int x0 = stats.at<int>(i, cv::CC_STAT_LEFT);
        int y0 = stats.at<int>(i, cv::CC_STAT_TOP);
        int w  = stats.at<int>(i, cv::CC_STAT_WIDTH);
        int h  = stats.at<int>(i, cv::CC_STAT_HEIGHT);

        double cx_n = static_cast<double>(x0 + w / 2.0) / W_img;
        double cy_n = static_cast<double>(y0 + h / 2.0) / H_img;
        double w_n  = static_cast<double>(w) / W_img;
        double h_n  = static_cast<double>(h) / H_img;

        char line[256];
        snprintf(line, sizeof(line), "%d %.4f %.4f %.4f %.4f", 
                 CLASSE, cx_n, cy_n, w_n, h_n);
        yolo_lines.push_back(std::string(line));
        std::cout << std::setw(4) << CLASSE << std::fixed << std::setprecision(4)
                  << std::setw(9) << cx_n << std::setw(9) << cy_n 
                  << std::setw(9) << w_n << std::setw(9) << h_n << std::endl;

        cv::rectangle(img_bbox_cv, cv::Point(x0, y0), cv::Point(x0 + w, y0 + h), 
                      cv::Scalar(0, 255, 0), 4);
        cv::putText(img_bbox_cv, "moeda", cv::Point(x0 + 8, y0 + 60),
                    cv::FONT_HERSHEY_SIMPLEX, 3.8, cv::Scalar(255, 0, 0), 5, cv::LINE_AA);
    }

    // Export annotation file in YOLO format
    std::ofstream f("moedas.txt");
    for (size_t i = 0; i < yolo_lines.size(); i++) {
        f << yolo_lines[i];
        if (i < yolo_lines.size() - 1) f << "\n";
    }
    f.close();
    std::cout << "\nAnnotation saved in moedas.txt" << std::endl;

    // Copy back to mm::Image for display
    mm::Image img_bbox(img_bbox_cv.rows, img_bbox_cv.cols, img_bbox_cv.channels());
    std::memcpy(img_bbox.data.data(), img_bbox_cv.data, img_bbox.data.size());

    mm::show(
        std::vector<mm::Image>{img_coins_gray, img_final, img_bbox},
        MM_OUT,
        std::vector<std::string>{"Original", "Final Segmentation", "Bounding Boxes (YOLO format)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_coins_gray, "tmp/fig_04_bbox_0.png");
mm::write(img_final, "tmp/fig_04_bbox_1.png");
mm::write(img_bbox, "tmp/fig_04_bbox_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_04_bbox.cpp


In [69]:
!g++ -I. -std=c++17 -DMM_USE_OPENCV -I/usr/include/opencv4 tmp/fig_04_bbox.cpp -o tmp/fig_04_bbox -lopencv_imgproc -lopencv_imgcodecs -lopencv_core \
  && ./tmp/fig_04_bbox \
  && test -f "tmp/fig_04_bbox.png" \
  || echo "⚠ mm::show não gravou tmp/fig_04_bbox.png"

 cls     cx_n     cy_n      w_n      h_n  <- YOLO format
------------------------------------------------------
   0   0.7753   0.1102   0.2880   0.2117
   0   0.4784   0.0984   0.2359   0.1828
   0   0.1331   0.1225   0.2255   0.1637
   0   0.4464   0.3217   0.2469   0.1816
   0   0.7523   0.3471   0.2807   0.2074
   0   0.1664   0.3812   0.2745   0.2016
   0   0.4862   0.6104   0.3474   0.2598
   0   0.8115   0.6154   0.2760   0.2012
   0   0.1724   0.6023   0.2250   0.1711
   0   0.1893   0.8258   0.2536   0.1922
   0   0.7763   0.8652   0.2255   0.1734
   0   0.4854   0.8953   0.2750   0.2039

Annotation saved in moedas.txt


[1] Original
[2] Final Segmentation
[3] Bounding Boxes (YOLO format)


In [70]:
mm.show(
    [
        mm.read("tmp/fig_04_bbox_0.png"),
        mm.read("tmp/fig_04_bbox_1.png"),
        mm.read("tmp/fig_04_bbox_2.png"),
    ],
    titles=[
        'Original',
        'Segmentação Final',
        'Bounding Boxes (formato YOLO)',
    ],
    cols=3,
    figsize=(18, 6),
)

<Figure size 2700x900 with 3 Axes>

**Figure 4.29:** *Bounding boxes* derivadas dos componentes conexos sobrepostas à imagem original. As anotações são exportadas no formato YOLO (*classe cx cy w h*), com coordenadas normalizadas para o intervalo [0,1].


The YOLO format stores each object in a line containing five fields:

$$
\texttt{classe}\;\;\texttt{cx}\;\;\texttt{cy}\;\;\texttt{w}\;\;\texttt{h}
$$

where $(cx,cy)$ represents the center of the *bounding box* and $(w,h)$ its dimensions. All geometric values are normalized to the interval $[0,1]$ with respect to the image's width and height. The **class** is an integer identifier associated with a category defined by the dataset (e.g., `0 → coin`). When there are multiple categories — such as gold coin (`0`), silver coin (`1`), and plastic disk (`2`) — one simply assigns the corresponding identifier to each object before export, maintaining exactly the same annotation format. In the previous example, this information was stored in the `moedas.txt` file.

The workflow presented in this chapter — segmentation → labeling → *bounding box* extraction — conceptually corresponds to the **annotation** (*labeling*) step employed in building training sets for modern detectors. Specialized tools, such as Label Studio and Roboflow, automate this process in complex images, but the fundamental logic remains the same: associating each object with a region of interest and a class. In controlled scenarios, with a uniform background and well-separated objects, morphological techniques can even generate annotations automatically or serve as a starting point for manual labeling, significantly reducing the effort of dataset construction. In more complex real-world applications, however, human validation remains necessary to ensure the quality of annotations.

> ### 📝 Evaluation: IoU (*Intersection over Union*)
>
> A simple way to evaluate the quality of a segmentation is to compare it against a reference mask (*ground truth*). The most widely used metric for this purpose is **IoU** (*Intersection over Union*):
>
> <a id="eq-04-iou"></a>
$$
> \text{IoU}=
> \frac{|A\cap B|}
> {|A\cup B|}
> \tag{4.16}
$$

>
> where $A$ represents the segmentation produced by the algorithm and $B$ the reference segmentation.
>
> The IoU value ranges between 0 and 1. The higher the value, the greater the overlap between the masks. An IoU of 1 indicates perfect correspondence between the obtained segmentation and the reference.
>
> The same metric is also extensively used in object detection, applied to predicted and annotated *bounding boxes*. In this field, IoU values greater than 0.5 are often adopted as the minimum criterion to consider a detection correct.

> ### 💡 Beyond Morphology
>
> The techniques studied in this chapter segment objects by exploiting spatial connectivity, morphological operations, and topographic relief. However, there are alternative approaches based on feature clustering, such as the *k-means* algorithm, Gaussian mixture models (GMM), and more recent methods based on deep learning. These techniques will be revisited in Part II of the book, dedicated to Computer Vision.

## 4.6 Summary

This chapter presented the main segmentation and mathematical morphology techniques, concluding the study of DIP in the spatial domain.

* **Preprocessing and thresholding:** The combination of adaptive CLAHE equalization and Otsu's method proved effective in inducing bimodal separation in the histogram and simplifying the binarization of images with non-uniform illumination.
* **Erosion and dilation:** Fundamental morphological operators based on the search for local minima and maxima in a neighborhood defined by the structuring element $B$. These are dual operators by complement and were implemented using the functions `mm::ero` and `mm::dil`.
* **Opening and closing:** Compositions of erosion and dilation that allow removing noise, smoothing contours, and filling small gaps while preserving the overall structure of objects.
* **Morphological reconstruction:** An iterative geodesic process that propagates a marker within the limits imposed by a mask, forming the basis of operators such as `mm::clohole` and `mm::edgeoff`.
* **Binary cleaning pipeline:** A consolidated workflow composed of CLAHE → Otsu → opening → `mm::clohole` → restricted opening → `mm::edgeoff`, producing suitable masks for quantitative analysis.
* **Grayscale morphology:** An algebraic extension based on weighted minima and maxima, enabling operators such as morphological gradient and *top-hat* filters for enhancing local structures.
* **Distance Transform and Watershed:** The Distance Transform allowed generating automatic markers for the *watershed* algorithm, enabling the separation of adjacent or partially overlapping objects.
* **Connected components and descriptors:** Region labeling (`mm::label0`) and contour extraction (contour extraction) allowed computing geometric descriptors such as area, centroid, perimeter, circularity, and bounding boxes.
* **Connection with Modern Computer Vision:** The bounding boxes extracted by morphology were exported in YOLO format, highlighting the link between classical segmentation techniques and modern object detection systems.

Chapter 5 will introduce processing techniques in the **frequency domain**, addressing the **Fourier Transform**, spectral filtering, and the fundamentals of **image compression**, including DCT, JPEG, and wavelets.

## 4.7 🤖 Using Gemini Notebook as a Complementary Tutor

In this edition, the use of **Gemini Notebook** is encouraged as a complementary learning tool. Based on artificial intelligence, the system uses exclusively the documents provided by the author as its source of knowledge, producing responses aligned with the content and approach adopted throughout this chapter.

> ### ❗ 🎓 Study with the Intelligent Tutor
>
> [🚀 ACCESS GEMINI NOTEBOOK: CHAPTER 04](https://notebooklm.google.com/notebook/5dafcbfa-ad58-44f9-9707-4f761b0a6c70)
>
> #### 🌐 Language and Programming Language
>
> The project for this chapter in Gemini Notebook was built using only the text in **Portuguese** and the code examples in **Python**. If you are studying from the English or French edition, or following the C++ track, the tutor's responses may not correspond exactly to the version you are reading.
>
> #### ⚠️ Notice Regarding AI-Generated Content
>
> Although it is a valuable study support tool, Gemini Notebook may occasionally produce incomplete, inaccurate, or incorrect responses. It is recommended to validate the information by consulting the chapter material, books, scientific articles, and other reliable academic sources. Whenever possible, run and experiment with the practical examples presented throughout the text to consolidate your understanding of the concepts.

## 4.8 Exercise List

1. **(10%)** Manually implement Otsu's criterion without using `mm::threshold`. Compute the interclass variance $\sigma_B^2(T)$ for all thresholds $T \in [0,255]$ using `mm::hist`, identify the optimal threshold $T^*$ and compare the result with the value obtained by OpenCV. Plot $\sigma_B^2$ as a function of $T$ and highlight the maximum point.

2. **(15%)** Apply adaptive thresholding with block sizes of 11, 31, and 51 to an image containing non-uniform illumination. Compare the results with global Otsu thresholding and discuss the advantages and limitations of each approach.

3. **(15%)** Run the watershed *pipeline* on the coins image, varying the threshold applied to the Distance Transform ($0.3$, $0.5$, and $0.7$ times the maximum value). Explain how this parameter influences marker generation, the separation of adjacent objects, and the occurrence of over-segmentation.

4. **(15%)** Using `mm::drawImg`, construct a step-by-step visual demonstration of the erosion of a 7×7 binary image with a 3×3 square structuring element. For each analyzed position, indicate whether the structuring element is fully contained within the object and justify the value assigned to the output pixel.

5. **(15%)** Experimentally demonstrate the duality between erosion and dilation by verifying the identity $(A \ominus B)^c = A^c \oplus \hat{B}$
   using `mm::ero`, `mm::dil`, and `mm::bnot`. Compute the pixel-by-pixel difference between the two sides of the equation and present the result using `mm::histImg` or equivalent visualization.

6. **(15%)** Manually implement the morphological gradient using only `mm::ero` and `mm::dil`, comparing the result with `mm::gradm(img, B)`. Evaluate the effect of different structuring elements (3×3 square, 5×5 disk, and 1×9 line) on edge detection.

7. **(15%)** Construct a complete *pipeline* for counting and classifying coins by size (small, medium, and large) using area and circularity as descriptors. Manually generate a reference mask (*ground truth*) and compute the IoU metric (*Intersection over Union*) to assess segmentation quality. Present the results in a table and through visualizations produced with `mm::show`.

## Chapter References

The theoretical foundation of this chapter is based on the following works:

* Gonzalez (2018) for the concepts of segmentation, Otsu thresholding, Distance Transform, *watershed*, mathematical morphology, and shape descriptors.
* Matheron (1975) and Serra (1982) for the original theoretical foundation, algebraic formulation, and development of Mathematical Morphology.
* Szeliski (2022) for region-based segmentation, connected component labeling, marker-controlled *watershed*, and segmentation evaluation using the IoU metric.
* Bradski (2008) for the practical use of the OpenCV library, including functions such as `mm::dist`, `mm::watershed`, `mm::label0`, and contour extraction.
* Redmon (2016) for an introduction to modern detectors of the YOLO family and their relationship with geometric descriptors such as *bounding boxes* extracted by segmentation.
* Singh (2024) for the construction of complex mazes based on Hamiltonian cycles over quasicrystalline mosaics, used as an application example of the Geodesic Distance Transform and pathfinding algorithms.
* Zampirolli (2025) for the implementation of morphological operators, geodesic transforms, and maze solving by distance propagation in restricted domains.

## Chapter References


BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

LOTUFO, R.A.; ZAMPIROLLI, F.A. **Fast multidimensional parallel Euclidean distance transform based on mathematical morphology**. 2001.

MATHERON, G. **Random Sets and Integral Geometry**. New York, John Wiley \& Sons, 1975.

REDMON, Joseph *et al*. **You Only Look Once: Unified, Real-Time Object Detection**. 2016.

SERRA, Jean. **Image Analysis and Mathematical Morphology**. London, Academic Press, 1982.

SINGH, S.; LLOYD, J.; FLICKER, F. **Hamiltonian Cycles on Ammann-Beenker Tilings**. 2024.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

ZAMPIROLLI, Francisco de Assis *et al*. **Teaching Hands-On Digital Image Processing with morph.py: Methods and Comprehensive Results**. 2025.

*Reference not found for: staticmethod*